# Rice pest detector: train RF-DETR-M on Kaggle

**End-to-end reference notebook (E2E, REFERENCE). It runs the full published recipe and is not a release-grade tutorial under the owner's notebook standard.**

> **Use notice.** For research, training and evaluation only. The model is a teaching demo: it has not been validated in the field or by experts, and it is not a pest-surveillance or pest-alert system. Its counts are experimental estimates, not agronomic advice, and must not be the basis for any pest-management decision. Running this notebook downloads third-party model weights and a dataset under their own licences and uses your own Kaggle GPU quota. Results here are tutorial evidence, not benchmarks. Everything is provided "as is", without warranty of any kind.

This notebook retrains the RF-DETR-M detector (RF-DETR, medium size) of the open-source rice pest detector project with the recipe that produced its published model. It counts two rice pests, the rice black bug (RBB) and the white stemborer (WSB), in light-trap photos, as a teaching example. Choose **Run all**; no cell needs editing.

**What it does**

1. Checks the runtime (Linux x86_64, one T4 GPU, Internet on) before anything else.
2. Builds its own Python 3.12.12 environment from a hash-locked package list, without touching Kaggle's preinstalled packages.
3. Downloads the dataset archive from Zenodo and checks it by size, MD5 and SHA-256, and every image by SHA-256.
4. Validates the data: split sizes, file names, class ids, image sizes and the most crowded tray.
5. Fine-tunes `Roboflow/rf-detr-medium` for 40 epochs and keeps the checkpoint with the best validation species mAP50.
6. Evaluates that checkpoint on the validation photos at the published counting threshold (0.48).
7. Exports it to ONNX, reloads the file in a fresh process and checks it against the trained model.
8. Counts insects on sample validation photos with the reloaded ONNX model and writes a run summary.

**What it needs**

- A Kaggle account that is phone-verified (Kaggle requires this for GPU and Internet).
- Settings: **Accelerator: GPU T4 x2** (one T4 is used) and **Internet on**.
- Time: the full 40-epoch recipe, measured at 5.9 h on a Colab T4 (21,108 s; run RF-DETR-M, 17 Oct 2026), plus a few minutes to build the environment and about 20 minutes to download and check the images. Times on Kaggle are not yet measured.
- Kaggle ends a session after 12 hours. If the session ends early, the run is lost and starts again from the beginning, unless you attach an earlier run's output and set `RESUME_FROM`.
- The run uses about 6.5 hours of your weekly Kaggle GPU quota.

This is a reference notebook: it reproduces the published recipe in full instead of a short tutorial run.

**The published model.** The model this recipe produced is on Kaggle Models as [RF-DETR-M](https://www.kaggle.com/models/kurtvalcorza/rice-pest-detector/onnx/rf-detr-m). This notebook attaches it as an input, so the model page lists the notebook that trains it; the notebook does not load it. The run's own model is new and will not match the published file byte for byte.


## Problem and data

**Problem.** Light traps catch many insects overnight; counting the two pests in a photo of the tray by hand is slow. This notebook trains a detector that finds each rice black bug and white stemborer (female and male of each, four classes) and counts them per species. It is a teaching example of fine-tuning and checking a detector.

**Scope.** The model is a teaching demo. It is not for farmers, it has not been validated in the field, and nothing here is advice on what to do about pests.

**Data.** The photos and labels come from an open dataset by its authors (Deleña et al.), published on Zenodo under CC BY 4.0. The notebook downloads the archive at run time; no image is redistributed with it or saved in its output.

Data: Deleña, R.; Balleras, G.; Dequiña, B. J.; Jumawan, R.; Flores, C. *Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species-sex recognition of rice black bug and white stemborer.* Version 1. Zenodo. [doi:10.5281/zenodo.20066074](https://doi.org/10.5281/zenodo.20066074). Licence: [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). The embedded labels are derived from it; changes are listed in `data/README.md`.

**Label lineage.** The original YOLO labels were converted to COCO JSON; 652 label files held polygon rows, each converted to its bounding box; one image family was excluded because an image had no label file. Images were grouped into families (an original and its augmented variants), near-duplicate families were merged and the families were re-split 70 / 15 / 15 at random by family with a fixed seed (20261013), on the assumption that families are independent. The images are pre-stretched to 640 × 640 in the archive; RF-DETR-M resizes them to 576 × 576, its native input size.

**Why the notebook is about 4.5 MB.** The labels and the split manifest (about 4.4 MB of JSON, compressed in the data cells below) travel inside the notebook so that it needs no repository clone; the images do not, so that none is redistributed. Preprocessing is fixed (resize and division by 255), so nothing is fitted on any split.


In [ ]:
import json, os, platform, shutil, subprocess, time, urllib.error, urllib.request
SESSION_START = time.time()
print('Supported runtime: a Kaggle notebook on Linux x86_64 with the GPU T4 accelerator and Internet on.')
if platform.system() != 'Linux' or platform.machine() != 'x86_64':
    raise RuntimeError(f'Unsupported platform {platform.system()} {platform.machine()}: the package lock is built for '
                       'Linux x86_64 only. Run this notebook on Kaggle with the GPU T4 accelerator.')
try:
    _gpu = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
                          capture_output=True, text=True, timeout=60)
    GPUS = [n.strip() for n in _gpu.stdout.splitlines() if n.strip()] if _gpu.returncode == 0 else []
except (FileNotFoundError, subprocess.TimeoutExpired):
    GPUS = []
if not GPUS:
    raise RuntimeError('No GPU found. In the editor open Settings > Accelerator, choose GPU T4 x2, then Run all.')
if 'T4' not in GPUS[0]:
    raise RuntimeError(f'The GPU is {GPUS[0]}, not a T4. In Settings > Accelerator choose GPU T4 x2, then Run all.')
print('GPU:', ', '.join(GPUS), '(device 0 is used)')
try:
    urllib.request.urlopen(urllib.request.Request('https://zenodo.org/api/records/20066074', method='HEAD'),
                           timeout=30).close()
except urllib.error.HTTPError:
    pass  # an HTTP answer means the network is up
except (urllib.error.URLError, TimeoutError, OSError) as exc:
    raise RuntimeError('zenodo.org cannot be reached: Internet is off. In Settings turn Internet on (this needs a '
                       'phone-verified Kaggle account), then Run all.') from exc
print('Internet: zenodo.org reachable')
for _folder in ('/tmp', '/kaggle/working'):
    print(f'free space in {_folder}: {shutil.disk_usage(_folder).free / 1e9:.1f} GB')

In [ ]:
import json, os
from pathlib import Path
WORK = '/tmp/rpd'  # scratch: package, labels, archive, images and environment; not saved
RUN_DIR = '/kaggle/working/run-rfdetr-m'  # saved output
RESUME_FROM = ''  # @param {type:"string"}
EDITION = {'variant': 'rfdetr-m-kaggle',
 'model_id': 'rfdetr-m',
 'base_model': 'Roboflow/rf-detr-medium',
 'revision': '1b5b672408f86dd38e05dd3cf3f2e0834e545a59',
 'input_size': 576,
 'threshold': 0.48,
 'review_below': 0.63,
 'epochs': 40,
 'batch_size': 4,
 'score_tolerance_enforced': True,
 'score_waiver': None}
for _sub in ('rice_pest_detector', 'rpd_tools', 'data', 'images'):
    os.makedirs(os.path.join(WORK, _sub), exist_ok=True)
os.makedirs(RUN_DIR, exist_ok=True)
Path(WORK, 'edition.json').write_text(json.dumps(EDITION, indent=1))
print('scratch:', WORK, '| output:', RUN_DIR)
print(json.dumps(EDITION, indent=1))

In [ ]:
%%writefile /tmp/rpd/rice_pest_detector/__init__.py
"""Rice black bug / white stemborer detector (RT-DETR) for light-trap images."""

__version__ = "0.1.0"

CLASS_NAMES: tuple[str, ...] = ("RBB_FEMALE", "RBB_MALE", "WSB_FEMALE", "WSB_MALE")
SPECIES_OF_CLASS: dict[int, str] = {0: "RBB", 1: "RBB", 2: "WSB", 3: "WSB"}


In [ ]:
%%writefile /tmp/rpd/rice_pest_detector/dedup.py
"""Perceptual (dHash) duplicate screening across image families."""

import numpy as np
from PIL import Image


def dhash(image: Image.Image) -> int:
    small = np.asarray(image.convert("L").resize((9, 8), Image.Resampling.BILINEAR), dtype=np.int16)
    bits = (small[:, 1:] > small[:, :-1]).flatten()
    return int("".join("1" if b else "0" for b in bits), 2)


def _popcount(x: np.ndarray) -> np.ndarray:
    return np.unpackbits(x.view(np.uint8).reshape(-1, 8), axis=1).sum(axis=1)


def group_families(hashes: dict[str, list[int]], max_distance: int = 4) -> list[frozenset[str]]:
    families = sorted(hashes)
    owner = np.array([i for i, f in enumerate(families) for _ in hashes[f]], dtype=np.int64)
    values = np.array([h for f in families for h in hashes[f]], dtype=np.uint64)
    parent = list(range(len(families)))

    def find(i: int) -> int:
        while parent[i] != i:
            parent[i] = parent[parent[i]]
            i = parent[i]
        return i

    for k in range(len(values)):
        close = np.nonzero(_popcount(values ^ values[k]) <= max_distance)[0]
        for j in close:
            ra, rb = find(int(owner[k])), find(int(owner[j]))
            if ra != rb:
                parent[rb] = ra
    groups: dict[int, set[str]] = {}
    for i, f in enumerate(families):
        groups.setdefault(find(i), set()).add(f)
    return [frozenset(g) for g in groups.values()]


In [ ]:
%%writefile /tmp/rpd/rice_pest_detector/families.py
"""Map Roboflow export filenames to their source-image family."""

import re

_ROBOFLOW = re.compile(
    r"^(?P<stem>.+?)_(?:jpe?g|png)\.rf\.[0-9a-f]+\.(?:jpe?g|png)$",
    re.IGNORECASE,
)


def family_key(filename: str) -> str:
    """Return the lower-cased source stem shared by every export variant of one photo."""
    match = _ROBOFLOW.match(filename)
    if match is None:
        raise ValueError(f"not a Roboflow export filename: {filename!r}")
    return match.group("stem").lower()


In [ ]:
%%writefile /tmp/rpd/rice_pest_detector/fetch.py
"""Single-request archive download and manifest-scoped extraction (Colab data path)."""

import hashlib
import urllib.request
import zipfile
from pathlib import Path

ZENODO_URL = "https://zenodo.org/api/records/20066074/files/datasets.zip/content"
ZENODO_BYTES = 1_005_857_218
ZENODO_MD5 = "24960c245a3a21b63be95314cc7467da"
ZENODO_SHA256 = "f4a0409977e0615e34f3cf2b936a5ca9dd3a91b153d239e314334ea40c2537f7"


def download(url: str, dest: Path, expected_bytes: int, expected_md5: str, expected_sha256: str) -> Path:
    dest = Path(dest)
    part = dest.with_name(dest.name + ".part")
    md5, sha, size = hashlib.md5(), hashlib.sha256(), 0
    try:
        with urllib.request.urlopen(url, timeout=120) as response, part.open("wb") as out:
            while chunk := response.read(1 << 20):
                out.write(chunk)
                md5.update(chunk)
                sha.update(chunk)
                size += len(chunk)
    except BaseException:
        part.unlink(missing_ok=True)
        raise
    problems = []
    if size != expected_bytes:
        problems.append(f"bytes {size} != {expected_bytes}")
    if md5.hexdigest() != expected_md5:
        problems.append("md5 mismatch")
    if sha.hexdigest() != expected_sha256:
        problems.append("sha256 mismatch")
    if problems:
        part.unlink(missing_ok=True)
        raise ValueError(f"{url}: " + "; ".join(problems))
    part.replace(dest)
    return dest


def extract_members(archive: Path, manifest: dict, out_root: Path) -> int:
    out_root = Path(out_root).resolve()
    count = 0
    with zipfile.ZipFile(archive) as z:
        for rec in manifest["images"]:
            target = (out_root / rec["relative_path"]).resolve()
            if out_root not in target.parents:
                raise ValueError(f"{rec['relative_path']} escapes {out_root}")
            try:
                data = z.read(rec["archive_member"])
            except KeyError:
                raise KeyError(
                    f"archive member {rec['archive_member']!r} not found in {archive}; "
                    "check the archive prefix (see docs/evaluation-record.md, 'Colab data path')"
                ) from None
            if hashlib.sha256(data).hexdigest() != rec["sha256"]:
                raise ValueError(f"digest mismatch for {rec['archive_member']}")
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(data)
            count += 1
    return count


In [ ]:
%%writefile /tmp/rpd/rice_pest_detector/infer.py
"""Inference contract shared by export parity checks and downstream tools."""

from dataclasses import dataclass

import numpy as np
from PIL import Image, ImageOps

INPUT_SIZE = 640


@dataclass(frozen=True)
class Detection:
    class_id: int
    score: float
    x0: float
    y0: float
    x1: float
    y1: float


def preprocess(image: Image.Image, size: int = INPUT_SIZE) -> np.ndarray:
    """size is the model input side; the app contract is 640 (RF-DETR-M trains at 576)."""
    rgb = ImageOps.exif_transpose(image).convert("RGB")
    resized = rgb.resize((size, size), Image.Resampling.BILINEAR)
    array = np.asarray(resized, dtype=np.float32) / 255.0
    return array.transpose(2, 0, 1)[np.newaxis].copy()


def postprocess(logits: np.ndarray, boxes: np.ndarray, width: int, height: int,
                threshold: float = 0.5) -> list[Detection]:
    """Turn raw RT-DETR outputs into detections in original-image pixels.

    ``width`` and ``height`` MUST be the size of the EXIF-transposed image
    (``ImageOps.exif_transpose(img).size``), not the raw ``img.size``; otherwise
    boxes are wrong on rotated JPEGs.
    """
    probs = 1.0 / (1.0 + np.exp(-logits[0].astype(np.float64)))
    scores, classes = probs.max(axis=1), probs.argmax(axis=1)
    detections = []
    for q in np.nonzero(scores >= threshold)[0]:
        cx, cy, w, h = boxes[0, q].astype(np.float64)
        detections.append(Detection(int(classes[q]), float(scores[q]),
                                    float((cx - w / 2) * width), float((cy - h / 2) * height),
                                    float((cx + w / 2) * width), float((cy + h / 2) * height)))
    return sorted(detections, key=lambda d: -d.score)


In [ ]:
%%writefile /tmp/rpd/rice_pest_detector/labels.py
"""YOLO-format label parsing and validation."""

import math
from dataclasses import dataclass
from pathlib import Path

from rice_pest_detector import CLASS_NAMES

_EPS = 1e-6


@dataclass(frozen=True)
class Box:
    class_id: int
    cx: float
    cy: float
    w: float
    h: float


def parse_labels(text: str) -> list[Box]:
    boxes: list[Box] = []
    for number, line in enumerate(text.splitlines(), start=1):
        fields = line.split()
        if not fields:
            continue
        if len(fields) == 5:
            boxes.append(Box(int(fields[0]), *(float(v) for v in fields[1:])))
        elif len(fields) >= 7 and len(fields) % 2 == 1:
            # YOLO-seg polygon: class id followed by x,y pairs -> min/max bounding box.
            values = [float(v) for v in fields[1:]]
            xs, ys = values[0::2], values[1::2]
            boxes.append(Box(int(fields[0]), (min(xs) + max(xs)) / 2, (min(ys) + max(ys)) / 2,
                             max(xs) - min(xs), max(ys) - min(ys)))
        else:
            raise ValueError(f"line {number}: expected 5 fields or an odd polygon row, got {len(fields)}")
    return boxes


def validate_boxes(boxes: list[Box]) -> list[str]:
    errors: list[str] = []
    for i, b in enumerate(boxes):
        if not 0 <= b.class_id < len(CLASS_NAMES):
            errors.append(f"box {i}: class {b.class_id} not in 0..{len(CLASS_NAMES) - 1}")
        elif not all(math.isfinite(v) for v in (b.cx, b.cy, b.w, b.h)):
            errors.append(f"box {i}: non-finite coordinate")
        elif b.w <= 0 or b.h <= 0:
            errors.append(f"box {i}: non-positive area")
        elif (b.cx - b.w / 2 < -_EPS or b.cx + b.w / 2 > 1 + _EPS
              or b.cy - b.h / 2 < -_EPS or b.cy + b.h / 2 > 1 + _EPS):
            errors.append(f"box {i}: extends outside the image")
    return errors


def load_label_file(path: Path) -> list[Box]:
    return parse_labels(Path(path).read_text(encoding="utf-8"))


In [ ]:
%%writefile /tmp/rpd/rice_pest_detector/manifest.py
"""Build the leakage-free split manifest and COCO annotations from the local dataset."""

import argparse
import hashlib
import json
from collections import defaultdict
from pathlib import Path

import numpy as np
from PIL import Image

from rice_pest_detector import CLASS_NAMES
from rice_pest_detector.dedup import dhash, group_families
from rice_pest_detector.families import family_key
from rice_pest_detector.labels import Box, load_label_file, validate_boxes
from rice_pest_detector.split import Variant, select_eval_variant, split_groups, stratum

_SPLITS = ("train", "valid", "test")

# Measured on the real dataset: see "Data preparation" in docs/evaluation-record.md.
DEDUP_MAX_DISTANCE = 1


def _representative(items: list[dict]) -> tuple[dict, str | None]:
    """The family's representative image, chosen by the evaluation-variant rule."""
    variants = [Variant(i["path"].name, i["origin"], i["grey"]) for i in items]
    pick, rule = select_eval_variant(variants)
    return next(i for i in items if i["path"].name == pick.name), rule


def _is_grey(img: Image.Image) -> bool:
    a = np.asarray(img.convert("RGB"), dtype=np.int16)
    return bool(np.abs(a[..., 0] - a[..., 1]).max() <= 2 and np.abs(a[..., 1] - a[..., 2]).max() <= 2)


def _scan(root: Path) -> tuple[dict[str, list[dict]], list[dict]]:
    families: dict[str, list[dict]] = defaultdict(list)
    for origin in _SPLITS:
        image_dir = root / "02_detection_dataset" / origin / "images"
        for path in sorted(image_dir.glob("*")):
            if path.suffix.lower() not in {".jpg", ".jpeg", ".png"}:
                continue
            families[family_key(path.name)].append({"path": path, "origin": origin})
    exclusions: list[dict] = []
    for fam in sorted(families):
        for item in families[fam]:
            label = item["path"].parent.parent / "labels" / (item["path"].stem + ".txt")
            try:
                item["boxes"] = load_label_file(label)
            except FileNotFoundError:
                exclusions.append({"family": fam, "reason": f"missing label for {item['path'].name}"})
                break
            except ValueError as exc:
                exclusions.append({"family": fam, "reason": f"{item['path'].name}: {exc}"})
                break
            errors = validate_boxes(item["boxes"])
            if errors:
                exclusions.append({"family": fam, "reason": f"{item['path'].name}: {errors[0]}"})
                break
    for e in exclusions:
        families.pop(e["family"], None)
    return families, exclusions


def _coco(records: list[dict]) -> dict:
    images, annotations = [], []
    for image_id, rec in enumerate(records, start=1):
        w, h = rec["width"], rec["height"]
        images.append({"id": image_id, "file_name": rec["file"], "width": w, "height": h})
        for c, cx, cy, bw, bh in rec["boxes"]:
            box = [(cx - bw / 2) * w, (cy - bh / 2) * h, bw * w, bh * h]
            annotations.append({"id": len(annotations) + 1, "image_id": image_id, "category_id": c + 1,
                                "bbox": box, "area": box[2] * box[3], "iscrowd": 0})
    categories = [{"id": i + 1, "name": n} for i, n in enumerate(CLASS_NAMES)]
    return {"images": images, "annotations": annotations, "categories": categories}


def prepare(dataset_root: Path, out_dir: Path, seed: int = 20261013, num_queries: int = 300) -> dict:
    root = Path(dataset_root)
    families, exclusions = _scan(root)
    for fam, items in families.items():
        for item in items:
            if len(item["boxes"]) > num_queries:
                raise ValueError(f"{item['path'].name}: {len(item['boxes'])} boxes exceeds "
                                 f"num_queries={num_queries}")
            with Image.open(item["path"]) as img:
                item["grey"], item["size"] = _is_grey(img), img.size
                item["hash"] = dhash(img)
    rep_hashes = {f: [_representative(items)[0]["hash"]] for f, items in families.items()}
    groups = group_families(rep_hashes, max_distance=DEDUP_MAX_DISTANCE)
    group_of = {f: f"grp{n:05d}" for n, g in enumerate(sorted(groups, key=sorted)) for f in g}
    group_boxes: dict[str, list[Box]] = defaultdict(list)
    for fam, items in families.items():
        group_boxes[group_of[fam]].extend(b for i in items for b in i["boxes"])
    split_of_group = split_groups({g: stratum(b) for g, b in group_boxes.items()}, seed=seed)

    records: list[dict] = []
    for fam in sorted(families):
        items, split = families[fam], split_of_group[group_of[fam]]
        chosen, rule = items, None
        if split != "train":
            pick, rule = _representative(items)
            chosen = [pick]
        for item in chosen:
            rel = item["path"].relative_to(root).as_posix()
            data = item["path"].read_bytes()
            records.append({
                "file": item["path"].name, "relative_path": rel, "archive_member": f"datasets/{rel}",
                "family": fam, "group": group_of[fam], "split": split, "origin_split": item["origin"],
                "selection_rule": rule, "sha256": hashlib.sha256(data).hexdigest(), "bytes": len(data),
                "width": item["size"][0], "height": item["size"][1],
                "boxes": [[b.class_id, b.cx, b.cy, b.w, b.h] for b in item["boxes"]],
            })

    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)
    (out / "split_manifest.json").write_text(json.dumps({"seed": seed, "images": records}, indent=1),
                                             encoding="utf-8", newline="\n")
    (out / "exclusions.json").write_text(json.dumps(exclusions, indent=1), encoding="utf-8", newline="\n")
    for split in ("train", "val", "test"):
        coco = _coco([r for r in records if r["split"] == split])
        (out / f"annotations_{split}.json").write_text(json.dumps(coco), encoding="utf-8", newline="\n")
    counts = {s: sum(r["split"] == s for r in records) for s in ("train", "val", "test")}
    return {"images": counts, "families": len(families), "groups": len(groups),
            "excluded_families": len(exclusions)}


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--dataset-root", type=Path, required=True)
    parser.add_argument("--out", type=Path, default=Path("data"))
    args = parser.parse_args()
    print(json.dumps(prepare(args.dataset_root, args.out), indent=1))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /tmp/rpd/rice_pest_detector/metrics.py
"""Ship-gate metrics: species-grouped mAP50 and per-image count accuracy."""

import contextlib
import copy
import io

import numpy as np

from rice_pest_detector import SPECIES_OF_CLASS

_SPECIES = ("RBB", "WSB")


def count_within_tolerance(pred: int, ref: int) -> bool:
    if ref < 10:
        return abs(pred - ref) <= 1
    return abs(pred - ref) <= 0.10 * ref


def species_counts(class_ids: list[int]) -> dict[str, int]:
    counts = dict.fromkeys(_SPECIES, 0)
    for c in class_ids:
        counts[SPECIES_OF_CLASS[c]] += 1
    return counts


def count_accuracy(pred_counts: list[dict[str, int]], ref_counts: list[dict[str, int]]) -> float:
    passed = [all(count_within_tolerance(p[s], r[s]) for s in _SPECIES)
              for p, r in zip(pred_counts, ref_counts, strict=True)]
    return float(np.mean(passed))


_DENSITY_BINS = ((1, 9, "1-9"), (10, 29, "10-29"), (30, None, "30+"))


def count_errors(pred_counts: list[dict[str, int]], ref_counts: list[dict[str, int]]) -> dict:
    """Per-species counting errors to read beside count_accuracy, whose +-1 tolerance below a
    reference of 10 lets an image with one insect pass when nothing is detected.

    mae: mean |pred - ref| over all images; bias: mean (pred - ref), negative = undercount;
    exact: share of images counted exactly; presence_recall: share of images containing the
    species where at least one was counted; mae_by_density: mae over images whose reference
    count falls in each bin (images without the species are excluded).
    """
    out = {}
    for s in _SPECIES:
        pairs = [(p[s], r[s]) for p, r in zip(pred_counts, ref_counts, strict=True)]
        diffs = np.array([p - r for p, r in pairs], dtype=float)
        present = [(p, r) for p, r in pairs if r > 0]
        by_density = {}
        for lo, hi, label in _DENSITY_BINS:
            sub = [abs(p - r) for p, r in present if r >= lo and (hi is None or r <= hi)]
            by_density[label] = {"images": len(sub), "mae": float(np.mean(sub)) if sub else None}
        out[s] = {"mae": float(np.abs(diffs).mean()) if pairs else None,
                  "bias": float(diffs.mean()) if pairs else None,
                  "exact": float(np.mean(diffs == 0)) if pairs else None,
                  "presence_recall": float(np.mean([p > 0 for p, _ in present])) if present else None,
                  "mae_by_density": by_density}
    return out


def _box_iou(a: list[float], b: list[float]) -> float:
    ix = max(0.0, min(a[2], b[2]) - max(a[0], b[0]))
    iy = max(0.0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = ix * iy
    union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / union if union > 0 else 0.0


def confusion(gt: list[tuple[int, list[float]]], det: list[tuple[int, list[float]]],
              iou: float = 0.5) -> dict[tuple[str, str], int]:
    from rice_pest_detector import CLASS_NAMES

    table: dict[tuple[str, str], int] = {}
    unmatched = list(range(len(gt)))
    for d_cls, d_box in det:
        best = max(unmatched, key=lambda g: _box_iou(gt[g][1], d_box), default=None)
        if best is not None and _box_iou(gt[best][1], d_box) >= iou:
            unmatched.remove(best)
            key = (CLASS_NAMES[gt[best][0]], CLASS_NAMES[d_cls])
        else:
            key = ("spurious", CLASS_NAMES[d_cls])
        table[key] = table.get(key, 0) + 1
    for g in unmatched:
        key = (CLASS_NAMES[gt[g][0]], "missed")
        table[key] = table.get(key, 0) + 1
    return table


def _to_species(coco: dict, dets: list[dict]) -> tuple[dict, list[dict]]:
    species_id = {c: 1 if SPECIES_OF_CLASS[c - 1] == "RBB" else 2 for c in range(1, 5)}
    gt = copy.deepcopy(coco)
    for a in gt["annotations"]:
        a["category_id"] = species_id[a["category_id"]]
    gt["categories"] = [{"id": 1, "name": "RBB"}, {"id": 2, "name": "WSB"}]
    return gt, [{**d, "category_id": species_id[d["category_id"]]} for d in dets]


def map50(gt_coco: dict, detections: list[dict], group_species: bool) -> dict[str, float]:
    from pycocotools.coco import COCO
    from pycocotools.cocoeval import COCOeval

    gt_dict, dets = _to_species(gt_coco, detections) if group_species else (gt_coco, detections)
    with contextlib.redirect_stdout(io.StringIO()):
        gt = COCO()
        gt.dataset = copy.deepcopy(gt_dict)
        gt.createIndex()
        dt = gt.loadRes(dets) if dets else COCO()
        ev = COCOeval(gt, dt, "bbox")
        ev.params.maxDets = [1, 10, 300]  # trays hold up to 106 insects; COCO's default cap is 100
        ev.evaluate()
        ev.accumulate()
        ev.summarize()
    result = {"all": float(ev.stats[1])}  # AP at IoU 0.50, maxDets=300
    precision = ev.eval["precision"][0, :, :, 0, -1]  # IoU=0.50, all recall, area=all, maxDets=300
    for k, cat in enumerate(gt_dict["categories"]):
        p = precision[:, k]
        result[cat["name"]] = float(p[p > -1].mean()) if (p > -1).any() else 0.0
    return result


In [ ]:
%%writefile /tmp/rpd/rice_pest_detector/split.py
"""Family-group split (70/15/15, stratified) and evaluation-variant selection."""

import random
from dataclasses import dataclass

from rice_pest_detector import SPECIES_OF_CLASS
from rice_pest_detector.labels import Box


def stratum(boxes: list[Box]) -> str:
    species = {SPECIES_OF_CLASS[b.class_id] for b in boxes}
    if species == {"RBB"}:
        return "rbb"
    if species == {"WSB"}:
        return "wsb"
    return "mixed"


def split_groups(
    group_strata: dict[str, str],
    seed: int = 20261013,
    ratios: tuple[float, float, float] = (0.70, 0.15, 0.15),
) -> dict[str, str]:
    rng = random.Random(seed)
    assignment: dict[str, str] = {}
    for name in sorted(set(group_strata.values())):
        groups = sorted(g for g, s in group_strata.items() if s == name)
        rng.shuffle(groups)
        n_val = round(ratios[1] * len(groups))
        n_test = round(ratios[2] * len(groups))
        for i, g in enumerate(groups):
            assignment[g] = "val" if i < n_val else "test" if i < n_val + n_test else "train"
    return assignment


@dataclass(frozen=True)
class Variant:
    name: str
    origin_split: str
    grayscale: bool


def select_eval_variant(variants: list[Variant]) -> tuple[Variant, str]:
    ordered = sorted(variants, key=lambda v: v.name)
    for v in ordered:
        if v.origin_split in {"valid", "test"}:
            return v, "published-original"
    for v in ordered:
        if not v.grayscale:
            return v, "first-colour"
    return ordered[0], "first-any"


In [ ]:
%%writefile /tmp/rpd/rice_pest_detector/train.py
"""Fine-tune RT-DETR on the split manifest; resumable checkpoints; validation-only selection."""

import hashlib
import json
import math
import random
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np
import torch
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModelForObjectDetection, RTDetrConfig, RTDetrForObjectDetection

from rice_pest_detector import CLASS_NAMES
from rice_pest_detector.infer import postprocess, preprocess
from rice_pest_detector.metrics import confusion, count_accuracy, count_errors, map50, species_counts

EVALUATOR_VERSION = 2  # bump whenever evaluate() changes what it measures


@dataclass
class TrainConfig:
    base_model: str
    revision: str
    epochs: int = 60
    batch_size: int = 8
    lr: float = 1e-4
    backbone_lr: float = 1e-5
    weight_decay: float = 1e-4
    grad_clip: float = 0.1
    seed: int = 20261013
    num_queries: int = 300
    fp16: bool = True
    num_workers: int = 2
    input_size: int = 640  # model input side; 640 is the app contract


class ManifestDataset(Dataset):
    def __init__(self, manifest: dict, image_root: Path, split: str, size: int = 640):
        self.records = [r for r in manifest["images"] if r["split"] == split]
        self.root = Path(image_root)
        self.size = size

    def __len__(self) -> int:
        return len(self.records)

    def __getitem__(self, i: int):
        rec = self.records[i]
        with Image.open(self.root / rec["relative_path"]) as img:
            pixels = torch.from_numpy(preprocess(img, self.size)[0])
        boxes = rec["boxes"]
        target = {"class_labels": torch.tensor([b[0] for b in boxes], dtype=torch.long),
                  "boxes": torch.tensor([b[1:] for b in boxes], dtype=torch.float32).reshape(-1, 4)}
        return pixels, target


def collate(batch):
    return torch.stack([b[0] for b in batch]), [b[1] for b in batch]


def _tiny_config() -> RTDetrConfig:  # replaced in tests
    raise RuntimeError("tiny config only available in tests")


def build_model(cfg: TrainConfig):
    """RT-DETR or D-FINE: the checkpoint's own config picks the class (same sigmoid logits / cxcywh boxes)."""
    if cfg.base_model == "__tiny__":
        model = RTDetrForObjectDetection(_tiny_config())
    else:
        id2label = dict(enumerate(CLASS_NAMES))
        model = AutoModelForObjectDetection.from_pretrained(
            cfg.base_model, revision=cfg.revision, num_labels=len(CLASS_NAMES), id2label=id2label,
            label2id={v: k for k, v in id2label.items()}, num_queries=cfg.num_queries,
            ignore_mismatched_sizes=True,
        )
    model.loss_function = fp32_loss(model.loss_function)
    return model


def _to_fp32(value):
    if torch.is_tensor(value):
        return value.float() if value.is_floating_point() else value
    if isinstance(value, dict):
        return {k: _to_fp32(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return type(value)(_to_fp32(v) for v in value)
    return value


def fp32_loss(loss_fn):
    """Run RT-DETR's matcher and losses in float32 even under fp16 autocast.

    The Hungarian matcher takes log(p + 1e-8) of sigmoid scores; in float16 the epsilon
    underflows and saturated logits give inf costs, which scipy rejects with ValueError
    before the training loop's finite-loss check can run. The final, auxiliary and encoder
    outputs all pass through this one call.
    """
    def wrapped(*args, **kwargs):
        device_type = "cuda" if torch.cuda.is_available() else "cpu"
        with torch.autocast(device_type=device_type, enabled=False):
            return loss_fn(*_to_fp32(args), **_to_fp32(kwargs))
    wrapped.__wrapped__ = loss_fn
    return wrapped


def _sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def _sha256_json(obj) -> str:
    return _sha256_bytes(json.dumps(obj, sort_keys=True).encode("utf-8"))


def experiment_id(cfg: TrainConfig, manifest: dict, val_annotations: bytes) -> dict:
    """What makes two runs the same experiment; num_workers only changes speed."""
    settings = {k: v for k, v in asdict(cfg).items() if k != "num_workers"}
    if settings.get("input_size") == 640:  # runs from before the field existed keep their identity
        del settings["input_size"]
    return {"config": settings, "manifest_sha256": _sha256_json(manifest),
            "annotations_val_sha256": _sha256_bytes(val_annotations)}


def check_experiment(saved: dict | None, current: dict) -> None:
    if saved is None:
        print("WARNING: last.pt predates experiment fingerprints; resuming without an identity check")
        return
    if saved != current:
        changed = sorted(k for k in saved["config"].keys() | current["config"].keys()
                         if saved["config"].get(k) != current["config"].get(k))
        changed += [k for k in ("manifest_sha256", "annotations_val_sha256") if saved.get(k) != current.get(k)]
        raise RuntimeError(f"run directory holds a different experiment (changed: {', '.join(changed)}); "
                           "use a new RUN_DIR instead of resuming")


def _rng_state() -> dict:
    name, keys, pos, has_gauss, cached = np.random.get_state()
    return {"python": random.getstate(), "numpy": [name, keys.tolist(), pos, has_gauss, cached],
            "torch": torch.get_rng_state(),
            "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else []}


def _set_rng_state(state: dict) -> None:
    version, internal, gauss = state["python"]
    random.setstate((version, tuple(internal), gauss))
    name, keys, pos, has_gauss, cached = state["numpy"]
    np.random.set_state((name, np.array(keys, dtype=np.uint32), pos, has_gauss, cached))
    torch.set_rng_state(state["torch"])
    if state["cuda"] and torch.cuda.is_available():
        torch.cuda.set_rng_state_all(state["cuda"])


def _atomic_save(obj, path: Path) -> None:
    path = Path(path)
    tmp = path.with_name(path.name + ".tmp")
    torch.save(obj, tmp)
    tmp.replace(path)


def _atomic_write_text(path: Path, text: str) -> None:
    path = Path(path)
    tmp = path.with_name(path.name + ".tmp")
    tmp.write_text(text)
    tmp.replace(path)


def save_checkpoint(path: Path, model, optimizer, scheduler, epoch: int, best_metric: float,
                    scaler=None, experiment: dict | None = None) -> None:
    _atomic_save({"model": model.state_dict(), "optimizer": optimizer.state_dict(),
                  "scheduler": scheduler.state_dict(), "epoch": epoch, "best_metric": best_metric,
                  "scaler": scaler.state_dict() if scaler is not None else {},
                  "experiment": experiment, "rng": _rng_state()}, path)


def load_checkpoint(path: Path, model, optimizer, scheduler, scaler=None,
                    experiment: dict | None = None) -> tuple[int, float]:
    state = torch.load(path, map_location="cpu", weights_only=True)
    if experiment is not None:
        check_experiment(state.get("experiment"), experiment)
    model.load_state_dict(state["model"])
    optimizer.load_state_dict(state["optimizer"])
    scheduler.load_state_dict(state["scheduler"])
    if scaler is not None and state.get("scaler"):
        scaler.load_state_dict(state["scaler"])
    if state.get("rng"):
        _set_rng_state(state["rng"])
    return state["epoch"] + 1, state["best_metric"]


def load_best(path: Path) -> tuple[dict, dict]:
    """Return (state_dict, metadata) from best.pt; older runs saved a bare state_dict."""
    state = torch.load(path, map_location="cpu", weights_only=True)
    if "model" in state and "map50_species" in state:
        return state["model"], {k: v for k, v in state.items() if k != "model"}
    return state, {}


def record_best(run_dir: Path, model, score: float, epoch: int, best: float,
                experiment: dict | None = None) -> float:
    """Save weights, score and epoch together in one atomic best.pt if `score` beats `best`.

    best.json is a human-readable copy written afterwards; resume trusts best.pt, so an
    interruption between the two writes cannot pair new weights with an old score.
    """
    if not math.isfinite(score) or score <= best:
        return best
    run_dir = Path(run_dir)
    _atomic_save({"model": model.state_dict(), "map50_species": score, "epoch": epoch,
                  "experiment": experiment}, run_dir / "best.pt")
    _atomic_write_text(run_dir / "best.json", json.dumps({"map50_species": score, "epoch": epoch}))
    return score


MAX_NONFINITE_STEPS = 50


def read_history(path: Path) -> list[dict]:
    """Parse history.jsonl, dropping a final line cut short by a disconnect; any other bad line raises."""
    lines = [ln for ln in Path(path).read_text().splitlines() if ln.strip()]
    rows = []
    for i, ln in enumerate(lines):
        try:
            rows.append(json.loads(ln))
        except json.JSONDecodeError:
            if i != len(lines) - 1:
                raise
            print(f"WARNING: dropping truncated final history line: {ln[:60]!r}")
    return rows


def trim_history(path: Path, start: int) -> None:
    """Drop history rows with epoch >= start so a resumed run does not duplicate epochs."""
    path = Path(path)
    if not path.exists():
        return
    rows = [r for r in read_history(path) if r["epoch"] < start]
    _atomic_write_text(path, "".join(json.dumps(r) + "\n" for r in rows))


def resume_best(run_dir: Path, ckpt_best: float) -> float:
    """Best metric on resume: last.pt may lag best.pt by one epoch after an interruption."""
    run_dir = Path(run_dir)
    if (run_dir / "best.pt").exists():
        _, meta = load_best(run_dir / "best.pt")
        if meta:
            return max(ckpt_best, meta["map50_species"])
    sidecar = run_dir / "best.json"  # runs from before best.pt carried its own score
    if sidecar.exists():
        return max(ckpt_best, json.loads(sidecar.read_text())["map50_species"])
    return ckpt_best


def _quantiles(values: list[float]) -> dict[str, float]:
    if not values:
        return {}
    q = np.quantile(values, [0.5, 0.9, 1.0])
    return {"p50": float(q[0]), "p90": float(q[1]), "max": float(q[2])}


@torch.no_grad()
def evaluate(model, dataset: ManifestDataset, coco_gt: dict, threshold: float = 0.5) -> dict:
    model.eval()
    device = next(model.parameters()).device
    dets, preds, refs, table, top_scores, kept_total = [], [], [], {}, [], 0
    for image_id, rec in enumerate(dataset.records, start=1):
        assert coco_gt["images"][image_id - 1]["file_name"] == rec["file"], (
            f"COCO image {image_id} is {coco_gt['images'][image_id - 1]['file_name']!r} "
            f"but manifest record is {rec['file']!r}")
        pixels, _ = dataset[image_id - 1]
        out = model(pixel_values=pixels[None].to(device))
        found = postprocess(out.logits.float().cpu().numpy(), out.pred_boxes.float().cpu().numpy(),
                            rec["width"], rec["height"], threshold=0.0)
        dets += [{"image_id": image_id, "category_id": d.class_id + 1, "score": d.score,
                  "bbox": [d.x0, d.y0, d.x1 - d.x0, d.y1 - d.y0]} for d in found]
        top_scores.append(found[0].score if found else 0.0)
        kept = [d for d in found if d.score >= threshold]
        kept_total += len(kept)
        preds.append(species_counts([d.class_id for d in kept]))
        refs.append(species_counts([b[0] for b in rec["boxes"]]))
        w, h = rec["width"], rec["height"]
        gt = [(c, [(cx - bw / 2) * w, (cy - bh / 2) * h, (cx + bw / 2) * w, (cy + bh / 2) * h])
              for c, cx, cy, bw, bh in rec["boxes"]]
        for key, n in confusion(gt, [(d.class_id, [d.x0, d.y0, d.x1, d.y1]) for d in kept]).items():
            table[key] = table.get(key, 0) + n
    zero = [species_counts([]) for _ in refs]
    return {"map50_species": map50(coco_gt, dets, group_species=True)["all"],
            "map50_by_class": map50(coco_gt, dets, group_species=False),
            "threshold": threshold,
            "count_accuracy": count_accuracy(preds, refs),
            "count_accuracy_no_detection_baseline": count_accuracy(zero, refs),
            "count_errors": count_errors(preds, refs),
            "detections_kept": kept_total,
            "top_score_per_image": _quantiles(top_scores),
            "confusion": {f"{g}->{p}": n for (g, p), n in sorted(table.items())}}


def evaluation_id(best_path: Path, manifest: dict, coco_path: Path, threshold: float) -> dict:
    """Everything a cached test result depends on, not just the weights."""
    return {"best_pt_sha256": _sha256_bytes(Path(best_path).read_bytes()),
            "manifest_sha256": _sha256_json(manifest),
            "annotations_sha256": _sha256_bytes(Path(coco_path).read_bytes()),
            "threshold": threshold, "evaluator_version": EVALUATOR_VERSION}


def check_cached_test(prior: dict, current: dict) -> None:
    """Refuse a cached test result that no longer describes the current evaluation."""
    saved = prior.get("evaluation")
    if saved is None:  # legacy record: bound to the weights only
        if prior.get("best_pt_sha256") != current["best_pt_sha256"]:
            raise RuntimeError("best.pt changed after the test split was evaluated: protocol violation, "
                               "stop and ask the owner")
        return
    changed = [k for k in current if saved.get(k) != current[k]]
    if changed:
        raise RuntimeError(f"test split already evaluated under a different setup (changed: "
                           f"{', '.join(changed)}): protocol violation, stop and ask the owner")


def fit(cfg: TrainConfig, manifest: dict, image_root: Path, run_dir: Path) -> Path:
    random.seed(cfg.seed)
    np.random.seed(cfg.seed)
    torch.manual_seed(cfg.seed)
    run_dir = Path(run_dir)
    run_dir.mkdir(parents=True, exist_ok=True)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    amp = cfg.fp16 and device == "cuda"
    val_bytes = (run_dir / "annotations_val.json").read_bytes()
    experiment = experiment_id(cfg, manifest, val_bytes)
    model = build_model(cfg).to(device)
    backbone = [p for n, p in model.named_parameters() if "backbone" in n]
    rest = [p for n, p in model.named_parameters() if "backbone" not in n]
    opt = torch.optim.AdamW([{"params": rest, "lr": cfg.lr}, {"params": backbone, "lr": cfg.backbone_lr}],
                            weight_decay=cfg.weight_decay)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg.epochs)
    scaler = torch.amp.GradScaler(device, enabled=amp)
    start, best = 0, -1.0
    if (run_dir / "last.pt").exists():
        start, best = load_checkpoint(run_dir / "last.pt", model, opt, sched, scaler=scaler,
                                      experiment=experiment)
        best = resume_best(run_dir, best)
        trim_history(run_dir / "history.jsonl", start)
        print(f"resumed at epoch {start}, best val mAP50 {best:.4f}")
    train_set = ManifestDataset(manifest, image_root, "train", size=cfg.input_size)
    if len(train_set) == 0:
        raise ValueError("train split is empty")
    val_set = ManifestDataset(manifest, image_root, "val", size=cfg.input_size)
    val_coco = json.loads(val_bytes)
    _atomic_write_text(run_dir / "config.json", json.dumps({**asdict(cfg), "experiment": experiment}, indent=1))
    for epoch in range(start, cfg.epochs):
        train = DataLoader(train_set, batch_size=cfg.batch_size, shuffle=True, collate_fn=collate,
                           num_workers=cfg.num_workers, generator=torch.Generator().manual_seed(cfg.seed + epoch))
        model.train()
        total, steps, updates, bad_loss, bad_grad, bad = 0.0, 0, 0, 0, 0, 0
        step_no = -1
        for pixels, targets in train:
            step_no += 1
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device, enabled=amp):
                loss = model(pixel_values=pixels.to(device), labels=targets).loss
            if not torch.isfinite(loss.detach()):
                bad_loss += 1
                bad += 1
            else:
                total += loss.detach().item()
                steps += 1
                scaler.scale(loss).backward()
                scaler.unscale_(opt)
                norm = torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
                if torch.isfinite(norm):
                    scaler.step(opt)
                    updates += 1
                    bad = 0
                else:  # skip the update; under AMP the scaler also lowers its scale
                    bad_grad += 1
                    bad += 1
                scaler.update()
            if bad >= MAX_NONFINITE_STEPS:
                raise RuntimeError(
                    f"loss or gradient non-finite for {bad} consecutive steps (epoch {epoch}, step {step_no}); "
                    "training diverged, stopping")
        if updates == 0:
            raise RuntimeError(f"epoch {epoch}: no optimizer updates (all losses or gradients non-finite)")
        sched.step()
        metrics = evaluate(model, val_set, val_coco)
        row = {"epoch": epoch, "train_loss": total / steps, "updates": updates,
               "nonfinite_loss_steps": bad_loss, "skipped_updates": bad_grad,
               "grad_scale": scaler.get_scale() if amp else None, **metrics}
        print(json.dumps({k: v for k, v in row.items() if k != "confusion"}))
        with (run_dir / "history.jsonl").open("a") as fh:
            fh.write(json.dumps(row) + "\n")
        best = record_best(run_dir, model, metrics["map50_species"], epoch, best, experiment=experiment)
        save_checkpoint(run_dir / "last.pt", model, opt, sched, epoch, best, scaler=scaler,
                        experiment=experiment)
    return run_dir / "best.pt"


In [ ]:
%%writefile /tmp/rpd/rpd_tools/export_onnx.py
"""Export a trained RT-DETR checkpoint to ONNX under the inference contract (docs/inference-contract.md) and check
parity with PyTorch: same kept detections (count and classes), box IoU >= 0.99, score diff <= 1e-3.
Detections are paired by class and box overlap, not by list order, so near-equal scores that the two
runtimes rank differently do not count as a failure.

Export runs in inference mode (eval + no_grad). Exporting outside it froze input-dependent values
into the graph and outputs diverged on any input other than the one used to trace.

Run: python -m tools.export_onnx --checkpoint runs/run-r18/best.pt --out models/rice-pest-r18-1.onnx
"""

import argparse
import hashlib
import json
import sys
from pathlib import Path

import numpy as np
import onnxruntime as ort
import torch

from rice_pest_detector.infer import INPUT_SIZE, postprocess

OPSET = 17
IOU_MIN, SCORE_DIFF_MAX = 0.99, 1e-3


class _ContractOutputs(torch.nn.Module):
    """Expose only the contract's two outputs, under their contract names."""

    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, pixel_values):
        out = self.model(pixel_values=pixel_values)
        return out.logits, out.pred_boxes


class _FrozenPosEncodings:
    """RF-DETR's DINOv2 backbone resizes its position embeddings with antialiased bicubic interpolation,
    which has no ONNX op, and always runs it while tracing. At a fixed input size the result is a constant:
    compute it eagerly (exact training-time values), serve it during the trace, then restore the method."""

    def __init__(self, model, example):
        self.model, self.example, self.patched = model, example, []

    def __enter__(self):
        for module in self.model.modules():
            original = getattr(module, "interpolate_pos_encoding", None)
            if original is None or not callable(original):
                continue
            cache = {}

            def frozen(embeddings, height, width, _original=original, _cache=cache):
                key = (int(height), int(width))  # traced shapes arrive as 0-d tensors
                if key not in _cache:
                    _cache[key] = _original(embeddings, height, width).detach()
                return _cache[key]

            module.interpolate_pos_encoding = frozen
            self.patched.append(module)
        if self.patched:
            with torch.no_grad():
                self.model(pixel_values=self.example)  # eager pass fills the caches
        return self

    def __exit__(self, *exc):
        for module in self.patched:
            del module.interpolate_pos_encoding  # drop the instance attribute; the class method is back
        return False


def export_onnx(model, out: Path, size: int = INPUT_SIZE) -> dict:
    out = Path(out)
    out.parent.mkdir(parents=True, exist_ok=True)
    wrapper = _ContractOutputs(model).eval()
    example = torch.zeros(1, 3, size, size)
    with torch.no_grad(), _FrozenPosEncodings(wrapper.model, example):
        torch.onnx.export(wrapper, (example,), str(out), input_names=["pixel_values"],
                          output_names=["logits", "pred_boxes"], opset_version=OPSET, dynamo=False)
    data = out.read_bytes()
    return {"sha256": hashlib.sha256(data).hexdigest(), "bytes": len(data), "opset": OPSET,
            "input": f"pixel_values (1,3,{size},{size}) float32, [0,1], NCHW", "outputs": ["logits", "pred_boxes"]}


def _iou(a, b) -> float:
    ix = max(0.0, min(a.x1, b.x1) - max(a.x0, b.x0))
    iy = max(0.0, min(a.y1, b.y1) - max(a.y0, b.y0))
    inter = ix * iy
    union = (a.x1 - a.x0) * (a.y1 - a.y0) + (b.x1 - b.x0) * (b.y1 - b.y0) - inter
    return inter / union if union > 0 else 1.0


def _pair(a, b):
    """Pair each ONNX detection with a PyTorch detection of the same class, greedily by highest box IoU.

    Detections are not compared by list position: when scores are nearly equal, the two runtimes can rank the
    same detections in a different order (and break exact ties differently from one platform to another).
    Returns the pairs, or None when the counts differ or a detection has no same-class partner."""
    if len(a) != len(b):
        return None
    candidates = sorted(((_iou(da, db), i, j) for i, da in enumerate(a) for j, db in enumerate(b)
                         if da.class_id == db.class_id), reverse=True)
    used_a, used_b, pairs = set(), set(), []
    for _, i, j in candidates:
        if i not in used_a and j not in used_b:
            used_a.add(i)
            used_b.add(j)
            pairs.append((a[i], b[j]))
    return pairs if len(pairs) == len(a) else None


def parity(model, onnx_path: Path, images: list[np.ndarray], threshold: float = 0.5,
           width: int = INPUT_SIZE, height: int = INPUT_SIZE) -> dict:
    """Compare kept detections from PyTorch and ONNX Runtime on preprocessed (1,3,S,S) inputs."""
    sess = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
    model = model.eval()
    failures, max_score, min_iou, compared = [], 0.0, 1.0, 0
    for i, x in enumerate(images):
        logits, boxes = sess.run(["logits", "pred_boxes"], {"pixel_values": x})
        with torch.no_grad():
            ref = model(pixel_values=torch.from_numpy(x))
        a = postprocess(logits, boxes, width, height, threshold)
        b = postprocess(ref.logits.numpy(), ref.pred_boxes.numpy(), width, height, threshold)
        pairs = _pair(a, b)
        if pairs is None:
            failures.append({"image": i, "reason": f"kept {len(a)} vs {len(b)} or classes differ"})
            continue
        compared += len(pairs)
        # Judge each image on its own pairs; the running max/min are only for the overall report.
        img_score = max((abs(da.score - db.score) for da, db in pairs), default=0.0)
        img_iou = min((_iou(da, db) for da, db in pairs), default=1.0)
        max_score, min_iou = max(max_score, img_score), min(min_iou, img_iou)
        if img_score > SCORE_DIFF_MAX or img_iou < IOU_MIN:
            failures.append({"image": i, "reason": f"score diff {img_score:.2e}, iou {img_iou:.4f}"})
    return {"images": len(images), "threshold": threshold, "max_score_diff": max_score,
            "min_box_iou": min_iou, "detections_compared": compared, "failures": failures}


def load_trained(checkpoint: Path, base_model: str, revision: str):
    """Rebuild the fine-tuned architecture from the base config (no base weights) and load best.pt."""
    from transformers import AutoConfig, AutoModelForObjectDetection

    from rice_pest_detector import CLASS_NAMES
    from rice_pest_detector.train import load_best

    id2label = dict(enumerate(CLASS_NAMES))
    config = AutoConfig.from_pretrained(base_model, revision=revision, num_labels=len(CLASS_NAMES),
                                        id2label=id2label, label2id={v: k for k, v in id2label.items()},
                                        num_queries=300)
    config.use_pretrained_backbone = False
    model = AutoModelForObjectDetection.from_config(config)
    state, meta = load_best(checkpoint)
    model.load_state_dict(state)
    return model.eval(), meta


def main(argv: list[str] | None = None) -> int:
    ap = argparse.ArgumentParser()
    ap.add_argument("--checkpoint", type=Path, required=True)
    ap.add_argument("--out", type=Path, required=True)
    ap.add_argument("--base-model", default="PekingU/rtdetr_r18vd")
    ap.add_argument("--revision", default="ac77a11ff0170a41b771c03264987f8ce2b0d753")
    ap.add_argument("--size", type=int, default=INPUT_SIZE, help="square input size the model was trained at")
    args = ap.parse_args(argv)
    model, meta = load_trained(args.checkpoint, args.base_model, args.revision)
    info = export_onnx(model, args.out, size=args.size)
    info["checkpoint_sha256"] = hashlib.sha256(args.checkpoint.read_bytes()).hexdigest()
    info["checkpoint_epoch"] = meta.get("epoch")
    args.out.with_suffix(".json").write_text(json.dumps(info, indent=1) + "\n", encoding="utf-8")
    print(json.dumps(info, indent=1))
    return 0


if __name__ == "__main__":
    sys.exit(main())


In [ ]:
%%writefile /tmp/rpd/kaggle_driver.py
"""Stage driver for the Kaggle editions, run inside the notebook's own Python environment.

The Kaggle notebook never imports torch, transformers, NumPy or Pillow in its kernel. It builds an isolated
environment from the committed hash lock and runs each stage as a child process:

    <env python> -u kaggle_driver.py --stage <name> --work /tmp/rpd --run-dir /kaggle/working/run-<variant>

The module itself imports only the standard library at the top; each stage imports what it needs, so the
helpers can be tested without torch. The package (`rice_pest_detector`) and `rpd_tools/export_onnx.py` are
the repository's own files, written by the notebook into the work folder.
"""

import argparse
import csv
import datetime
import hashlib
import json
import os
import platform
import shutil
import sys
import time
from pathlib import Path

RESUME_FILES = ("last.pt", "best.pt", "best.json", "history.jsonl", "config.json")
IMAGE_SUFFIXES = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp", ".gif")
ARCHIVE_SUFFIXES = (".zip", ".tar", ".gz", ".tgz", ".7z")
OUTPUT_BUDGET_BYTES = 5 * 1000**3
SPLIT_SIZES = {"train": 4948, "val": 253, "test": 252}
MAX_QUERIES = 300
RELOAD_PHOTOS = 10
INFERENCE_PHOTOS = (10, 22)  # validation records 10-21, after the reload photos
CROWDED = 30
UV_VERSION = "0.12.15"
LOCK_NAME = "requirements-kaggle.lock.txt"


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def write_json(path: Path, obj) -> None:
    path = Path(path)
    tmp = path.with_name(path.name + ".tmp")
    tmp.write_text(json.dumps(obj, indent=1) + "\n", encoding="utf-8")
    tmp.replace(path)


def read_json(path: Path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


class Context:
    """Paths and settings of one run; `edition.json` is written by the notebook's `paths` cell."""

    def __init__(self, work: Path, run_dir: Path, input_root: Path = Path("/kaggle/input")):
        self.work, self.run_dir, self.input_root = Path(work), Path(run_dir), Path(input_root)
        self.edition = read_json(self.work / "edition.json")
        self.images = self.work / "images"
        self.data = self.work / "data"

    def manifest(self) -> dict:
        return read_json(self.data / "split_manifest.json")

    def records(self, split: str) -> list[dict]:
        return [r for r in self.manifest()["images"] if r["split"] == split]


# ---------------------------------------------------------------- fetch


def pins_ok(path: Path, size: int, md5: str, sha256: str) -> bool:
    path = Path(path)
    if not path.is_file() or path.stat().st_size != size:
        return False
    h_md5, h_sha = hashlib.md5(), hashlib.sha256()
    with path.open("rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h_md5.update(chunk)
            h_sha.update(chunk)
    return h_md5.hexdigest() == md5 and h_sha.hexdigest() == sha256


def find_attached_archive(root: Path, size: int, md5: str, sha256: str) -> Path | None:
    """A reader's own attached `datasets.zip` anywhere under `root`, used only if it passes all three pins."""
    root = Path(root)
    if not root.is_dir():
        return None
    for candidate in sorted(root.rglob("datasets.zip")):
        if pins_ok(candidate, size, md5, sha256):
            return candidate
        print(f"ignoring {candidate}: it does not match the Zenodo size, MD5 and SHA-256")
    return None


def stage_fetch(ctx: Context) -> None:
    from rice_pest_detector import fetch

    pins = (fetch.ZENODO_BYTES, fetch.ZENODO_MD5, fetch.ZENODO_SHA256)
    archive = find_attached_archive(ctx.input_root, *pins)
    if archive is not None:
        print("using the attached archive (size, MD5 and SHA-256 match the Zenodo record)")
    else:
        dest = ctx.work / "datasets.zip"
        if pins_ok(dest, *pins):
            archive = dest
            print("verified archive already in scratch, skipping download")
        else:
            print(f"downloading the dataset archive from Zenodo ({fetch.ZENODO_BYTES / 1e9:.2f} GB) to scratch")
            try:
                archive = fetch.download(fetch.ZENODO_URL, dest, *pins)
            except ValueError as exc:
                raise SystemExit(f"Download check failed (size, MD5 or SHA-256): {exc}. Nothing was trained; "
                                 "run the notebook again later.") from None
            except OSError as exc:
                raise SystemExit(f"Download from Zenodo failed: {exc}. Check that Internet is on in Settings "
                                 "and that zenodo.org is reachable, then run again.") from None
    n = fetch.extract_members(archive, ctx.manifest(), ctx.images)
    print(n, "images extracted to scratch and verified against the committed split manifest (SHA-256)")
    shutil.copy(ctx.data / "annotations_val.json", ctx.run_dir / "annotations_val.json")


# ---------------------------------------------------------------- validate


def validation_problems(manifest: dict, image_sizes: dict[str, tuple[int, int]] | None = None) -> tuple[dict, list]:
    """Checks made before training. Returns (table, problems); `image_sizes` maps file -> (width, height)."""
    records = manifest["images"]
    problems = []
    sizes = {s: sum(r["split"] == s for r in records) for s in SPLIT_SIZES}
    if sizes != SPLIT_SIZES:
        problems.append(f"split sizes {sizes} != {SPLIT_SIZES}")
    names = [r["file"] for r in records]
    duplicates = sorted({n for n in names if names.count(n) > 1}) if len(set(names)) != len(names) else []
    if duplicates:
        problems.append(f"duplicate file names in the manifest: {duplicates[:5]}")
    classes = sorted({b[0] for r in records for b in r["boxes"]})
    if any(c not in (0, 1, 2, 3) for c in classes):
        problems.append(f"class ids outside 0-3: {classes}")
    most = max((len(r["boxes"]) for r in records), default=0)
    if most > MAX_QUERIES:
        problems.append(f"most crowded image has {most} insects, more than the model's {MAX_QUERIES} queries")
    checked = 0
    if image_sizes is not None:
        for r in records:
            seen = image_sizes.get(r["file"])
            if seen is None:
                problems.append(f"image missing: {r['relative_path']}")
            elif tuple(seen) != (r["width"], r["height"]):
                problems.append(f"{r['file']}: size {seen} != manifest {(r['width'], r['height'])}")
            else:
                checked += 1
    table = {"split_sizes": sizes, "unique_file_names": not duplicates, "class_ids": classes,
             "max_insects_per_image": most, "queries": MAX_QUERIES, "images_size_checked": checked}
    return table, problems


def stage_validate(ctx: Context) -> None:
    from PIL import Image

    manifest = ctx.manifest()
    seen = {}
    for r in manifest["images"]:
        path = ctx.images / r["relative_path"]
        if path.is_file():
            with Image.open(path) as img:
                img.verify()
            with Image.open(path) as img:
                seen[r["file"]] = img.size
    table, problems = validation_problems(manifest, seen)
    write_json(ctx.run_dir / "validate.json", {**table, "problems": problems})
    print("| check | value |\n| :-- | :-- |")
    for key, value in table.items():
        print(f"| {key} | {value} |")
    if problems:
        raise SystemExit("Data validation failed: " + "; ".join(problems[:10]))
    print("data validation: all checks passed")


# ---------------------------------------------------------------- resume


def resume_from(source: str, run_dir: Path) -> dict:
    """Copy an earlier run's files (attached by the reader) into `run_dir`. Empty `source` does nothing."""
    run_dir = Path(run_dir)
    if not source or not str(source).strip():
        print("resume: off (RESUME_FROM is empty); training starts from the base model")
        return {"resumed": False}
    found = sorted(Path(source).rglob("last.pt"))
    if not found:
        raise SystemExit(f"resume: no last.pt found under {source}; attach an earlier run's output or leave "
                         "RESUME_FROM empty")
    origin = found[0].parent
    print("WARNING: a .pt checkpoint is loaded with PyTorch's pickle-based format, so loading it can run code. "
          "Only resume from a run you produced yourself.")
    copied = []
    for name in RESUME_FILES:
        if (origin / name).is_file():
            shutil.copy(origin / name, run_dir / name)
            copied.append(name)
    print("resume: copied", ", ".join(copied), "from", origin)
    record = {"resumed": True, "from": str(origin), "files": copied}
    write_json(run_dir / "resume.json", record)
    return record


def stage_resume(ctx: Context, source: str) -> None:
    resume_from(source, ctx.run_dir)


# ---------------------------------------------------------------- model helpers


def train_config(edition: dict):
    from rice_pest_detector.train import TrainConfig

    extra = {"input_size": edition["input_size"]} if edition["input_size"] != 640 else {}
    return TrainConfig(base_model=edition["base_model"], revision=edition["revision"],
                       batch_size=edition["batch_size"], epochs=edition["epochs"], **extra)


def export_module():
    """`rpd_tools.export_onnx` in the notebook's work folder; the repository's `tools.export_onnx` in tests."""
    try:
        from rpd_tools import export_onnx
    except ModuleNotFoundError as exc:
        if exc.name not in ("rpd_tools", "rpd_tools.export_onnx"):
            raise
        from tools import export_onnx
    return export_onnx


def load_selected(ctx: Context):
    """The selected checkpoint (best.pt) on the architecture rebuilt from the base config, no base weights."""
    if ctx.edition["base_model"] == "__tiny__":  # tests only
        from rice_pest_detector.train import build_model, load_best

        model = build_model(train_config(ctx.edition))
        state, meta = load_best(ctx.run_dir / "best.pt")
        model.load_state_dict(state)
        return model.eval(), meta
    return export_module().load_trained(ctx.run_dir / "best.pt", ctx.edition["base_model"], ctx.edition["revision"])


def device():
    import torch

    return "cuda" if torch.cuda.is_available() else "cpu"


# ---------------------------------------------------------------- fit


def stage_fit(ctx: Context) -> None:
    from rice_pest_detector.train import fit

    cfg = train_config(ctx.edition)
    print("recipe:", json.dumps({k: getattr(cfg, k) for k in ("base_model", "revision", "epochs", "batch_size",
                                                               "lr", "backbone_lr", "input_size", "seed", "fp16")}))
    best = fit(cfg, ctx.manifest(), ctx.images, ctx.run_dir)
    print("selected checkpoint:", best, read_json(ctx.run_dir / "best.json"))


# ---------------------------------------------------------------- evaluate


def evaluate_with_detections(model, dataset, coco: dict, threshold: float) -> tuple[dict, list[dict]]:
    """Run the repository's own `train.evaluate` and keep the raw detections it scores (no second copy)."""
    from rice_pest_detector import train

    captured = []
    original = train.map50

    def capturing(gt, dets, group_species):
        if group_species:
            captured[:] = [dict(d) for d in dets]
        return original(gt, dets, group_species)

    train.map50 = capturing
    try:
        metrics = train.evaluate(model, dataset, coco, threshold=threshold)
    finally:
        train.map50 = original
    return metrics, captured


def headline(metrics: dict) -> dict:
    by_class = {k: v for k, v in metrics["map50_by_class"].items() if k != "all"}
    errors = metrics["count_errors"]
    return {"map50_species": metrics["map50_species"],
            "map50_four_class": sum(by_class.values()) / len(by_class),
            "map50_by_class": by_class,
            "count_accuracy": metrics["count_accuracy"],
            "count_accuracy_no_detection_baseline": metrics["count_accuracy_no_detection_baseline"],
            "mae": {"rbb": errors["RBB"]["mae"], "wsb": errors["WSB"]["mae"]},
            "wsb30_mae": errors["WSB"]["mae_by_density"]["30+"]["mae"],
            "wsb30_images": errors["WSB"]["mae_by_density"]["30+"]["images"]}


def stage_evaluate(ctx: Context) -> None:
    from rice_pest_detector.train import ManifestDataset

    model, meta = load_selected(ctx)
    model = model.to(device())
    threshold = ctx.edition["threshold"]
    dataset = ManifestDataset(ctx.manifest(), ctx.images, split="val", size=ctx.edition["input_size"])
    coco = read_json(ctx.data / "annotations_val.json")
    metrics, dets = evaluate_with_detections(model, dataset, coco, threshold)
    files = {i: r["file"] for i, r in enumerate(dataset.records, start=1)}
    write_json(ctx.run_dir / "val_metrics.json", {**metrics, "headline": headline(metrics), "epoch": meta.get("epoch"),
                                                  "label": "tutorial evidence: one validation pass at the "
                                                           "published threshold"})
    write_json(ctx.run_dir / "val_detections.json",
               {"threshold": threshold, "score_floor": 0.0,
                "images": [{"image_id": i, "file": files[i],
                            "detections": [{k: d[k] for k in ("category_id", "score", "bbox")}
                                           for d in dets if d["image_id"] == i]} for i in files]})
    h = headline(metrics)
    print(f"Validation, one pass at threshold {threshold} (tutorial evidence, not the recorded out-of-fold figure)")
    print(f"  species mAP50 {h['map50_species']:.3f}; four-class mAP50 {h['map50_four_class']:.3f}")
    print("  per class:", ", ".join(f"{k} {v:.3f}" for k, v in h["map50_by_class"].items()))
    print(f"  count accuracy {h['count_accuracy']:.3f} (count-nothing baseline "
          f"{h['count_accuracy_no_detection_baseline']:.3f})")
    print(f"  count MAE RBB {h['mae']['rbb']:.2f}, WSB {h['mae']['wsb']:.2f}; crowded trays (WSB {CROWDED}+, "
          f"{h['wsb30_images']} images) MAE {h['wsb30_mae']}")


# ---------------------------------------------------------------- export and reload


def stage_export(ctx: Context) -> None:
    model, meta = load_selected(ctx)
    info = export_module().export_onnx(model.cpu().eval(), ctx.run_dir / "model.onnx", size=ctx.edition["input_size"])
    info.update(checkpoint_sha256=sha256_file(ctx.run_dir / "best.pt"), checkpoint_epoch=meta.get("epoch"),
                selection="best validation species mAP50", base_model=ctx.edition["base_model"],
                revision=ctx.edition["revision"], input_size=ctx.edition["input_size"])
    write_json(ctx.run_dir / "export_record.json", info)
    print(json.dumps(info, indent=1))


def onnx_detect(model_path: Path, records: list[dict], image_root: Path, size: int, threshold: float) -> list:
    """ONNX Runtime detections per record; used by the fresh `python -I` process and by inference."""
    import onnxruntime as ort
    from PIL import Image, ImageOps

    from rice_pest_detector.infer import postprocess, preprocess

    session = ort.InferenceSession(str(model_path), providers=["CPUExecutionProvider"])
    out = []
    for r in records:
        with Image.open(Path(image_root) / r["relative_path"]) as img:
            width, height = ImageOps.exif_transpose(img).size
            x = preprocess(img, size)
        logits, boxes = session.run(["logits", "pred_boxes"], {"pixel_values": x})
        out.append([d.__dict__ for d in postprocess(logits, boxes, width, height, threshold)])
    return out


def torch_detect(model, records: list[dict], image_root: Path, size: int, threshold: float) -> list:
    import torch
    from PIL import Image, ImageOps

    from rice_pest_detector.infer import postprocess, preprocess

    model = model.cpu().eval()
    out = []
    for r in records:
        with Image.open(Path(image_root) / r["relative_path"]) as img:
            width, height = ImageOps.exif_transpose(img).size
            x = preprocess(img, size)
        with torch.no_grad():
            ref = model(pixel_values=torch.from_numpy(x))
        out.append([d.__dict__ for d in postprocess(ref.logits.float().numpy(), ref.pred_boxes.float().numpy(),
                                                    width, height, threshold)])
    return out


def compare_detections(onnx_dets: list, torch_dets: list, score_tolerance_enforced: bool) -> dict:
    """`parity()`'s pairing and tolerances, applied per photo; returns the report with its verdict."""
    from rice_pest_detector.infer import Detection
    ex = export_module()
    IOU_MIN, SCORE_DIFF_MAX, _iou, _pair = ex.IOU_MIN, ex.SCORE_DIFF_MAX, ex._iou, ex._pair

    photos, failures, compared, max_score, min_iou = [], [], 0, 0.0, 1.0
    for i, (a_raw, b_raw) in enumerate(zip(onnx_dets, torch_dets, strict=True)):
        a, b = [Detection(**d) for d in a_raw], [Detection(**d) for d in b_raw]
        pairs = _pair(a, b)
        if pairs is None:
            failures.append({"photo": i, "check": "kept detections",
                             "detail": f"ONNX kept {len(a)}, PyTorch kept {len(b)}, or classes differ"})
            photos.append({"photo": i, "onnx": len(a), "torch": len(b), "paired": False})
            continue
        score = max((abs(x.score - y.score) for x, y in pairs), default=0.0)
        iou = min((_iou(x, y) for x, y in pairs), default=1.0)
        compared += len(pairs)
        max_score, min_iou = max(max_score, score), min(min_iou, iou)
        photos.append({"photo": i, "onnx": len(a), "torch": len(b), "paired": True,
                       "max_score_diff": score, "min_iou": iou})
        if iou < IOU_MIN:
            failures.append({"photo": i, "check": "box overlap", "detail": f"min IoU {iou:.4f} < {IOU_MIN}"})
        if score > SCORE_DIFF_MAX:
            entry = {"photo": i, "check": "score difference", "detail": f"{score:.2e} > {SCORE_DIFF_MAX}"}
            (failures if score_tolerance_enforced else photos[-1].setdefault("waived", [])).append(entry)
    return {"photos": len(photos), "detections_compared": compared, "max_score_diff": max_score,
            "min_iou": min_iou, "iou_min": IOU_MIN, "score_diff_max": SCORE_DIFF_MAX,
            "score_tolerance_enforced": score_tolerance_enforced, "per_photo": photos, "failures": failures,
            "verdict": "FAIL" if failures else "PASS"}


def judge_reload(report: dict) -> None:
    if report["failures"]:
        first = report["failures"][0]
        raise RuntimeError(f"fresh-process ONNX reload check failed: {first['check']} on photo {first['photo']} "
                           f"({first['detail']}); {len(report['failures'])} failure(s) in reload_check.json")


def stage_reload(ctx: Context) -> None:
    import subprocess

    records = ctx.records("val")[:RELOAD_PHOTOS]
    size, threshold = ctx.edition["input_size"], ctx.edition["threshold"]
    model, _ = load_selected(ctx)
    torch_dets = torch_detect(model, records, ctx.images, size, threshold)
    onnx_out = ctx.run_dir / "reload_onnx_detections.json"
    # A fresh, isolated interpreter (-I) that loads only model.onnx: nothing from this process carries over.
    subprocess.run([sys.executable, "-I", str(Path(__file__).resolve()), "--stage", "onnx-detect",
                    "--work", str(ctx.work), "--run-dir", str(ctx.run_dir), str(onnx_out)], check=True)
    onnx_dets = read_json(onnx_out)
    report = compare_detections(onnx_dets, torch_dets, ctx.edition["score_tolerance_enforced"])
    report.update(threshold=threshold, files=[r["file"] for r in records],
                  waiver=ctx.edition.get("score_waiver"))
    write_json(ctx.run_dir / "reload_check.json", report)
    print(f"fresh-process ONNX reload check: {report['detections_compared']} detections on {report['photos']} "
          f"photos; max score difference {report['max_score_diff']:.2e}; min box IoU {report['min_iou']:.5f}; "
          f"score tolerance {'enforced' if report['score_tolerance_enforced'] else 'waived: ' + str(report['waiver'])}")
    judge_reload(report)
    print("reload check: PASS (the ONNX file loaded from disk in a fresh process gives the same kept detections)")


def stage_onnx_detect(ctx: Context, out: str) -> None:
    records = ctx.records("val")[:RELOAD_PHOTOS]
    dets = onnx_detect(ctx.run_dir / "model.onnx", records, ctx.images, ctx.edition["input_size"],
                       ctx.edition["threshold"])
    write_json(Path(out), dets)


# ---------------------------------------------------------------- inference


def review_reason(scores: list[float], review_below: float, crowded: int = CROWDED) -> str:
    reasons = []
    if any(s < review_below for s in scores):
        reasons.append(f"a counted insect scored below {review_below}")
    if len(scores) > crowded:
        reasons.append(f"more than {crowded} insects")
    return "; ".join(reasons)


def stage_inference(ctx: Context) -> None:
    from rice_pest_detector import SPECIES_OF_CLASS

    lo, hi = INFERENCE_PHOTOS
    records = ctx.records("val")[lo:hi]
    dets = onnx_detect(ctx.run_dir / "model.onnx", records, ctx.images, ctx.edition["input_size"],
                       ctx.edition["threshold"])
    rows = []
    for r, found in zip(records, dets, strict=True):
        pred = [SPECIES_OF_CLASS[d["class_id"]] for d in found]
        true = [SPECIES_OF_CLASS[b[0]] for b in r["boxes"]]
        reason = review_reason([d["score"] for d in found], ctx.edition["review_below"])
        rows.append({"file": r["file"], "pred_rbb": pred.count("RBB"), "pred_wsb": pred.count("WSB"),
                     "true_rbb": true.count("RBB"), "true_wsb": true.count("WSB"),
                     "review_flag": bool(reason), "review_reason": reason})
    with (ctx.run_dir / "predictions_val_sample.csv").open("w", newline="", encoding="utf-8") as fh:
        writer = csv.DictWriter(fh, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)
    for row in rows:
        print(row)


# ---------------------------------------------------------------- test look


def stage_test_look(ctx: Context) -> None:
    from rice_pest_detector.train import (
        ManifestDataset,
        _atomic_write_text,
        check_cached_test,
        evaluate,
        evaluation_id,
    )

    threshold = ctx.edition["threshold"]  # the published threshold, from the model's manifest
    coco_path = ctx.data / "annotations_test.json"
    best, manifest = ctx.run_dir / "best.pt", ctx.manifest()
    identity = evaluation_id(best, manifest, coco_path, threshold)
    out = ctx.run_dir / "test_metrics_reproduction.json"
    print("The test split's looks are spent; this is a reproduction and must not be used to choose anything.")
    if out.exists():
        prior = read_json(out)
        check_cached_test(prior, identity)
        print(json.dumps(prior, indent=1))
        return
    model, meta = load_selected(ctx)
    test_set = ManifestDataset(manifest, ctx.images, split="test", size=ctx.edition["input_size"])
    metrics = evaluate(model.to(device()), test_set, read_json(coco_path), threshold=threshold)
    metrics = {**metrics, "label": "reproduction of an already-spent test look; not for choosing anything",
               "best_pt_sha256": identity["best_pt_sha256"], "epoch": meta.get("epoch"), "evaluation": identity}
    _atomic_write_text(out, json.dumps(metrics, indent=1))
    print(json.dumps(metrics, indent=1))


# ---------------------------------------------------------------- env log and summary


def versions() -> dict:
    import numpy
    import onnxruntime
    import PIL
    import torch
    import transformers

    cuda = torch.cuda.is_available()
    return {"python": platform.python_version(), "torch": torch.__version__, "cuda": torch.version.cuda,
            "transformers": transformers.__version__, "pillow": PIL.__version__, "numpy": numpy.__version__,
            "onnxruntime": onnxruntime.__version__,
            "gpu": torch.cuda.get_device_name(0) if cuda else None,
            "gpu_count": torch.cuda.device_count() if cuda else 0}


def stage_env_log(ctx: Context) -> None:
    from importlib import metadata

    stamp = datetime.datetime.now(datetime.UTC).strftime("%Y%m%dT%H%M%SZ")
    v = versions()
    freeze = sorted(f"{d.metadata['Name']}=={d.version}" for d in metadata.distributions())
    lines = [f"utc: {stamp}", f"python: {sys.version}", *(f"{k}: {val}" for k, val in v.items()),
             "--- packages in the environment ---", *freeze]
    (ctx.run_dir / f"env-{stamp}.txt").write_text("\n".join(lines) + "\n", encoding="utf-8")
    print(json.dumps(v, indent=1))
    print("environment log written to", f"env-{stamp}.txt")


def output_listing(run_dir: Path) -> tuple[list[tuple[str, int]], int, list[str]]:
    """Every file under `run_dir` with its size, the total, and problems (budget, images, archives)."""
    run_dir = Path(run_dir)
    files = sorted((p.relative_to(run_dir).as_posix(), p.stat().st_size) for p in run_dir.rglob("*") if p.is_file())
    total = sum(size for _, size in files)
    problems = [f"image file in the output: {name}" for name, _ in files if name.lower().endswith(IMAGE_SUFFIXES)]
    problems += [f"archive in the output: {name}" for name, _ in files if name.lower().endswith(ARCHIVE_SUFFIXES)]
    if total > OUTPUT_BUDGET_BYTES:
        problems.append(f"outputs total {total / 1e9:.2f} GB, over the {OUTPUT_BUDGET_BYTES / 1e9:.0f} GB budget")
    return files, total, problems


def _artifact(path: Path) -> dict:
    return {"bytes": path.stat().st_size, "sha256": sha256_file(path)} if path.is_file() else None


def build_summary(ctx: Context, runtime: dict) -> dict:
    run = ctx.run_dir
    files, total, problems = output_listing(run)
    if problems:
        raise RuntimeError("output check failed: " + "; ".join(problems))
    times = read_json(run / "stage_times.json") if (run / "stage_times.json").exists() else {}
    history = [json.loads(ln) for ln in (run / "history.jsonl").read_text().splitlines() if ln.strip()]
    best = read_json(run / "best.json")
    resume = read_json(run / "resume.json") if (run / "resume.json").exists() else {"resumed": False}
    val = read_json(run / "val_metrics.json")["headline"]
    reload_check = read_json(run / "reload_check.json")
    lock = ctx.work / LOCK_NAME
    durations = {k: times.get(k) for k in ("environment", "fetch", "fit", "evaluate", "export", "reload")}
    durations["total"] = round(sum(v for v in times.values() if isinstance(v, (int, float))), 1)
    return {
        "schema": 1, "status": "PASS", "variant": ctx.edition["variant"],
        "model": {k: ctx.edition[k] for k in ("base_model", "revision", "threshold", "input_size")},
        "environment": {"uv": UV_VERSION, "lock_sha256": sha256_file(lock) if lock.exists() else None,
                        "build_seconds": times.get("environment")},
        "runtime": runtime, "durations_s": durations,
        "training": {"epochs": ctx.edition["epochs"], "epochs_run_this_session": len(history) if not
                     resume["resumed"] else None, "resumed": resume["resumed"], "best_epoch": best["epoch"],
                     "best_val_map50_species": best["map50_species"],
                     "skipped_updates_total": sum(r.get("skipped_updates", 0) for r in history)},
        "validation": val,
        "artifacts": {name: _artifact(run / name) for name in ("best.pt", "model.onnx")},
        "reload_check": {"photos": reload_check["photos"], "detections_compared": reload_check["detections_compared"],
                         "max_score_diff": reload_check["max_score_diff"], "min_iou": reload_check["min_iou"],
                         "score_tolerance_enforced": reload_check["score_tolerance_enforced"],
                         "verdict": reload_check["verdict"]},
        "outputs_total_bytes": total,
        "outputs": [{"file": name, "bytes": size} for name, size in files],
        "test_look": "ran (reproduction)" if (run / "test_metrics_reproduction.json").exists() else "skipped",
    }


def stage_summary(ctx: Context) -> None:
    summary = build_summary(ctx, versions())
    write_json(ctx.run_dir / "run_summary.json", summary)
    v, t, r = summary["validation"], summary["training"], summary["reload_check"]
    onnx = summary["artifacts"]["model.onnx"]
    print(f"Run summary: {summary['variant']} ({summary['model']['base_model']}), status {summary['status']}")
    print(f"  GPU {summary['runtime']['gpu']}; torch {summary['runtime']['torch']}; "
          f"transformers {summary['runtime']['transformers']}")
    print(f"  best epoch {t['best_epoch']} (validation species mAP50 {t['best_val_map50_species']:.3f})")
    print(f"  validation at {summary['model']['threshold']}: species mAP50 {v['map50_species']:.3f}, count accuracy "
          f"{v['count_accuracy']:.3f} (baseline {v['count_accuracy_no_detection_baseline']:.3f})")
    print(f"  model.onnx {onnx['bytes']:,} bytes, SHA-256 {onnx['sha256']}")
    print(f"  fresh-process ONNX reload check: {r['verdict']} ({r['detections_compared']} detections, "
          f"max score difference {r['max_score_diff']:.2e}, min IoU {r['min_iou']:.5f})")
    print(f"  test split: {summary['test_look']}")
    print(f"  outputs: {summary['outputs_total_bytes'] / 1e9:.2f} GB in {ctx.run_dir}")
    for item in summary["outputs"]:
        print(f"    {item['bytes']:>14,}  {item['file']}")


STAGES = {"fetch": stage_fetch, "validate": stage_validate, "fit": stage_fit, "evaluate": stage_evaluate,
          "export": stage_export, "reload": stage_reload, "inference": stage_inference,
          "test-look": stage_test_look, "summary": stage_summary, "env-log": stage_env_log}


def main(argv: list[str] | None = None) -> int:
    parser = argparse.ArgumentParser(description=__doc__.splitlines()[0])
    parser.add_argument("--stage", required=True, choices=sorted([*STAGES, "resume", "onnx-detect"]))
    parser.add_argument("--work", type=Path, required=True)
    parser.add_argument("--run-dir", type=Path, required=True)
    parser.add_argument("--input-root", type=Path, default=Path("/kaggle/input"))
    parser.add_argument("options", nargs="*")
    args = parser.parse_args(argv)
    sys.path.insert(0, str(args.work))  # the carried package and rpd_tools, written by the notebook
    os.environ.setdefault("HF_HUB_DISABLE_TELEMETRY", "1")
    ctx = Context(args.work, args.run_dir, args.input_root)
    started = time.time()
    if args.stage == "resume":
        stage_resume(ctx, args.options[0] if args.options else "")
    elif args.stage == "onnx-detect":
        stage_onnx_detect(ctx, args.options[0])
    else:
        STAGES[args.stage](ctx)
    print(f"[{args.stage}] done in {time.time() - started:.1f} s")
    return 0


if __name__ == "__main__":
    sys.exit(main())


In [ ]:
%%writefile /tmp/rpd/requirements-kaggle.lock.txt
# This file was autogenerated by uv via the following command:
#    uv pip compile notebooks/kaggle/requirements-kaggle.in --python-version 3.12 --python-platform x86_64-manylinux_2_28 --generate-hashes --only-binary :all: -o notebooks/kaggle/requirements-kaggle.lock.txt
annotated-doc==0.0.5 \
    --hash=sha256:117bac03a25ede5df5440e855b32d556049ca169ead221505badf432fed4b101 \
    --hash=sha256:c7e58ce09192557605d8bbd92836d7e1d520ac9580096042c0bfd197efacf1bb
    # via typer
anyio==4.15.1 \
    --hash=sha256:6152fdbbf9a77fdec97731721bebf7c4c44f7c29b424b0065826173efc7ed101 \
    --hash=sha256:9f28306018cbd6d329e64a36d58256edff76dd996fe423bc957326e578b82a94
    # via httpx2
click==8.5.0 \
    --hash=sha256:255bc9599cf7748b4b1a446ccc735421bd08a2ae529a8b88597d3de5664ee360 \
    --hash=sha256:ba0d2089de75ea0310e2dde03160e6ca10009947fb95a182f9b54021bb272e34
    # via huggingface-hub
cuda-bindings==13.4.4 \
    --hash=sha256:03a8fd2a25530dbae26ecd4462d418156028e35b3f3b081440ff6368506e10eb \
    --hash=sha256:0c3f4cc887eac0d5edb5d7396cc78b336bee151a97e2ba7acf59e8b9380132d0 \
    --hash=sha256:0ce9208c7cf03a947ac11553b08ee2fca594ac63d34360639a9060980f546a62 \
    --hash=sha256:0d714d5a85174530b58b39a8e06664ebc6296e62a90d66b0b2eb4dbdfaaabe5b \
    --hash=sha256:433efa31e33c2868621b5f6036bd3bf5de9c45c5546cb1361a5f6ff3415b8bb5 \
    --hash=sha256:4a429132f06d0c3a853aa419aabe4c0a7bfc3f3886fb4c336d81729f63754d31 \
    --hash=sha256:50bc70ef3b225f4794f5b374773629836bbfdf4a21fc975ac25e7daad67494b8 \
    --hash=sha256:5d6adc44f098472d8ac2d6211923eea6081830521852dc77f8c9237bf9320a59 \
    --hash=sha256:6781fe08ed5f4b5163a3c6577e079bc28955811382d35995c916bf052ee0994e \
    --hash=sha256:680e70dcb8b61fdae16d3bb4876c41d55aadc9fe8b3825ea567f2f2c85606b0a \
    --hash=sha256:76dafd37ff5191f183ea9c1d981f7bb541df5c0bec502bcc4cc84a3eea706b40 \
    --hash=sha256:787936778fb69a66afa5d7769086640cbf938a3b83f2b8171174a46d115e87ce \
    --hash=sha256:85d961133658fff3167cfcf69de1beac8e46453c15fd4bcaa4a96d3ef1110c69 \
    --hash=sha256:87899d738c2c093821b8306e96585fbceb3a9d69a1e1598b9e4639aabf9548ce \
    --hash=sha256:90f85a00fc89dd4a8f7c4beb0a2f3f788b845c8496fe84adc1217830206db9e0 \
    --hash=sha256:939d2672199048649fd2118e4df9df770a517711e50d25c05f880f3606b78142 \
    --hash=sha256:93ad74e8181a89e7886be9d06414a772701081c9edcf6c84d9fc1ccda50f6767 \
    --hash=sha256:978927d7b911e80fa5a7308284c39c79b1182c48f47ed62717fc7f5c22f6ded4 \
    --hash=sha256:a4c97f8430d7f7c4a4d0f245510d26647dd81ea3fc65941f7a815ad395cbc30e \
    --hash=sha256:a8109e8ca1e40b666fc94d7bc4a172dfb99746a7d3fd933ee17b3f36f48f24ab \
    --hash=sha256:b346bfe1dda49537537c06cc4727b2210d949d8aee883cb591bca46581629d00 \
    --hash=sha256:b8b2e4235b552edb42cdbe39de7a9259ceec631e21c7344b76177db9cc6c58b4 \
    --hash=sha256:baa0f5f1f9a05032e7deb97051e85aa18b4e3c6db914f1c813bb2f02e5c72db4 \
    --hash=sha256:caba27cd9a5f910aaadb54878f52ad91be1abbe3b3cfb955c999709c1a1798b0 \
    --hash=sha256:cc1acee5720dbe077777fe540eee84ec4547286bbed3cba24d1c764d4b930317 \
    --hash=sha256:cc8f35176be380fa241dfe9b8583a16db9c4b043fecb4b2932898e1699e5a0fb \
    --hash=sha256:d9e00ee968781e06ab09f8c87f75586f04829988d66b4a0d6efbccfd03576b71 \
    --hash=sha256:db0cdac6a8d8bd72f3dcb755ad4fcc904e0bd75a30091422fedbcfa9fc06b09a \
    --hash=sha256:e1cef1eaf5a08cc3bf3675ca9d3f97f3fc7085998556c848b4053e74ba4ba392 \
    --hash=sha256:f5bf66473514863aeede6cdefd887e6e22d74dbbac0933cd166ab4498935d04e \
    --hash=sha256:f8ac692e0377c51d0bda32ae6782caafa007a5de171571ac844e616391c89a16
    # via torch
cuda-pathfinder==1.8.3 \
    --hash=sha256:e29e59829c297a7a5233bd9cc71094fc5bddbd076951482670178f9eade39b1f
    # via cuda-bindings
cuda-toolkit==13.0.3.0 \
    --hash=sha256:d693caaa261214ddd7dbb60d68e71cbed884e68c2be7509778f3051da0b91c3f
    # via torch
filelock==4.2.4 \
    --hash=sha256:301241438c57936ef93f4243a94f2559022268bfcf9b375a9a72eb0f8db8358e \
    --hash=sha256:c6d8663535e1650ee430507b4ffbac7b464dd64b7647ea0112a9f406499d163d
    # via
    #   huggingface-hub
    #   torch
flatbuffers==25.12.19 \
    --hash=sha256:7634f50c427838bb021c2d66a3d1168e9d199b0607e6329399f04846d42e20b4
    # via onnxruntime
fsspec==2026.9.0 \
    --hash=sha256:0f08147951c8cb31d844c3547d631053b127863b60be04cf06e121333ee0e2fe \
    --hash=sha256:8dd6e646e99ea382bd85f97a45e6b526a442d79423a7dc673f1e2756d05fcb5f
    # via
    #   huggingface-hub
    #   torch
h11==0.16.0 \
    --hash=sha256:4e35b956cf45792e4caa5885e69fba00bdbc6ffafbfa020300e549b208ee5ff1 \
    --hash=sha256:63cf8bbe7522de3bf65932fda1d9c2772064ffb3dae62d55932da54b31cb6c86
    # via httpcore2
hf-xet==1.7.0 \
    --hash=sha256:0a9e802f33bf50c851abe45fc5380e61f959e2d369647d6742b79ad9d6c27cab \
    --hash=sha256:19c0e64f14175ccb6a1aff69e0d2ab9ec5269a560e6687abaf2b3fa4f73de7cd \
    --hash=sha256:2814a6e999d13464c4d679b788cc5d784eb5a4edfc638a31f10e9a11ab531ef8 \
    --hash=sha256:2b7bb5727889b0f2436dbaaad8fc4c3e66b8240d992716989e0c086b4278b1bc \
    --hash=sha256:4ee5e05a627f5ab5bad7a86582277d645556ea1e199903aae19e033a392aa13a \
    --hash=sha256:57bc157b8b7fe3bee9dcb9af7f3da8de41801c3b31a9ef68a77a33c6a6be382f \
    --hash=sha256:59fba37039233c7fcbe196817d6cdcf1b40dfb17b410f229d85b0cf0a1848da4 \
    --hash=sha256:757168feb5679647c0bb13ee5d0faebe799c4dff9051419885a566ebd79f949d \
    --hash=sha256:80f79dae613ce9e0ea1fd1ae15616ca9ac74aed4c770aabc199c4f03ebecc863 \
    --hash=sha256:87dab080f8f7d32781c2586904e3603f4e60d09bfc727706c3ae419e0829beeb \
    --hash=sha256:acc3851cf2576a8fb2ae926da863f4efabe21303cf292e9a44332802ab0dcc6a \
    --hash=sha256:b01fe18dbbd151a2403d2c64ed30dc6547b00d6babab9a617d77c7acdb81ee66 \
    --hash=sha256:b91569d5f1b61c34b043687da02c05dd3604f3d329e7868510bf3f7971599006 \
    --hash=sha256:d406ec79053c0871817f700c2ac8c36ba0d87f9c34b7458b0f0063bb218b0466 \
    --hash=sha256:e3e88a7a75d7d95cbee1f37dc31341d6201124cf21c6c4b1dfab8ccba9b09e0f \
    --hash=sha256:fa029678be1ba7f953c409b0b27bf15cc69cd1c9b3a674fbd78856ebefca1052 \
    --hash=sha256:fcfd6c22418e57dd5b3aea649e813b2e2cfb2aebf317b210d90f1fe4b3018b52
    # via huggingface-hub
httpcore2==2.15.1 \
    --hash=sha256:730f603058c260a3288378217b464713350c23e3b8fe8bba753701054032d98a \
    --hash=sha256:a94b6b3e6260e1a3f035915408d3dcd7dc8e844c2b544b642e227cf98a747af0
    # via httpx2
httpx2==2.15.1 \
    --hash=sha256:75841cd83d16b417f7d6da8f2089ce8120806efdbf96af37d46c5129c49015be \
    --hash=sha256:ff75485308fbe441e75b1218546e4e78d884df41e1a9741c1f79335ed8fcfef7
    # via huggingface-hub
huggingface-hub==2.2.0 \
    --hash=sha256:1667f145dc56dc210d60966069397df9ecfca9607a5d43db88b308c89dae56b3 \
    --hash=sha256:5d1b47537394e4215cb858aa12fd493d0f7ef7f58990f5dcd24bc173107b2871
    # via
    #   tokenizers
    #   transformers
idna==3.20 \
    --hash=sha256:a7db850025b95ded1eae8a46181a1a6c56c92c96f0e2b005d9ff8dc0210cab44 \
    --hash=sha256:ab7ae7122974553370f0bdb919e1a960b2cd1bc1ef0276416d896db81c14582c
    # via
    #   anyio
    #   httpx2
jinja2==3.1.6 \
    --hash=sha256:0137fb05990d35f1275a587e9aee6d56da821fc83491a0fb838183be43f66d6d \
    --hash=sha256:85ece4451f492d0c13c5dd7c13a64681a86afae63a5f347908daf103ce6d2f67
    # via torch
markdown-it-py==4.2.0 \
    --hash=sha256:04a21681d6fbb623de53f6f364d352309d4094dd4194040a10fd51833e418d49 \
    --hash=sha256:9f7ebbcd14fe59494226453aed97c1070d83f8d24b6fc3a3bcf9a38092641c4a
    # via rich
markupsafe==3.0.4 \
    --hash=sha256:007e1ffd9bf65bb6ee96df7b258fc632a4868dd5566037986c64781f35a36e98 \
    --hash=sha256:02fa4acbc6a3fc5c693c34d4dd8c1130b7fe99cc915181b0ddd6f72aeb296002 \
    --hash=sha256:03470d1a8268e692ecf79ecd565593e59d44219377a7ead61f1f1b94c1f7ff6b \
    --hash=sha256:04e7902ba80ee4bac1d50a549606527a1dcf0476cd81403db41099d3b60ec653 \
    --hash=sha256:051417f74bcaaefa316276e0ff723f541616ca51043d070da00249d9bddd3e3c \
    --hash=sha256:05295589e619b9bed252a86b532b8e27350abc372d18ba89b59375325e91ec1e \
    --hash=sha256:06de8ef6331f6e822c28d577dc8bf43fe398800477c49498f38fc38b67ff33fc \
    --hash=sha256:0764a13d34cae40db7bbf3a09b7e9b491bf4603e20b263a7a9d6b8e324975d0a \
    --hash=sha256:077293e425f28ec737dbcad442a71752e28f8ae27cde3d68acd1fb212091cd92 \
    --hash=sha256:0930db9bdc62d22944e10b066448bb65dc9abe9112880c7cab8da54db4284d5f \
    --hash=sha256:0cee7cb0f9a1b6892ea482237d9403b3d1b4603aee057d0ff01f0fac2d019a97 \
    --hash=sha256:0d9c47709875fdb321452056622e930c52afbc07a7d780762fbb8b4d91ce6fa4 \
    --hash=sha256:11935df9bf455ed0c04eb87bcd720f02b1fe5e02128a9430f23aed6f93336fc7 \
    --hash=sha256:12a606a492de952afcb43b59a14aaaaad120e708d3663dd0fdf2d738d427a691 \
    --hash=sha256:14bd2d845d62ab678eaf81da89d7b621b51756c72346745c1a594c09d49207a2 \
    --hash=sha256:15ba9e28640feef770374b116a6f019c21f52404aeabe516aa7f800587b98cfc \
    --hash=sha256:18a801868a884f216e784d7d14db2a4077143ce7610440aee2ce8f734e7cfcde \
    --hash=sha256:1c0df495a977d10460a94941799c72d5b5ab03d3858d949b55b5a66c8f371c99 \
    --hash=sha256:1caa2fa5a6184fb233153b35f654e6687bd555476f6170f29d8ee9be1a8b0af9 \
    --hash=sha256:1e1451fab512d1bcc3dc26988ec1edb0b82c2db909132872cd9356070a6b63df \
    --hash=sha256:1f1f9477e174582b0a1b583d60b66e1f2cf5d3fe12cee985e4aedf44766600e5 \
    --hash=sha256:2628d3a8cb648ecebb3c5d6b0a1052d400e4d8b7ac0fb786be8d285b50040d17 \
    --hash=sha256:26e9867520db70d37f7fb421a7f0d8adb40171011fb84ce869afa1a83370dfa8 \
    --hash=sha256:2a6ef68ae94aed8721934072b27a3b654ea2100b97e4ab864cf1489c90926fbc \
    --hash=sha256:2b2b1e18af909b448bb3cf9e3433366f7a8726271fc214e8b10e0f62a78c724b \
    --hash=sha256:2cb3dd71fc6be918ad4264346a8ed69485f9b7ed7bf35495d8e22807cd6b8bea \
    --hash=sha256:2d1b7d9308288661f56672b1b157d75fc536714d3638487bbea17b6318a78248 \
    --hash=sha256:2dad610540cb2e6272855c178f08ae9a1c7ac258a7fb71660553a5f104b42741 \
    --hash=sha256:2e5a7cd7fdd14fcb1ae5d7d8bf23d24fbd1daefd1fbca2580132e1ea75f098b5 \
    --hash=sha256:2e9ad7dd851bf45fab9f75cbff4cb493fee9979e8d8c7c9c3ee119022518edd6 \
    --hash=sha256:340cbb1957ba99929cbf19a75626d36ba1ae21d1730b287d1cf7f824a20c4fc7 \
    --hash=sha256:34bdde374c5932765d7dc685c4a1d191a3207852d67e8e0a9eb6ea85156181f1 \
    --hash=sha256:353bd63081912ab8cfa6a0c7d185934cdf8426f04c618bba6bc4b394f2069b67 \
    --hash=sha256:387d8cd30e69b3f0a72877b9ae717033396404e19095b17fe89753a981fda44f \
    --hash=sha256:3882fb412298575bae3b9c46868251f15cc69307359f87bb1b382e53d6e5a2c9 \
    --hash=sha256:38fc55594dab834470b6733dead2ee9e3f657fb0608c769dcafa0ba5ab52f45c \
    --hash=sha256:396ec4e65cc889f69786b3b89478b471cee5a3bcf468b9d9bb03e1a30fb291fc \
    --hash=sha256:39dbacefc411633db5b4378b066a9aca70a3d7e2922c9e578d825f844026eeba \
    --hash=sha256:3a93d9616ddecfb393727a0041a562cf0b15a244e20f2bd25efc7949be4c4f17 \
    --hash=sha256:3d23795802fc8bd72534836d64489bbf0f67c088959091bdb22e10735a5107bf \
    --hash=sha256:434139499bb20b502ed3baa1f169e618f924a97e7a777fea1a49446d80106cf6 \
    --hash=sha256:436e3ffc6310d3c41878c601db29098102fe5d8a467c49da4a4125254e0980f2 \
    --hash=sha256:489505b03f692c3f376394e49194fa7a7f9e8558d6e293a7056a0032b0c38163 \
    --hash=sha256:4a540e2d3192792fc84eced57bef37851ccb2b41f73291bb17408eea77bcd278 \
    --hash=sha256:4a7cdc2a420ca01058182da4253329764d4bfa055564d1eced90e6ba1e8b1d3d \
    --hash=sha256:4bced6e2a6dba6a28f7dd3c6ce14df1b2dd495923f16ea484cad03decd463b2b \
    --hash=sha256:4cf3468d5ec187ffffcaca8e61929a37448f215dafc1386a12c750a72fe53634 \
    --hash=sha256:4e2c4809c14559aa7ef426f27fb35afbb38104c349a903bf8f3600456764bb38 \
    --hash=sha256:4ed644d75aa94a2baf7ec3a96eaa160ea58c742eb9d27c6506053c5c40fc84ed \
    --hash=sha256:4f6e0852a0283b1b1fd776eeb7b766a5f440b3e2bd31ab51af3b400585f3965c \
    --hash=sha256:5066b244f576f91afc8ee3ba029a89f99d39c79b1853fe9d39bea9f0afbec148 \
    --hash=sha256:5086f9975abb1ab531ee6afca1761e4b59a19b446f3f6522ed776963228cfe5a \
    --hash=sha256:50b5bedc9ed8a94fc8857a42ef4f84a81ea88f8d4f05dc8705fb23ee6d8dcca7 \
    --hash=sha256:52704c5d36eb6dda8866493decd61111fff86244c9b1ad225ca01b9e91e5970f \
    --hash=sha256:55ffd6ce583d97dc71dc92e930324c8c0d25aea7e3ade6ae54ef77cedb096811 \
    --hash=sha256:569d65055d367e3dcdf30c3f41119467b73d9ee9faf332bdf40402644f5ac08e \
    --hash=sha256:57f9947a7e57a081c1e3e0a2dd0d2dcf290a4531450e6f611e30084c222a7295 \
    --hash=sha256:5989cb26b2e1efc6a42216a9f6b5ee495ce5ace2e5b352a9af489976b32d1ee2 \
    --hash=sha256:5c22873ad1f0532ba40fa1727f3c0fc1bbbaab6d373d4cbe3f0dc74b2e2521c7 \
    --hash=sha256:5e8b3d0b18fd623afa12ecb2ce8d8becef69f9b5440c6330c7972200e0bb84b0 \
    --hash=sha256:61631e08084be9e21a8967ec3139c7616ed7c5e9368e05c86d1b39562c8a57b6 \
    --hash=sha256:64511c54db4e4987aef4c41923235927428729e8174c5dba488429be70a998ed \
    --hash=sha256:6669c1bf34080161ce49c589cc512ef24d4c704ac9d2b2d3667f519c60418378 \
    --hash=sha256:672d207103e6b16ca098611b0f9efad6bc00afd47c03d6ef62186495ca677dc0 \
    --hash=sha256:6768d67d1bce64270e0fdc2e69309d68b9b18ae56ddf6c711d168e9d051c2cac \
    --hash=sha256:6a45c3d514f2436064db00d7fc8778d888f0236ebfed649b53d13a59e69ad51b \
    --hash=sha256:6bd9e1788e15bfcf6a9082de42e30387e7b85d211ab21e57a939bb8cfaaf8d96 \
    --hash=sha256:6d2a9efe686f9de00d0d1ea32a4a5a86d558a2277501bd78d964214eab625e59 \
    --hash=sha256:6da83a088f8ef93b2d483a8232a4dbf4d69d3d8496b568a03c56becac43e1808 \
    --hash=sha256:7018d4af1cd272e847aa5917983ab5e83e4f6579f9dbfecd4a79c0ca80b144c2 \
    --hash=sha256:71f88e749ea29f67f21f3b36433c1dc54c7729ed2a6d9e2da2e0d9e0d7b224eb \
    --hash=sha256:737c9c3981998eba27f11786f84fddcbabc74068b72a4a1f454ea02094b57b65 \
    --hash=sha256:73e77980c7207854f00fc4e71fb1626868d5740ab4012623d55c7a99ad122a72 \
    --hash=sha256:799c39bdf5e2f1292fedd3009f7b3c9e760f10b2420cb9638d56920840ff6db8 \
    --hash=sha256:7a83aa6e4805df46fed18e989d3d16f86ef60cb50bbc8d9ce3a6be89165fbf6e \
    --hash=sha256:7d3391b2188d18737cb2fa147028b1096236eaa7e156446c650a489fa2cadc91 \
    --hash=sha256:7e1636da3d8dfc220b6dd10264db5f2b165e4888c4518594898fbe381049af8a \
    --hash=sha256:805c8b84534fa10891890f0e4be39f3a99e94615d93e8836bf9fa1fdca2feeb2 \
    --hash=sha256:811d02d5122171c1941357efd8f9bf4ffe907b7f0a1a4e729a880e4be3f46e3e \
    --hash=sha256:8138eb83940ec7299024d92d4dee45f601b9e6c5ffde9d25f4e35e326203c707 \
    --hash=sha256:83b3944fea42a8400edf92fd1770fb8d0d4f7de651353bd2d8525a92dba69a21 \
    --hash=sha256:849dd2bb0e5e4ab2b71c7191726a4a8d5aa8a610daa584728cbee0b710ddc4ef \
    --hash=sha256:8698d70a8081ee8c090dbb394768b5789a1da8b131b5499f89d071dd3cfaf6be \
    --hash=sha256:8781a792a070cf2bd1b86d3aa943894115faaba6e88122a7bf32d62072742453 \
    --hash=sha256:88d59b473bfb03259722600839af9bbd7fa13a2eb514beefeedb95997882f69a \
    --hash=sha256:8909c2f1c6dd65e054ac4b573a91c8384d1492281e55d82d159d653f7a13adf6 \
    --hash=sha256:8965520ac587c94a4ac48b729be3d8b8de00af39699b17585dfb599babe77977 \
    --hash=sha256:8b5d563170ff8ba3181caa967c99a3c804d1dedb702c7cb93a6a7c32247da978 \
    --hash=sha256:8e124f974786f831d6043728e38296969d3579db8896fe004682f5758e613581 \
    --hash=sha256:8f0fac8b13d14bb06c68195f849371924ae53dd7b1c00fed24650f704383b692 \
    --hash=sha256:9240187afb63d2f9ddc3e032c670356fe941f6e20662ea168a5dc3f1f317e1b3 \
    --hash=sha256:925f929d6b59a8b3f8b8c6ac363cd0af7eecc81efb3071770b3c6717c450a369 \
    --hash=sha256:9348cbb300d224fe3b89793262cb093504d4ae927004468463f745188a193e4a \
    --hash=sha256:9388003072b95f2f1e3fd908604194d653ba21330d811961a78b7da1a77e9e36 \
    --hash=sha256:9438a2648b2195980cb2dd8e53ed7b8df91319e2d0b70ae61a9e1d1bc8d3bec9 \
    --hash=sha256:94e4c421742086aeee4c32a506eec8859d7634aad943f7e6aacf70f813478768 \
    --hash=sha256:94f5407f7bc64fa6463906b896f9904beeeb7dd8dc116ee8e9056c8714ff9916 \
    --hash=sha256:971a3bbb75d97ae4e2e8f7d4834236f86f85f0c85e04ab2e191db1123b04f80b \
    --hash=sha256:9e227f3dbe6bde7491cf0a9965d00b88c6b1a4a95d11480ddf88bb96d397c19f \
    --hash=sha256:9e25feb9e330b63edb0278a0acdf85e50d0cb0fbf49c3084abbe4e24ae195346 \
    --hash=sha256:9f098115c247e11d138ab83a28fa0323c77015007ea2df73ba5fd714dfefd67c \
    --hash=sha256:a18f38cafc329bac5e3c2b96c765b4c96d3d103421ed22ab7988c1e3fce27464 \
    --hash=sha256:a4bbd2d87dd233b9fc5812160c3d0ffbe42edc22a26ce0469f58479ede633fe9 \
    --hash=sha256:a5fcffb37e602b0b3c1638a97746b9b96125caa9bcf6fa41d337a9261de231ee \
    --hash=sha256:a8e9f292fcda89b324f2f5c91d13f1424a153e40fc2756f38ee23b15835ff300 \
    --hash=sha256:a9f54054101545a9a9cccefddf54316aa6e4491611fcbef9e91b3b6bebec04f6 \
    --hash=sha256:aa2c838cc024642cc04c6854232f32b43e5e22833dd11119c1766c7873b8370d \
    --hash=sha256:ac0c7c9f1609b0c4c114feb1d7a3409564c7fb77e360bed9e97e5d25dfeaf868 \
    --hash=sha256:add96447a86d205ab616665d53b2950ee81083757f56e6ea833c8b2917646b46 \
    --hash=sha256:ae9dcb8fbe244cb82f8a6458b455b927a03685e383d9bacf1ea5ce180b96dc97 \
    --hash=sha256:b4a635a0487774f841cb1fb62e907e7195cc95bc761e053184b8acc3ceb20733 \
    --hash=sha256:b4d12837e0203bbace818ff4a7461afdcd78bcd782351cea148139180d7bcffe \
    --hash=sha256:b61687d0828e72bf5cda24a2690188f37170bd31c9359ac97e4e66569f120a16 \
    --hash=sha256:b807e598953730f82e4eae3bd30f6a122cf6b31c398c6b504c0e04c13c170429 \
    --hash=sha256:b8cd1f918b26fd7b1832ece557cc18f2d8747309ff8b3f0ef9d4250c5ad67a39 \
    --hash=sha256:b91cc9d336957239ff200f30097e6fea2dc6d6fb3c81e853eaa09eac904fd894 \
    --hash=sha256:bd3ce56ae2cbae3ba82b683bc425cd7e48d2ed8b10f3e818186b6f5646d9271c \
    --hash=sha256:be6cb0c799abb0e2ba3e618e6d28ddddf7e485f6c2ce938dfa237daf3905072c \
    --hash=sha256:befb4158af32106b9a93db8d6d1d1cbbd418c0d5aca0cabb7b1780abf0c89169 \
    --hash=sha256:bf053da3c97a4bc5ecfbb218cdd2983febd91c617be8367d139882aa11e490aa \
    --hash=sha256:c02e8f18bdedba082cef725942ac823b9b60656db07f7e265cb31618dfd00d77 \
    --hash=sha256:c1bc67752d5f21013cfe430df4062441714eab79f65a6a05e01505957e9c35fe \
    --hash=sha256:c61750fadcd119d0825bcb7d7d675dd264dcc89cc05292aab5be68ebdbb374ad \
    --hash=sha256:c90d5b3d4e944e065a301d741b3c1d784f6bd1f503aa68b4967e32b2ba313d85 \
    --hash=sha256:c9a7f43c0b202b334cc9184af09bb8f21d3a209e038efaf106936fb69e6b026e \
    --hash=sha256:cb96e6e088d6cf71c1ea977510948320234824cf226e32f6f6e044f7a9c82b34 \
    --hash=sha256:cf63c214fe879a65e69a386f915e36104fc84254ab141240f8854602d8e0be2a \
    --hash=sha256:d1aca03ede943eb80ab3d63bb082c84b7aab85ea83bd0fd0c200260945fb49d9 \
    --hash=sha256:d2e56fd3b00222722abfb3f5f0759ddbae4b90811b5ad4343c64030ad1bde70c \
    --hash=sha256:d5f93ebbeb8032d47e349328ec8662d973d9b05a70b3c35df1f91fe419b84749 \
    --hash=sha256:d882a373d8093c2941e01291b7ced96e9cbe4781da9a7751ca7e6c70385e5214 \
    --hash=sha256:d920abdfa61279ba1a2ef9484aab07bf03331f8c08a10120fa332353d06e6932 \
    --hash=sha256:da2af0d7aebfc2074080d72efa6ab8317c62481ef1f896f65d9999c1c01f4494 \
    --hash=sha256:dd8ea6ebee7aedbf7c749fa80521d9ccf1ba473e0d1e14805caafbaad281c889 \
    --hash=sha256:de8b364c423ef0a4bad9069657d617f9a5d2b2062457a89b1fa16ee199c399c1 \
    --hash=sha256:df1ae86ff54725a01fa1a0510b914ca53a161b7050be74f6204e24aded5971d0 \
    --hash=sha256:dff05cb7016dff1e9fd68f4122c127b65dfc59de5306cfb7ad92f956f230bee2 \
    --hash=sha256:e1a622f13970d81f95d0c72f9dc090dce9085fccfa4c9f2174377ee32bd15786 \
    --hash=sha256:e49fb0d1ce92cfa0cb198cc5b1b11cdf9d0638658e2a2db2687e39db7c87fc78 \
    --hash=sha256:e5c802729725bd07e2bc3ab7b76dc7e0bbfc53129d8f1eb1c002c24cf774717e \
    --hash=sha256:e841068dc0be4cb6dfb5c890eb88cbdcff2f4a332393c7ec94e8e618bd32c1a8 \
    --hash=sha256:e916035e3e9930cbdfdd10abf48861340221857f45509565898e012263f7b289 \
    --hash=sha256:eba154571c16e032112afac0dc2dfe9e63c2ceb7aedd07bb7eecf2ce26d4dd4c \
    --hash=sha256:f03460ff076f70ab595bb45a0205ccea1971443575b6920c52e755dec2b3fbfe \
    --hash=sha256:f0ec3b750b59375eab5b0fb2b9254810c00a3375be6d789899f1055a1d556237 \
    --hash=sha256:f291bcf42ae98eb5107edb162c3c998b4a89648fd8e99ed4cbd12705292788cd \
    --hash=sha256:f61efe1d2fe0de16158a5fe1d1cf3c14bdb6aecd54d8938fd26512c525c1f624 \
    --hash=sha256:f68edfc67aabac33708941f26f22a7b8e9f81429bc0cf249fcf7d66b23af8d19 \
    --hash=sha256:fa95848c929b6a75f6848d3c9793e59db365ee436776e57db835cdbfa79ba977 \
    --hash=sha256:fd9f8797427910198f95bced71ddfed61130d7e349213bfb8466c9c99e2c46a8 \
    --hash=sha256:fdb4ca07ab75ffadab4a8b135ad59cdbb3156b99310f3d565370da74a15d6bd3
    # via jinja2
mdurl==0.1.2 \
    --hash=sha256:84008a41e51615a49fc9966191ff91509e3c40b939176e643fd50a5c2196b8f8 \
    --hash=sha256:bb413d29f5eea38f31dd4754dd7377d4465116fb207585f97bf925588687c1ba
    # via markdown-it-py
ml-dtypes==0.6.0 \
    --hash=sha256:008382aeab529df5d3f00501ad9a7dcd64494d4b5b1971fc4c79019e6c1f5010 \
    --hash=sha256:03ce583adfce34ad33aa9e1fc7a8344dcf90ea776cc4ef0e5a48d4eae84e5d20 \
    --hash=sha256:084dfe51a7ad58b171f05115f8226ed4233a454a1611371947e806e76f0c638d \
    --hash=sha256:26b1f1fa4f0435a2946859823f6e2bf06796f1e9f10f5a05b08a5e3c8f46ff69 \
    --hash=sha256:28d676428b104bb9717b0928bc5c5129f2d6b51b6727587cc4289e7bf8713cb5 \
    --hash=sha256:2a3e9d53925597fbffafd2a37048dadeddd0bdaba58058f6ae0869ed709a184d \
    --hash=sha256:3035518e3e19add1a4cac9236ab22888b208a4074912514313ccb2d6d242cde8 \
    --hash=sha256:317be9967fb84b0ce4e80e6b1bf71213d21971621cf6f1e501a63602a95297bf \
    --hash=sha256:31f1ce979d31a357e95aa81812f20412c8c954fa43c44ee3ead1e1c8a78575ef \
    --hash=sha256:37da32aa97749251025666d62372775019594577b9c9e9cfda83bed48d778fdb \
    --hash=sha256:3b4a480aa8fd54a1805b8ac10f3f91763926a74f73c0c364c10f9231854f4170 \
    --hash=sha256:3be9911d953f97cddded4b9961d7b650473b7e55806d20f6176f8356dfe7b38e \
    --hash=sha256:3e169214e0d80ff1c038e1b3017e33c23e43bdf948d42d31de8283111c7e2fa3 \
    --hash=sha256:488c99ab181a2f59d9ec3b12c5fa11ec904e92be2c4ba18cded54dd7501208fe \
    --hash=sha256:5359c588cc62de6f78d7430f06b65853d884955494d86d6ad90b6dd64a3f3a08 \
    --hash=sha256:573b11f3c327e17ef3826d266e676cf1149a1f3016f822a05f2306c55d8246bf \
    --hash=sha256:57ed0d6b4ac5e7868361303a9c57fbcf63b768236ee14456f585dfcf260d0292 \
    --hash=sha256:5a519c9e95a216fbcb8e759793ef7fb40793fc803ed839142d6dc5be9be5bc89 \
    --hash=sha256:5e60251d32ced5598972e4d5e06a2f044341f9291402551a3f6f0ec44f9299b0 \
    --hash=sha256:6c8e39b53e90afda8ce52859c93de4dba3e02b76d85dcf091cc469f9184c6dae \
    --hash=sha256:6eaed129a4afe90694b8685e2f9b6294849f5eda4af9a15be83a4326eeebd775 \
    --hash=sha256:6ec0d244a5bba12239025389ad88bbfb45f9f10e25ab4f678e9a4768ebd47532 \
    --hash=sha256:7728c0420ec1c338564fc8b01015ff2d58567e70f17fedce5a0a7c0308c0d5b9 \
    --hash=sha256:84fa136b8602c8c39e3b6cb24918960cd6f36cade7a70376f56770729cd56510 \
    --hash=sha256:8f490c003369ce60e514a0c3b12374f05274c101fee1bead6740ec8a564032b0 \
    --hash=sha256:9c6ad60af4102789a5c09824004beade2f7f28cd1cd581ee5c170d9dc2fbb00e \
    --hash=sha256:b1b503864fada3f74fabf8d9fee7b4c1cbe956301e6fdece975d5f77c2fce958 \
    --hash=sha256:b76fa1d3f92967d58289ac47ab7458ede66e6f3527fff3e59142aee57d9307cd \
    --hash=sha256:bad8d1dd5bed060a29332b99d63d0e5c2969081e1c6ea54adfbccfdfa783be44 \
    --hash=sha256:ce7563e0b1a4482cbc1b4a6272145e54e4489e54fe7428f94908c3d87103abfa \
    --hash=sha256:d4f1b9329a251e4affe3bb58f4d3e2db22a714396fd7ffb40d0b5db423c24d17 \
    --hash=sha256:d574c2b28921dc72e869df248f1a278f6eee176a1f237c8642e1a71eb15f3977 \
    --hash=sha256:de9d14748dbf3968951436ef514a29c9d1fe438aa680d110134ee2f7a9f9df18 \
    --hash=sha256:e25bb3b0ad1217b60626e4ed45b10ca170c41d99fbe44a12bebc1e07ec4aad55 \
    --hash=sha256:e2d6149f3a57f405bcad5fb41e03218b8373936253f23e1ca84c0108abbc3392 \
    --hash=sha256:e74266ca8e97874a937b7646378c178025650a236584f7474d10d8086a6edea3 \
    --hash=sha256:f4adb4af61516510d786cf8c01851a66f6d3ddfa79e1144deaa5b40d8507231e \
    --hash=sha256:f4f59f83c82ab480e924b988e7b1b4eb4de836dfcf5390c6f59148d1a00e1d02 \
    --hash=sha256:f6cb525101b6b903779188c1e9e9490c343b455ab822883e02cf01e5547338d2 \
    --hash=sha256:fb87f46b4f7ad7b5d3ad8f4b452b024bd4229d44c8ff934798c1fe656210387a
    # via onnx
mpmath==1.3.0 \
    --hash=sha256:7a28eb2a9774d00c7bc92411c19a89209d5da7c4c9a9e227be8330a23a25b91f \
    --hash=sha256:a0b2b9fe80bbcd81a6647ff13108738cfb482d481d826cc0e02f5b35e5c88d2c
    # via sympy
networkx==3.7 \
    --hash=sha256:e3fd2c13a7814cee3746340d8d7f8598a67f16a58bf47fb7f8793fab6efca1b0 \
    --hash=sha256:fd77a511bd90f39f3d016351345b52cf5319b813bdca01de3f755d3cca62e96a
    # via torch
numpy==2.5.4 \
    --hash=sha256:067374eb538c34c745436365cf7b0112595c1d326f21ce4ff340f61230239fbb \
    --hash=sha256:0b4724a19de67bea8cfc4970798efa78bcbbe2ac2613cfac16721a42d44de2a5 \
    --hash=sha256:0f02a46e49cfb6c73bdb7aea1c0d3461dbae9aba613542b65f657cd3d17b9fab \
    --hash=sha256:1c2e71b04c6cad90026e544501bbe0ab9290fa8a4d845e7e8c0d124fb429c988 \
    --hash=sha256:1ef3aa6d7e29bb13677323114280b05acc57607fa2300e66432d665d5418a162 \
    --hash=sha256:2132418bf8dd124a427ca9e6a1daf9ee1a87185344c95119ceae868b99466da1 \
    --hash=sha256:2199ed071f460487c8db2c0e5c0b564494190edb4772fe80f9aad88b2604def5 \
    --hash=sha256:2377da2dd3ba2c1200956acbab2a358c83b8e1f8531191672d1cd6ad83250d53 \
    --hash=sha256:298eca75243f2cbbfdb460560b9fb2a1792a33cf2ab4286efd43d92e8d3df508 \
    --hash=sha256:2c2c4afffdeb7920e445028dd71eb932cac3e704792e964bc2a232426d4f1255 \
    --hash=sha256:2ca144f15135b6212a5c47b1e2aeca6e412f102f95a2d5d88d8aec77eb255de3 \
    --hash=sha256:2fa3328f784fc8277fc48026f6cad516f5c561c5d8e2e39b3c9e0c8f23223b34 \
    --hash=sha256:325518d4245b9e331387702aa58c2ce1dc4cdcbb41dfb4ccd5dcbc7e08db1266 \
    --hash=sha256:332f3378fe077dd850e677ec01bdcc4f22368fb5d50ef10b2c79230b1bf5a592 \
    --hash=sha256:3573cd22564692a5b899ec344e5d5b9cc4576f2985b96f22af3564ed54f2710f \
    --hash=sha256:381a7a3d2e65e64c0ec302795ab9dc12bb1e73f150904699c153716177eebdaf \
    --hash=sha256:38f47be9f74ab870d2633b5456ae519c43758a8d1fd05342f0ce4ecc034396ee \
    --hash=sha256:4054173604cd8658796053f1f3bc0befb68ec1c0762c57fdad61e199256a8617 \
    --hash=sha256:468397ba3c64427474706e5c9123fe266395496714dc684294eac75cd4930d1e \
    --hash=sha256:4e263278bfb5ee6409db8aedbc4cc32973b1b82bc1e8d3c668551d04d83a7e37 \
    --hash=sha256:5258bc06526964be5face2fc6f756857a3f24f21ec3e72ca131337a75b165d6c \
    --hash=sha256:56733449d2544178beaa4545cee357370440cf056c197f9c7bfb19dbfdd0e86d \
    --hash=sha256:5ec3753760c1a6d8bb91200666e545c3a9728e6269dfb5d6ce02340996698aa3 \
    --hash=sha256:5fbf7141bbfd63aea22f435c9062a032b9ea0082fe9845dad7f021d3f1234e71 \
    --hash=sha256:64d1c8ac28a4077cf987e0a71a7a0ef7e2df70722f07f0baa42dbb7eb6938647 \
    --hash=sha256:64f9c9878c1938476365e11ccfb6b770f3b9e5f045ccddc514235041e6959365 \
    --hash=sha256:6c109eac9cd439193678f69d70733c1108487546ca8eafc107b510ae10c1aecd \
    --hash=sha256:6d6a71b9d9a97c03633aa12565ef2825ffa036cc1d99cfd50dacf0f128af4fe2 \
    --hash=sha256:6ffa07666f8da0eef81d149934a626d0d95fbd6838432a33e66245423a9062c0 \
    --hash=sha256:7415db95818b39ec475a5eea54d9e3b6bc83e3912158e46da3438cdce399804d \
    --hash=sha256:77045a4b175bbf5316ec08003880804336c78f92281a1b72222b274ea85ec5ac \
    --hash=sha256:7a14a461d9340f1b46b8648578aed9cdb8b3b018a8fac6c1dde2c9192a01a87f \
    --hash=sha256:80d6ef6e8620eb2c2b4c4caad50b5935d6db3cde2d51581b55dcc79e14016d1d \
    --hash=sha256:81e3420b27048b65eb14c3acf0c174a8cb0e023277716110347d2dcb26026dad \
    --hash=sha256:823874a507a84af050493b622affde94b6f7c3a0dc22cb2801381bc03b871c00 \
    --hash=sha256:8b4d2fd2d34e5f8c9235ee787de5631a37a28402b15cb80814df973d2be54129 \
    --hash=sha256:8dddfbee2e68d26d0d7d7d9cb247b1fd4409241cce32d815a11d97ec2cfde179 \
    --hash=sha256:950ea81d57ef070665581b6e1b5f6a029306423cd1739c5b95fe78aa30db6b9d \
    --hash=sha256:956555e0603a4d38019ae6925711cb9dc43195c076a928accf7ea5d50bddfe53 \
    --hash=sha256:98b053943e5a0474ec0da309d2cb9d3f18ea57f8a2067c2ab7b5f763d1068380 \
    --hash=sha256:9968ab7e49b93ac6e1c3b2239732183152c9150f16308d30b66a372cffe3483c \
    --hash=sha256:9a94cf751c9ad8ebaa835bcd3d40dacf8534ad086b88c38029b65123c7999d2a \
    --hash=sha256:9cb18a327b49c5c337f972b03682f6a49855525faaf3c0d3e9c96cd0fd8880a8 \
    --hash=sha256:a7b1b6353e36a7e50de2973a38d705c88ee93adcf120673cee7f45a4a3fa223a \
    --hash=sha256:a813ed7719bf45463c51779e6a98d0385fe905e48447526938a4b8337333d551 \
    --hash=sha256:aa1cce2ff3f8d953de38b76bf44602caeb69f101430208f64a10067f7cb4b1d3 \
    --hash=sha256:ad62a416ddcf863bf44bba76fbf6b53366ab0692e294f51cae4b5fbe0d246788 \
    --hash=sha256:aec3fc4b32ff82421274f5d205c559c51c840c8df66a78efd7f3612dd005a26a \
    --hash=sha256:b1185012870173de7ae33d370bd45b1cf5baee747ea4b97036b65f4e93016877 \
    --hash=sha256:b11e8fda06a7d69f15ebf542660b74466c2e51094800c1fb794f47ad4faeef17 \
    --hash=sha256:b64a85f40e154983960a4167d4c1d57a50c7f109b3d3264a3a984154e90a8454 \
    --hash=sha256:b86966fbe4ad7de710422175572bcdc75fdedadfb54bc6fab7deabccddd7780b \
    --hash=sha256:b89d0aaae2fe498c648f4c4795c084db535af5bd98ef942b2a3681fb74ce8645 \
    --hash=sha256:bc39ac66a7a9a3fbd6134fda43136b60ffde99c8f4501e64e0d2b24da137babf \
    --hash=sha256:c05ede731b03fb1b7591faca9389ade3267d2bddf1ad8882bb3f2cc5e101694f \
    --hash=sha256:c6342f54c67093cae5c0227eb0eb772fdb79f2a2c37a6eb278b9909ee06aa356 \
    --hash=sha256:c668b2f0d651605b58892644b0e302c7157f7159544227758c896982ef384b18 \
    --hash=sha256:c9b80cdf5cedba0e90d93fa5f9a333c4d65bd545cd669b71bb97ce2b703c9d73 \
    --hash=sha256:cfd73180400042a7c532d30c5e287bdd03c59ff9ee1b4c0316af0539e29dfe23 \
    --hash=sha256:d4cccbbc78717966f764cd3af4fb70276fa01fc7a2688af11c78901fa5c04f05 \
    --hash=sha256:d549420b8858885cea8838a727842249218b9c1da24dd517e25c9c7a948310a3 \
    --hash=sha256:d8200f16437b289a5bb927c6e184eccc3e8389bc0070fea4cd5b9e13c1757959 \
    --hash=sha256:e94aef2c639da4a960ad0db8e06471208d8589974953d78b61d345b4eb99e394 \
    --hash=sha256:fbde6962867ee75b48b0ee29b2b9372ec5d617799dbaf38e82dc0596f2f7738a \
    --hash=sha256:fe4d21ab149f15e4e6043dfb0de87e6e5f34ac176cde83060e9802981fca2ac2 \
    --hash=sha256:ffa6ce09a1c6a08e9667dd9c97aa0b14184e8d18f2a14b78b2a2328c9147f076
    # via
    #   -r notebooks/kaggle/requirements-kaggle.in
    #   ml-dtypes
    #   onnx
    #   onnxruntime
    #   pycocotools
    #   scipy
    #   transformers
nvidia-cublas==13.1.1.3 \
    --hash=sha256:37936a16db8fe4ac1f065c2139360608a543a09275cb1a1af612e08cfa065436 \
    --hash=sha256:b6cdce694e47ff6aadf0a69df1cab6628d696f5ff56e8d16af50309d855fa20f \
    --hash=sha256:b7a210458267ac818974c53038fbec2e969d5c99f305ab15c72522fa9f001dd5
    # via
    #   cuda-toolkit
    #   nvidia-cudnn-cu13
    #   nvidia-cusolver
nvidia-cuda-cupti==13.0.85 \
    --hash=sha256:4eb01c08e859bf924d222250d2e8f8b8ff6d3db4721288cf35d14252a4d933c8 \
    --hash=sha256:683f58d301548deeefcb8f6fac1b8d907691b9d8b18eccab417f51e362102f00 \
    --hash=sha256:796bd679890ee55fb14a94629b698b6db54bcfd833d391d5e94017dd9d7d3151
    # via cuda-toolkit
nvidia-cuda-nvrtc==13.0.88 \
    --hash=sha256:6bcd4e7f8e205cbe644f5a98f2f799bef9556fefc89dd786e79a16312ce49872 \
    --hash=sha256:ad9b6d2ead2435f11cbb6868809d2adeeee302e9bb94bcf0539c7a40d80e8575 \
    --hash=sha256:d27f20a0ca67a4bb34268a5e951033496c5b74870b868bacd046b1b8e0c3267b
    # via
    #   cuda-toolkit
    #   nvidia-cublas
nvidia-cuda-runtime==13.0.96 \
    --hash=sha256:7f82250d7782aa23b6cfe765ecc7db554bd3c2870c43f3d1821f1d18aebf0548 \
    --hash=sha256:ef9bcbe90493a2b9d810e43d249adb3d02e98dd30200d86607d8d02687c43f55 \
    --hash=sha256:f79298c8a098cec150a597c8eba58ecdab96e3bdc4b9bc4f9983635031740492
    # via cuda-toolkit
nvidia-cudnn-cu13==9.24.0.43 \
    --hash=sha256:67a7273b5cf062f9446fd76cf464351a1c0f66501e6cd78f6675c0d604d8ac87 \
    --hash=sha256:71f181cd810e90f9b6023b01186fe82d13d65f0ec098581ee201d39fad769e4b \
    --hash=sha256:a6812a554a1ff0413e9c52b84c26c050380649ab9615f9c16bded368ce9f421f
    # via torch
nvidia-cufft==12.0.0.61 \
    --hash=sha256:2708c852ef8cd89d1d2068bdbece0aa188813a0c934db3779b9b1faa8442e5f5 \
    --hash=sha256:2abce5b39d2f5ae12730fb7e5db6696533e36c26e2d3e8fd1750bdd2853364eb \
    --hash=sha256:6c44f692dce8fd5ffd3e3df134b6cdb9c2f72d99cf40b62c32dde45eea9ddad3
    # via cuda-toolkit
nvidia-cufile==1.15.1.6 \
    --hash=sha256:08a3ecefae5a01c7f5117351c64f17c7c62efa5fffdbe24fc7d298da19cd0b44 \
    --hash=sha256:bdc0deedc61f548bddf7733bdc216456c2fdb101d020e1ab4b88d232d5e2f6d1
    # via cuda-toolkit
nvidia-curand==10.4.0.35 \
    --hash=sha256:133df5a7509c3e292aaa2b477afd0194f06ce4ea24d714d616ff36439cee349a \
    --hash=sha256:1aee33a5da6e1db083fe2b90082def8915f30f3248d5896bcec36a579d941bfc \
    --hash=sha256:65b1710aa6961d326b411e314b374290904c5ddf41dc3f766ebc3f1d7d4ca69f
    # via cuda-toolkit
nvidia-cusolver==12.0.4.66 \
    --hash=sha256:02c2457eaa9e39de20f880f4bd8820e6a1cfb9f9a34f820eb12a155aa5bc92d2 \
    --hash=sha256:0a759da5dea5c0ea10fd307de75cdeb59e7ea4fcb8add0924859b944babf1112 \
    --hash=sha256:16515bd33a8e76bb54d024cfa068fa68d30e80fc34b9e1090813ea9362e0cb65
    # via cuda-toolkit
nvidia-cusparse==12.6.3.3 \
    --hash=sha256:2b3c89c88d01ee0e477cb7f82ef60a11a4bcd57b6b87c33f789350b59759360b \
    --hash=sha256:80bcc4662f23f1054ee334a15c72b8940402975e0eab63178fc7e670aa59472c \
    --hash=sha256:cbcf42feb737bd7ec15b4c0a63e62351886bd3f975027b8815d7f720a2b5ea79
    # via
    #   cuda-toolkit
    #   nvidia-cusolver
nvidia-cusparselt-cu13==0.8.1 \
    --hash=sha256:4dca476c50bf4780d46cd0bfbd82e2bc10a08e4fef7950917ce8d7578d22a23f \
    --hash=sha256:786ce87568c303fadb5afcc7102d454cd3040d75f6f8626f5db460d1871f4dd0 \
    --hash=sha256:dccbd362f91a7b9024d1f55ee9f548ac065027ff15d8c8b0db889ab3a8f31215
    # via torch
nvidia-nccl-cu13==2.30.7 \
    --hash=sha256:ca786ffa5a647c75d4d1f5cc72a6c4f537947e2ba8823d7c8aaf768e7a7b9f77 \
    --hash=sha256:cefa7fdb9710efd0f39c5f1be1d61ff6fc9a996c451265bd7fbdcf9455ed4b50
    # via torch
nvidia-nvjitlink==13.4.92 \
    --hash=sha256:25f74fad0d654271c921ac4dca614bd6258bc21791242fc7b2289dad7ae9c099 \
    --hash=sha256:9e4a7ff4f0cafa8c624917055b863dc11f5c2912ead23c462889e166f3b0e57d \
    --hash=sha256:b286f3a4f227a9363efdec263c7b91788cef1478d2b8a5fa8bab7f3e82ff82fd \
    --hash=sha256:e0391f24ed94ec879b84e3da4d4ec320c879aff681f2c7a638462f7199284323
    # via
    #   cuda-toolkit
    #   nvidia-cufft
    #   nvidia-cusolver
    #   nvidia-cusparse
nvidia-nvshmem-cu13==3.4.5 \
    --hash=sha256:290f0a2ee94c9f3687a02502f3b9299a9f9fe826e6d0287ee18482e78d495b80 \
    --hash=sha256:6dc2a197f38e5d0376ad52cd1a2a3617d3cdc150fd5966f4aee9bcebb1d68fe9
    # via torch
nvidia-nvtx==13.0.85 \
    --hash=sha256:4936d1d6780fbe68db454f5e72a42ff64d1fd6397df9f363ae786930fd5c1cd4 \
    --hash=sha256:cb7780edb6b14107373c835bf8b72e7a178bac7367e23da7acb108f973f157a6 \
    --hash=sha256:d66ea44254dd3c6eacc300047af6e1288d2269dd072b417e0adffbf479e18519
    # via cuda-toolkit
onnx==1.23.2 \
    --hash=sha256:008cb0467b2bbee41448acc7da8b6f4e704624cb0d327a2d5adafc7ce19bc5b8 \
    --hash=sha256:0100e6c3f30db8ff10876d8cfd0cb27296166d5a612ab37c3998e07e83b3fde8 \
    --hash=sha256:03334d6c834767c7acd37c7db51c98e98c8ceb61a964f6df96386e13272d2870 \
    --hash=sha256:16ef247e51dbf42e32bd92f47ad772d17dda77f64c4017e0ded9725ff9ab3922 \
    --hash=sha256:1b8680ce1e6a9a4736374a9dce4de14ea8ee05e0dccf0784a78a6e5646bdc1f6 \
    --hash=sha256:1e6cbca3d808f811141ed0a0939e71b3a6c9fdefb2435f4a862ec776336718fe \
    --hash=sha256:32fd9c92244c2aea2b2c9e0e7b18fedcf6000434124ab6fc8796e22baa602d30 \
    --hash=sha256:419bbbe3fbdf45a7658ee0aa1a54cd170ea15f3e5a60ace6e8d94f1577b3674b \
    --hash=sha256:612f5dccea6d53c5517309c52496b6dae1115757e3b79f31be24d4c40fa45ca3 \
    --hash=sha256:77674dc4fda2bde9a13aee67fb9ff658080159eb516d3a5b3fb2418d44dc70be \
    --hash=sha256:7abf381d278f31ac62487fddedc9dd42da842dce94d5d43536836ee3efdf4a2b \
    --hash=sha256:80cef0fad59524d02c21ec93f4fbccdcc6223f1c33339d597519a2d27cac19a7 \
    --hash=sha256:83b3fc8321303c9da62824730457ba2f7ae0970f0e2f7fc0117912df7f8a4826 \
    --hash=sha256:9b382ba898a7c142a0801d03cf04ecabced96c1543c7b643a86f0928143802de \
    --hash=sha256:a203efdbaabbbe8f25e854e2b2921382d6fcf4c67895656f939044b0632974e8 \
    --hash=sha256:a2b88d7e3634662f8d030117a7b02d864cfc965800547089ba62d3a9ceab3564 \
    --hash=sha256:a40265d62b7a614041593e11370d316880f9628eb5a0d49d9028c9c0e7f1cc08 \
    --hash=sha256:b0b8dae0d33dd8606370bc264b0b1d6e64cfdf8b83d7c676fab8eff6b88ca409 \
    --hash=sha256:b2c07abb24f1c2c50ff5996c567eb9757470827f6d55b7f0af9d62c8e658bd7f \
    --hash=sha256:c03ecf6b835d136108eeaeeafbd0026fc7b3cf98661409fbc6b63d5a29361348 \
    --hash=sha256:e79e35e152d3095c6910ae81013bbc68679e32bfc0ca76f840968d4b6fdfb864 \
    --hash=sha256:f8b9a5e25a390cc291600e5fd619f4b79708287a6bbc41a37209f364e08a63da \
    --hash=sha256:fb3e892f19f3a793b9722587349941b074f74091ad33e794a7798fe03fdc0c9c \
    --hash=sha256:fcbbd53e3482434dbf2c27f4a8727ad4865e21bbc0b5530e7557669f8d8f587b
    # via -r notebooks/kaggle/requirements-kaggle.in
onnxruntime==1.31.0 \
    --hash=sha256:09d56445c1753e66e0912de69d3f0184016ad9a191dcd6925bf5dd570d2bfbe5 \
    --hash=sha256:0ba02a44acb6203040354d9a1f160e3f37a43feac7bb05caa3e0ea545efed505 \
    --hash=sha256:1ecc1450af28d2cf362990e188ccc81b51388f317f641ad973ab4301473200f2 \
    --hash=sha256:278e0dc922ec69b05a28f59110d5421e2ec8b1d0dd46c6b10c063069a4051e72 \
    --hash=sha256:317608967b03807ed4661113b08293fac02a1db6496a6863a07d9f19232936ad \
    --hash=sha256:35758d7606d578ec5b9d65f6e8a1f488013194c3f6097038a3223cb26d35ef9a \
    --hash=sha256:37c7dfe398550afdf9670a29315dbb88e49d8afc473ffaf1f410376efbb9c80a \
    --hash=sha256:37fd78cee5160c7a43a1730ccb3682ffd880af9c9e80385d625c0c2f8b125809 \
    --hash=sha256:5c54a0eb7b2b4eef3eb9dcfaf82f5ce880db07288dc309574f6657e9da5cc754 \
    --hash=sha256:5e129d6c56abd53e659cb70f00a108d6824086470ff99c2e47a82e5786563db3 \
    --hash=sha256:73e0165d58ece068c2a8a1c477c90b38e5a8adbbd399fdfdfd4bd79cbc28ff8d \
    --hash=sha256:83e3dbcf6abc6189c4bdf7d329c07ba1133c88172134c266d84b4409aa3b9dbf \
    --hash=sha256:984c0a2c1ad6a41fbc101dc3949abe4a72254892d01a5e70d9b792711e0bfa54 \
    --hash=sha256:aaab9b3af536b06ca27ab5e35e3d429c97457ce76cf298af103f687e8b9975c0 \
    --hash=sha256:ad663106f6eeff3d454f24a786450459d07f30e74863851104fc1b8b3f368127 \
    --hash=sha256:cbf1a7f6470ddfe9dbc781966af8ce4a10e1858d75a93f93cc6b9367c9587870 \
    --hash=sha256:d25cd65874b75fdf16149120a04d0cd4551f860a3c8e2ecec785a1903e41d8aa \
    --hash=sha256:d2d5ac22f896c810be2b2b171392bb908f80b6c9a7e2d592ddb7435c928044e1 \
    --hash=sha256:d4092b78fc5bab77ce6522393098cdb2535423045ecdcff15cc0d022162d6b66 \
    --hash=sha256:e0e050bf9ec754950a6ba9830e4032f4004d972c6f38c5642fef26d44d894965 \
    --hash=sha256:e4efa4a1a0bb0b5173c6a3292c181d518b8323f9d56e978635d0c09d38c94d1a \
    --hash=sha256:e51d10d2e2e1e5bbf9b126a0cd9853d3e6c4e21424518dd50160b91471be33dc \
    --hash=sha256:e85c1632c0a8cf488bd8f1039f5320877b864c8f9ebd4122fb8bb909f83b7096 \
    --hash=sha256:e93d7c5fad20afa697ac16f376fd0306ed180f9a376e86106cc0b7d84f53ef87
    # via -r notebooks/kaggle/requirements-kaggle.in
packaging==26.3 \
    --hash=sha256:94edc256424af38762eb31306eed28beb9f0efc50a8837492c9d6fd6004aed79 \
    --hash=sha256:d7193f7c8e4e93f444fde0262bf90af30e16fa0ad0ad44cb553c87339b23cd1c
    # via
    #   huggingface-hub
    #   onnxruntime
    #   transformers
pillow==12.3.0 \
    --hash=sha256:00808c5e14ef63ac5161091d242999076604ff74b883423a11e5d7bbb38bf756 \
    --hash=sha256:04f01d28a6aaff387bf842a13be313df23ba0597a44f1a976c9feb3c6ff4711a \
    --hash=sha256:06ff022112bc9cbf83b60f8e028d94ad87b60621706487e65f673de61610ab59 \
    --hash=sha256:0740a512dc522224c77d9aa5a8d70d8b7d73fb91f2c21125d8d025d3b8990e45 \
    --hash=sha256:0847a763afefb695bc912d7c131e7e0632d4edc1d8698f58ddabec8e46b8b6d3 \
    --hash=sha256:0dd2064cbc55aaec028ef5fbb60fa47bb6c3e7918e07ff17935284b227a9d2df \
    --hash=sha256:0feb2e9d6ad6c9e3c06effe9d00f3f1e618a6643273576b016f591e9315a7139 \
    --hash=sha256:10e41f0fbf1eec8cfd234b8fe17a4caac7c9d0db4c204d3c173a8f9f6ef3232b \
    --hash=sha256:1182d52bc2d5e5d7d0949503aa7e36d12f42205dc287e4883f407b1988820d39 \
    --hash=sha256:164b31cd1a0490ab6efae01aa5df49da7061be0af1b30e035b6e9a1bfe34ee6e \
    --hash=sha256:1657923d2d45afb66526e5b933e5b3052e6bdea196c90d3abb2424e18c77dae8 \
    --hash=sha256:186941b6aef820ad110fb01fb06eb925374dc3a21b17e37ec9a53b250c6fe2d1 \
    --hash=sha256:1cca606cd25738df4ed873d5ad46bbdb3d83b5cbca291f6b4ff13a4df6b0bbe8 \
    --hash=sha256:21900ce7ba264168cd50defae43cd75d25c833ad4ad6e73ffc5596d12e25ac89 \
    --hash=sha256:236ff70b9312fb68943c703aa842ca6a758abfa45ac187a5e7c1452e96ef72b5 \
    --hash=sha256:23aceaa007d6172b02c277f0cd359c79492bbb14f7072b4ede9fbcaf20648130 \
    --hash=sha256:23d27a3e0307ec2244cc51e7287b919aa68d097504ebe19df4e76a98a3eea5bd \
    --hash=sha256:24870b09b224f7ae3c39ed07d10e819d06f8720bc551847b1d623832b5b0e28d \
    --hash=sha256:251bf95b67017e27b13d82f5b326234ca62d70f9cf4c2b9032de2358a3b12c7b \
    --hash=sha256:25b9b82bb22e6e2b3cd07b39c68b7b862001226cb3dff7130d1cb914121b39ed \
    --hash=sha256:28ce87c5ab450a9dd970b52e5aca5fe63ed432d18a2eaddd1979a00a1ba24ace \
    --hash=sha256:300557495eb45ebb8aec96c2da9c4be642fbf7cd937278b4013ba894ea8eb0eb \
    --hash=sha256:30f2aa603c41533cc25c05acd0da21636e84a315768feb631c937177db558931 \
    --hash=sha256:331b624368d4f1d069149002f25f44bc61c8919ce8ddb3c45bdad8f6e2d89510 \
    --hash=sha256:37d6d0a00072fd2948eb22bce7e1475f34569d90c87c59f7a2ec59541b77f7a6 \
    --hash=sha256:37dc8f7bbb66efe481bb60defacef820c950c24713fb44962ed6aa2a50966de1 \
    --hash=sha256:3b8182a766685eaa002637e28b4ec8d6b18819a0c71f579bf0dbaa5830297cce \
    --hash=sha256:3edce1d53195db527e0191f84b71d02022de0540bf43a16ed734ed7537b07385 \
    --hash=sha256:446c34dcc4324b084a53b705127dc15717b22c5e140ae0a3c38349d4efec071e \
    --hash=sha256:4998562bf62a445225f22e07c896bb04b35b1b1f2eb6d760584c9c51d7a5f78c \
    --hash=sha256:4b0a7fe987b14c31ebda6083f74f22b561fd3739bc0ac51e019622e3d72668c7 \
    --hash=sha256:4e8c2a84d977f50b9daed6eeaf3baef67d00d5d74d932288f02cb94518ee3ace \
    --hash=sha256:4f883547d4b7f0495ebe7056b0cc2aea76094e7a4abc8e933540f3271df27d9c \
    --hash=sha256:514435a37670e3e5e08f3945b68718b6ed329bb84367777e16f9f4dfe1e61a0f \
    --hash=sha256:53aa02d20d10c3d814d536aa4e5ac9b84ca0ff5a88377963b085ad6822f93e64 \
    --hash=sha256:5594fc43d548a7ed94949d139aa1341b270f1863f11cfd37f5a6c8b778a6b67f \
    --hash=sha256:571b9fcb07b97ef3a492028fb3d2dc0993ca23a06138b0315286566d29ef718a \
    --hash=sha256:57b3d78c95ba9059768b10e28b813002261d3f3dfc55cc48b0c988f625175827 \
    --hash=sha256:5afb51d599ea772b8365ae807ae557f18bccfe46ab261fd1c2a9ed700fc6eb17 \
    --hash=sha256:6b02afb9b97f65fbca5f31db6a2a3ba21aa93030225f150fa3f249717e938fb4 \
    --hash=sha256:6c0016e7b354317c4e9e525b937ac8596c38d2d232b419529b9cd7a1cd46e39a \
    --hash=sha256:71d6097b330eea8fd15097780c8e89cb1a8ce7838669f48c5bacd6f663dd4701 \
    --hash=sha256:756c768d0c9c2955feb7a56c37ea24aea2e369f8d36a88da270b6a9f19e62b5e \
    --hash=sha256:78cb2c6865a35ab8ff8b75fd122f6033b92a62c82801110e48ddd6c936a45d91 \
    --hash=sha256:7a743ff716f746fc19a9557f60dab1600d4613255f8a7aeb3cdde4db7eb15a66 \
    --hash=sha256:85f998ea1848bc6757289e739cfbdda3a04adfd58b02fc018ce54d754a5ce468 \
    --hash=sha256:8728f216dcdb6e6d555cf971cb34076139ad74b31fc2c14da4fafc741c5f6217 \
    --hash=sha256:877c3f311ff35410f690861c4409e7ccbf0cd2f878e50628a28e5a0bb689e658 \
    --hash=sha256:8cd2f7bdda092d99c9fc2fb7391354f306d01443d22785d0cbfafa2e2c8bb418 \
    --hash=sha256:8e95e1385e4998ae9694eeaa4730ba5457ff61185b3a55e2e7bea0880aef452a \
    --hash=sha256:962864dc93511324d51ddbb5b9f8731bf71675b93ca612a07441896f4688fb8c \
    --hash=sha256:9cf95fe4d0f84c82d282745d9bb08ad9f926efa00be4697e767b814ce40d4330 \
    --hash=sha256:9e881fca225083806662a5c43d627d215f258ff43c890f831966c7d7ba9c7402 \
    --hash=sha256:a2b55dd6b2a4c4b7d87ffa56bdb33fdc5fdb9a462173861a7bc097f17d91cb09 \
    --hash=sha256:a45650e8ce7fafffd731db8550230db6b0d306d181a90b67d3e6bca2f1990930 \
    --hash=sha256:a876864214e136f0eb367788dbd7df045f4806801518e2cfe9e13229cfe06d8f \
    --hash=sha256:ae26d61dfa7a47befdc7572b521024e8745f3d809bd95ca9505a7bba9ef849ec \
    --hash=sha256:af8d94b0db561cf68b88a267c5c44b49e134f525d0dc2cb7ed413a66bc23559a \
    --hash=sha256:b343699e8308bdc51978310e1c959c584e7869cc8c40780058c87da7781a1e94 \
    --hash=sha256:b3c777e849237620b022f7f297dd67705f9f5cf1685f09f02e46f93e92725468 \
    --hash=sha256:b629de27fda84b42cde7edef0d85f13b958b47f6e9bbcbba9b673c562a89bd8b \
    --hash=sha256:ba09209fbe443b4acccebe845d8a138b89a8f4fbaeedd44953490b5315d5e965 \
    --hash=sha256:ba54cfebe86920a559a7c4d6b9050791c20513650a1952ebe3368c7dc70306f8 \
    --hash=sha256:bcb46e2f9feff8d06323983bd83ed00c201fdcab3d74973e7072a889b3979fcd \
    --hash=sha256:bcc33feacfaefce60c12fd500a277533bdc02b10a19f7f6d348763d8140bbba7 \
    --hash=sha256:bf16ba1b4d0b6b7c8e534936632270cf70eb00dbe09005bc345b2677b726855c \
    --hash=sha256:cf1845d02ad822a369a49f2bb9345b1614744267682e7a03527dc3bf6eea1777 \
    --hash=sha256:d69141514cc30b774ceea5e3ed3a6635c8d8a96edf664689b890f4089111fb35 \
    --hash=sha256:d9c7f76c0673154f044e9d78c8655fb4213f6ca31a836df48b40fe5d187717b9 \
    --hash=sha256:dbce0b29841537a2fa4a214c2bbf14de3587c9680caa9b4e217568472490b28f \
    --hash=sha256:dc624f6bc473dacdf7ef7eb8678d0d08edf15cd94fad6ae5c7d6cc67a4e4902f \
    --hash=sha256:e158cb00350dc278f3b91551101aa7d12415a66ebf2c91d8d5ac14e56ddd3ad0 \
    --hash=sha256:e491916b378fba47242221bb9ead245211b70d504f495d105d17b14a24b4907c \
    --hash=sha256:e795b7eb908249c4e43c7c99fac7c2c75dab0c43566e37db472a355f63693d71 \
    --hash=sha256:e7e480451b9fa137494bccd3a7d69adbe8ac65a87d97be61e11f1b1050a5bac3 \
    --hash=sha256:e91206ee562682b51b98ef4b26a6ef48fd84e15fd4c4bc5ec768eb641d206838 \
    --hash=sha256:e9871b1ffbfa9656b60aeee92ed5136a5742696006fa322b29ea3d8da0ecc9cf \
    --hash=sha256:e9aeb04d6aef139de265b29683e119b638208f88cf73cdd1658aa07221165321 \
    --hash=sha256:ebaea975e03d3141d9d3a507df75c9b3ec90fa9d2ffd07567b3a978d9d790b26 \
    --hash=sha256:f0606c8bf2cdefea14a43530f7657cbbb7ecf1c4222512492ef4a4434a9501ec \
    --hash=sha256:f13c32a3abd6079a66d9526e18dad9b6d280384d49d7c54040cd57b6424041d9 \
    --hash=sha256:f7401aebd7f581d7f83a439d87d474999317ee099218e5ad25d125290990ba65 \
    --hash=sha256:fa4ecea169a355be7a3ade2c783e2ed12f0e40d2c5621cda8b3297faf7fbb9f5 \
    --hash=sha256:fbd139c8447d25dd750ab79ee274cc5e1fe80fc56340ab10b18a195e1b6eca3e \
    --hash=sha256:fdafc9cce40277e0f7a0feabce0ee50dd2fa1800f3b38015e51296b5e814048d \
    --hash=sha256:fe3cca2e4e8a592be0f269a1ca4835c25199d9f3ce815c8491048f785b0a0198 \
    --hash=sha256:ffd0c5368496f41b0944be820fcb7a838aa6e623d250b01acf2643939c3f99d7
    # via -r notebooks/kaggle/requirements-kaggle.in
protobuf==7.36.2 \
    --hash=sha256:497d0463ff3316681da6c0b9e8d06cb465d61abce00b613ab42226175644d1bb \
    --hash=sha256:89f23aa53c24553a2416fd4fd1ec06f74fa42b14b546d8883128813f775bbfd2 \
    --hash=sha256:912c1221170e16c08d1f086762f563dd61ff83c18b5fa6652952dfaded66f728 \
    --hash=sha256:a300819d441e078a5608c0d3c709796bb548136058fda017ae51d425b44fd353 \
    --hash=sha256:bdb3a345d48db958e6ce1f18e508beb0cc981d64f24088427549c866cd039f1e \
    --hash=sha256:cbc70b17ee27e28894c7fee8bb04be1abead49e936bc70eb60052531eee2079e \
    --hash=sha256:e11e1f0180583a2af89db6a2ecd9e8dc40aa6d2988ca175bfd0e6d12ea72d74e \
    --hash=sha256:f4fee11ec330d238b34a05c9b675f693c20415d1c5bd7d5320cc2f8a798eb9cf
    # via
    #   onnx
    #   onnxruntime
pycocotools==2.0.11 \
    --hash=sha256:04480330df5013f6edd94891a0ee8294274185f1b5093d1b0f23d51778f0c0e9 \
    --hash=sha256:08c79789fd79e801ae4ecfcfeec32b31e36254e7a2b4019af28c104975d5e730 \
    --hash=sha256:1192de413a23b4b94199197e8f8dbe1277cc24e6e9847bee6a71be3d8e543963 \
    --hash=sha256:18ba75ff58cedb33a85ce2c18f1452f1fe20c9dd59925eec5300b2bf6205dbe1 \
    --hash=sha256:34254d76da85576fcaf5c1f3aa9aae16b8cb15418334ba4283b800796bd1993d \
    --hash=sha256:368244f30eb8d6cae7003aa2c0831fbdf0153664a32859ec7fbceea52bfb6878 \
    --hash=sha256:484d33515353186aadba9e2a290d81b107275cdb9565084e31a5568a52a0b120 \
    --hash=sha256:4fc9889e819452b9c142036e1eabac8a13a8bd552d8beba299a57e0da6bfa1ec \
    --hash=sha256:63026e11a56211058d0e84e8263f74cbccd5e786fac18d83fd221ecb9819fcc7 \
    --hash=sha256:693417797f0377fd094eb815c0a1e7d1c3c0251b71e3b3779fce3b3cf24793c5 \
    --hash=sha256:6a2f4f036f5bfdaf8c9625279051e921721fc9d27f92c97a3f3355a07ba38513 \
    --hash=sha256:78bae4a9de9d34c4759754a848dfb3306f9ef1c2fcb12164ffbd3d013d008321 \
    --hash=sha256:7cd4cdfd2c676f30838aa0b1047441892fb4f97d70bf3df480bcc7a18a64d7d4 \
    --hash=sha256:7fd4121766cc057133534679c0ec3f9023dbd96e9b31cf95c86a069ebdac2b65 \
    --hash=sha256:81bdceebb4c64e9265213e2d733808a12f9c18dfb14457323cc6b9af07fa0e61 \
    --hash=sha256:83d896f4310379849dfcfa7893afb0ff21f4f3cdb04ab3f61b05dd98953dd0ad \
    --hash=sha256:87af87b8d06d5b852a885a319d9362dca3bed9f8bbcc3feb6513acb1f88ea242 \
    --hash=sha256:89e853425018e2c2920ee0f2112cf7c140a1dcf5f4f49abd9c2da112c3e0f4b3 \
    --hash=sha256:8cedb8ccb97ffe9ed2c8c259234fa69f4f1e8665afe3a02caf93f6ef2952c07f \
    --hash=sha256:8e159232adae3aef6b4e2d37b008bff107b26e9ed3b48e70ea6482302834bd34 \
    --hash=sha256:9dc8b388984c72aa84b1a68933f196ce71ab114c59232d0eab20c97cc1300875 \
    --hash=sha256:a1c05f91ccc658dfe01325267209c4b435da1722c93eeb5749fabc1d087b6882 \
    --hash=sha256:a2b018497ec198ffc737dd7e6306a2e69999779ca619a9e12950e4506e410c3e \
    --hash=sha256:a2e9634bc7cadfb01c88e0b98589aaf0bd12983c7927bde93f19c0103e5441f4 \
    --hash=sha256:a6b13baf6bfcf881b6d6ac6e23c776f87a68304cd86e53d1d6b9afa31e363c4e \
    --hash=sha256:a82d1c9ed83f75da0b3f244f2a3cf559351a283307bd9b79a4ee2b93ab3231dd \
    --hash=sha256:ac8aa17263e6489aa521f9fa91e959dfe0ea3a5519fde2cbf547312cdce7559e \
    --hash=sha256:b6a07071c441d0f5e480a8f287106191582e40289d4e242dfe684e0c8a751088 \
    --hash=sha256:bd7a1e19ef56a828a94bace673372071d334a9232cd32ae3cd48845a04d45c4f \
    --hash=sha256:c230f5e7b14bd19085217b4f40bba81bf14a182b150b8e9fab1c15d504ade343 \
    --hash=sha256:c3546b93b39943347c4f5b0694b5824105cbe2174098a416bcad4acd9c21e957 \
    --hash=sha256:ca9f120f719ec405ad0c74ccfdb8402b0c37bd5f88ab5b6482a0de2efd5a36f4 \
    --hash=sha256:d6b88557166794a24acd03f50296f2b95adf3e4206b4b8995e6bdca925c9cb72 \
    --hash=sha256:e21311ea71f85591680d8992858e2d44a2a156dc3b2bf1c5c901c4a19348177b \
    --hash=sha256:e40a3a898c6e5340b8d70cf7984868b9bff8c3d80187de9a3b661d504d665978 \
    --hash=sha256:e72527fdc00985d29dbe31d17b19cd2d16fbad7b01e974c567b593d5844de710 \
    --hash=sha256:eba35d6e06caaea28ce65a4b92e4343ecbb9abf4915f7ef7fca989b80839111c \
    --hash=sha256:eebd723503a2eb2c8b285f56ea3be1d9f3875cd7c40d945358a428db94f14015 \
    --hash=sha256:efd1694b2075f2f10c5828f10f6e6c4e44368841fd07dae385c3aa015c8e25f9 \
    --hash=sha256:f78cbb1a32d061fcad4bdba083de70a39a21c1c3d9235a3f77d8f007541ec5ef \
    --hash=sha256:f7eb43b79448476b094240450420b7425d06e297880144b8ea6f01e9b4340e43 \
    --hash=sha256:fd72b9734e6084b217c1fc3945bfd4ec05bdc75a44e4f0c461a91442bb804973 \
    --hash=sha256:ffe806ce535f5996445188f9a35643791dc54beabc61bd81e2b03367356d604f
    # via -r notebooks/kaggle/requirements-kaggle.in
pygments==2.21.0 \
    --hash=sha256:2363c69b61c4a97c838da3b130dcd6468f4848992b21a82f2a63ec34377137d9 \
    --hash=sha256:610ca751c9bc2492b38eb9a38a7fbc93edbbb2d7182edaf34e66ae493dee5c8c
    # via rich
pyyaml==6.0.3 \
    --hash=sha256:00c4bdeba853cc34e7dd471f16b4114f4162dc03e6b7afcc2128711f0eca823c \
    --hash=sha256:0150219816b6a1fa26fb4699fb7daa9caf09eb1999f3b70fb6e786805e80375a \
    --hash=sha256:02893d100e99e03eda1c8fd5c441d8c60103fd175728e23e431db1b589cf5ab3 \
    --hash=sha256:02ea2dfa234451bbb8772601d7b8e426c2bfa197136796224e50e35a78777956 \
    --hash=sha256:0f29edc409a6392443abf94b9cf89ce99889a1dd5376d94316ae5145dfedd5d6 \
    --hash=sha256:10892704fc220243f5305762e276552a0395f7beb4dbf9b14ec8fd43b57f126c \
    --hash=sha256:16249ee61e95f858e83976573de0f5b2893b3677ba71c9dd36b9cf8be9ac6d65 \
    --hash=sha256:1d37d57ad971609cf3c53ba6a7e365e40660e3be0e5175fa9f2365a379d6095a \
    --hash=sha256:1ebe39cb5fc479422b83de611d14e2c0d3bb2a18bbcb01f229ab3cfbd8fee7a0 \
    --hash=sha256:214ed4befebe12df36bcc8bc2b64b396ca31be9304b8f59e25c11cf94a4c033b \
    --hash=sha256:2283a07e2c21a2aa78d9c4442724ec1eb15f5e42a723b99cb3d822d48f5f7ad1 \
    --hash=sha256:22ba7cfcad58ef3ecddc7ed1db3409af68d023b7f940da23c6c2a1890976eda6 \
    --hash=sha256:27c0abcb4a5dac13684a37f76e701e054692a9b2d3064b70f5e4eb54810553d7 \
    --hash=sha256:28c8d926f98f432f88adc23edf2e6d4921ac26fb084b028c733d01868d19007e \
    --hash=sha256:2e71d11abed7344e42a8849600193d15b6def118602c4c176f748e4583246007 \
    --hash=sha256:34d5fcd24b8445fadc33f9cf348c1047101756fd760b4dacb5c3e99755703310 \
    --hash=sha256:37503bfbfc9d2c40b344d06b2199cf0e96e97957ab1c1b546fd4f87e53e5d3e4 \
    --hash=sha256:3c5677e12444c15717b902a5798264fa7909e41153cdf9ef7ad571b704a63dd9 \
    --hash=sha256:3ff07ec89bae51176c0549bc4c63aa6202991da2d9a6129d7aef7f1407d3f295 \
    --hash=sha256:41715c910c881bc081f1e8872880d3c650acf13dfa8214bad49ed4cede7c34ea \
    --hash=sha256:418cf3f2111bc80e0933b2cd8cd04f286338bb88bdc7bc8e6dd775ebde60b5e0 \
    --hash=sha256:44edc647873928551a01e7a563d7452ccdebee747728c1080d881d68af7b997e \
    --hash=sha256:4a2e8cebe2ff6ab7d1050ecd59c25d4c8bd7e6f400f5f82b96557ac0abafd0ac \
    --hash=sha256:4ad1906908f2f5ae4e5a8ddfce73c320c2a1429ec52eafd27138b7f1cbe341c9 \
    --hash=sha256:501a031947e3a9025ed4405a168e6ef5ae3126c59f90ce0cd6f2bfc477be31b7 \
    --hash=sha256:5190d403f121660ce8d1d2c1bb2ef1bd05b5f68533fc5c2ea899bd15f4399b35 \
    --hash=sha256:5498cd1645aa724a7c71c8f378eb29ebe23da2fc0d7a08071d89469bf1d2defb \
    --hash=sha256:5cf4e27da7e3fbed4d6c3d8e797387aaad68102272f8f9752883bc32d61cb87b \
    --hash=sha256:5e0b74767e5f8c593e8c9b5912019159ed0533c70051e9cce3e8b6aa699fcd69 \
    --hash=sha256:5ed875a24292240029e4483f9d4a4b8a1ae08843b9c54f43fcc11e404532a8a5 \
    --hash=sha256:5fcd34e47f6e0b794d17de1b4ff496c00986e1c83f7ab2fb8fcfe9616ff7477b \
    --hash=sha256:5fdec68f91a0c6739b380c83b951e2c72ac0197ace422360e6d5a959d8d97b2c \
    --hash=sha256:6344df0d5755a2c9a276d4473ae6b90647e216ab4757f8426893b5dd2ac3f369 \
    --hash=sha256:64386e5e707d03a7e172c0701abfb7e10f0fb753ee1d773128192742712a98fd \
    --hash=sha256:652cb6edd41e718550aad172851962662ff2681490a8a711af6a4d288dd96824 \
    --hash=sha256:66291b10affd76d76f54fad28e22e51719ef9ba22b29e1d7d03d6777a9174198 \
    --hash=sha256:66e1674c3ef6f541c35191caae2d429b967b99e02040f5ba928632d9a7f0f065 \
    --hash=sha256:6adc77889b628398debc7b65c073bcb99c4a0237b248cacaf3fe8a557563ef6c \
    --hash=sha256:79005a0d97d5ddabfeeea4cf676af11e647e41d81c9a7722a193022accdb6b7c \
    --hash=sha256:7c6610def4f163542a622a73fb39f534f8c101d690126992300bf3207eab9764 \
    --hash=sha256:7f047e29dcae44602496db43be01ad42fc6f1cc0d8cd6c83d342306c32270196 \
    --hash=sha256:8098f252adfa6c80ab48096053f512f2321f0b998f98150cea9bd23d83e1467b \
    --hash=sha256:850774a7879607d3a6f50d36d04f00ee69e7fc816450e5f7e58d7f17f1ae5c00 \
    --hash=sha256:8d1fab6bb153a416f9aeb4b8763bc0f22a5586065f86f7664fc23339fc1c1fac \
    --hash=sha256:8da9669d359f02c0b91ccc01cac4a67f16afec0dac22c2ad09f46bee0697eba8 \
    --hash=sha256:8dc52c23056b9ddd46818a57b78404882310fb473d63f17b07d5c40421e47f8e \
    --hash=sha256:9149cad251584d5fb4981be1ecde53a1ca46c891a79788c0df828d2f166bda28 \
    --hash=sha256:93dda82c9c22deb0a405ea4dc5f2d0cda384168e466364dec6255b293923b2f3 \
    --hash=sha256:96b533f0e99f6579b3d4d4995707cf36df9100d67e0c8303a0c55b27b5f99bc5 \
    --hash=sha256:9c57bb8c96f6d1808c030b1687b9b5fb476abaa47f0db9c0101f5e9f394e97f4 \
    --hash=sha256:9c7708761fccb9397fe64bbc0395abcae8c4bf7b0eac081e12b809bf47700d0b \
    --hash=sha256:9f3bfb4965eb874431221a3ff3fdcddc7e74e3b07799e0e84ca4a0f867d449bf \
    --hash=sha256:a33284e20b78bd4a18c8c2282d549d10bc8408a2a7ff57653c0cf0b9be0afce5 \
    --hash=sha256:a80cb027f6b349846a3bf6d73b5e95e782175e52f22108cfa17876aaeff93702 \
    --hash=sha256:b30236e45cf30d2b8e7b3e85881719e98507abed1011bf463a8fa23e9c3e98a8 \
    --hash=sha256:b3bc83488de33889877a0f2543ade9f70c67d66d9ebb4ac959502e12de895788 \
    --hash=sha256:b865addae83924361678b652338317d1bd7e79b1f4596f96b96c77a5a34b34da \
    --hash=sha256:b8bb0864c5a28024fac8a632c443c87c5aa6f215c0b126c449ae1a150412f31d \
    --hash=sha256:ba1cc08a7ccde2d2ec775841541641e4548226580ab850948cbfda66a1befcdc \
    --hash=sha256:bdb2c67c6c1390b63c6ff89f210c8fd09d9a1217a465701eac7316313c915e4c \
    --hash=sha256:c1ff362665ae507275af2853520967820d9124984e0f7466736aea23d8611fba \
    --hash=sha256:c2514fceb77bc5e7a2f7adfaa1feb2fb311607c9cb518dbc378688ec73d8292f \
    --hash=sha256:c3355370a2c156cffb25e876646f149d5d68f5e0a3ce86a5084dd0b64a994917 \
    --hash=sha256:c458b6d084f9b935061bc36216e8a69a7e293a2f1e68bf956dcd9e6cbcd143f5 \
    --hash=sha256:d0eae10f8159e8fdad514efdc92d74fd8d682c933a6dd088030f3834bc8e6b26 \
    --hash=sha256:d76623373421df22fb4cf8817020cbb7ef15c725b9d5e45f17e189bfc384190f \
    --hash=sha256:ebc55a14a21cb14062aa4162f906cd962b28e2e9ea38f9b4391244cd8de4ae0b \
    --hash=sha256:eda16858a3cab07b80edaf74336ece1f986ba330fdb8ee0d6c0d68fe82bc96be \
    --hash=sha256:ee2922902c45ae8ccada2c5b501ab86c36525b883eff4255313a253a3160861c \
    --hash=sha256:efd7b85f94a6f21e4932043973a7ba2613b059c4a000551892ac9f1d11f5baf3 \
    --hash=sha256:f7057c9a337546edc7973c0d3ba84ddcdf0daa14533c2065749c9075001090e6 \
    --hash=sha256:fa160448684b4e94d80416c0fa4aac48967a969efe22931448d853ada8baf926 \
    --hash=sha256:fc09d0aa354569bc501d4e787133afc08552722d3ab34836a80547331bb5d4a0
    # via
    #   huggingface-hub
    #   transformers
regex==2026.9.29 \
    --hash=sha256:01000ddf0e3ffef97f2413ceb514f6313040106b6d18a03ee00a4fe35c1eb1db \
    --hash=sha256:0166844493626c5015c6088ee15c9ca2fd060ca15b7641d1657da6a58432ae33 \
    --hash=sha256:044265d77d94f5e3cb2fd72c76723807c429cb8c533e9d4672d0334a6f14f588 \
    --hash=sha256:0476e5bcbe6e1ba3d1c4cc7bbb1c3ba78e3b979b5c8a88d0a6a8cdd4992b8c84 \
    --hash=sha256:066d0e3dbfdd739bce2bf8c2a41dd16f73e3d8adc2eb06dd803a36a307f56075 \
    --hash=sha256:0b65c72739f981377c9c22e0c5c3cd7f42da7bd8a3c9209330fac772c7d893ed \
    --hash=sha256:0c992c19cd45058a4b92f68f139c93db168b48fb1f322c9a7cd620806afb6b51 \
    --hash=sha256:0cc63b5e47c12a48d90c7e9d7de6a035dd14f62868aaedbb4e0ff8ba2b8bfe7b \
    --hash=sha256:0dd8af32e9f7b56b7f95cc1fd79b23054c3bdc172392ae560acc24d57b7ffe71 \
    --hash=sha256:0def9fb6abac55492d6d51cddb7225d07d6f279e774e0adc08569a54a5fc8d46 \
    --hash=sha256:0fd2c901cc307a745ad4bc87f20060d7a0825a3371d1e93488af22e7a387f78f \
    --hash=sha256:1043aedf5917caa861bcb25a9c11460049656bdf0017a90a309fa8f255467725 \
    --hash=sha256:121a76a0985db80ceae9e171c337f8c927868e37d01b54e3ce87bc87f9c6a208 \
    --hash=sha256:143533cc4b6fbc5b95aca0a5b8d541088d374831593def000ec89322c220221d \
    --hash=sha256:14e953ff3607c92d7675bf79c4d4509ef6782aa8c08509f179f9b3d6d0679e86 \
    --hash=sha256:18ae8eed4526e35bdb754d61562b90bf5c00a67fdcf3cc1380dd59597486631b \
    --hash=sha256:19959129885356df0e97556856f77eb2888380dac18bed075a7c05c5128c618d \
    --hash=sha256:1ba8c6a416569ce0d37e83e28a254a61dc99a419084dfb6476cea02d997f74fa \
    --hash=sha256:1c2a0026062abcc321a53db4a185ceba0b59a66b5d37b0808917a88b55a5257f \
    --hash=sha256:1d9fe8091b2e89d470df68a9331111ed008ae8aae6bf1e8e1fba4086a495c84e \
    --hash=sha256:2089fe39c406784d90101c726755ffa1497bb74638fd434300d2b88006186de8 \
    --hash=sha256:23ae6fdad9e63e54038f5ef78aba2933faca61e24d432786589e737bc5522ebb \
    --hash=sha256:26ec4ccce55aa533fbd603d08911b01101a8fcfec987845ac3ae2c7087b2bde3 \
    --hash=sha256:2f7f7aa47b229f2b39a2ae2596d2ad5625d77b5eb9856fac2dab3eb506cdd0a0 \
    --hash=sha256:31b003f9a070335e2a8233ee9b14a3ca8e6d792012ae011f741bf0aaf11744c5 \
    --hash=sha256:32ab11df9677ca80bcbb5fe4eb1da9109a5019239a054836efc6fa1c64e683cf \
    --hash=sha256:33026515aebc0e70d1c89978e53e8d695d35d9e472f8d5b34465ba3c74028650 \
    --hash=sha256:34b6925af9853bf461950e6508910f179fd6e9b1a7ec8548e069606b7e51a26b \
    --hash=sha256:352cf115a810b357caa35193ab656ecf5ef41056855e82f292c99e8514f8d954 \
    --hash=sha256:39ab5894d971f9ac68baa6eca5c50387db579cfcacf36ae8df3feceb1815e6d0 \
    --hash=sha256:3a21a9509d0ee88e7a70e1ad228cd2f0e0fd1e187458db132e8a8d18c97daf9d \
    --hash=sha256:3c5c2ef13797466aa64170cbb66ad98a32351dd4127694cea7199f80f213750d \
    --hash=sha256:3e778bfccd63075167709136afbc251c1f683758d5bf49c803c60ac3f894ce6b \
    --hash=sha256:3f1e6cb402a89457582cd696f982559217d13484a193202c394015297968c86d \
    --hash=sha256:42e82e578c904445d4c8a35b8f28052cf567593215fa5db06266fbc6f77aaa2e \
    --hash=sha256:4408b2b27a95ca8cc48b7411945753773353b5c93b307754781086c99d3a576f \
    --hash=sha256:446654b29bfaa30500d80947eda42cef1449dc8a87f4e3cf061cc8485d3a1f0b \
    --hash=sha256:45010bcfe66df41522d56c9b6114e87ecc597a08970ff6a2ced24415c141ae5f \
    --hash=sha256:49ee178ca31c94621294bf9b8b676a92a2e6bba8af0529591753719e57edb621 \
    --hash=sha256:4d7d93613b01b0199961330e49cfc52d479b3d5776c56c691db31130c0a07d91 \
    --hash=sha256:4fb41211d2333eb930a51e0546a65999761cf1f572a4da56ef9b8a62966c06f2 \
    --hash=sha256:4fe97894d1b306c919b4e50def1e6f6c522f4d03a7283811f4d108f1ce5d3ac2 \
    --hash=sha256:554bffadcbcb6d5f4e5fb10a61cc52084b9a63d1dab5f10bcd2c4343972e8e2c \
    --hash=sha256:59b49507f47479e299a9e1bc41b5cb83a7afda0540625f1dbae886615978acbf \
    --hash=sha256:5eeb8edc6110d9194a4d0d54610f64c37a31c605b5dbb7e407fc6ec7fa34a4a1 \
    --hash=sha256:612b709381c0355b70d89cdb51b7f670591ed5cbbc0e3b5337488019dc667b65 \
    --hash=sha256:61956f074ecd123f55adca68ee3eab46e6a07ad3f8e64e6db95dfacb444f55c4 \
    --hash=sha256:6398d5145689503412cc1748895242598d8846b8967b851133b20dc2ed1e21e8 \
    --hash=sha256:65b408d8fcb273e3499e7ef2ce796810da1becd208c7fb4373692a242d79d461 \
    --hash=sha256:686ac5350fceae63830bb98805fcb8039325bf4c06d9f6f048ff65229d5bffa5 \
    --hash=sha256:6a1a824fbed817e0a891103886b68f063b1e83cc51bc97192a90a60195a9291f \
    --hash=sha256:6abb75ab16bc3281714a5b99548a2225db70dba1f995f6d7f7419b76eb5a8fbe \
    --hash=sha256:6f7121a8914ed13fcfe2099f895341bfb789f004d4c5a0bdece8fa667da10849 \
    --hash=sha256:7020ed44df30b3aa492c00ee3b52d0548c1f30c2c6c5bb13ae897680900d3413 \
    --hash=sha256:720537c7ea6f80dc61913184edb0ce2497a306b39ef19f28505b322553d52bdb \
    --hash=sha256:724184b4aafed865e4f13ca313fdcb43024300c028ec67319cfa16847d84685e \
    --hash=sha256:7c03031610e3e6ed1768a2b7a8fc84637c1257b50c5eacaf094c6e17a84fc563 \
    --hash=sha256:80a5ea3b4fd9d6a5b9a44f7976a9acaaab35aa3c1f6b29e5bd857dfabaded223 \
    --hash=sha256:80c7cadd3fd2bfde5df8aa0787e315812cad0c313a753095d02f4c2b6c01677b \
    --hash=sha256:80ea96f5c1a30bf09007d48466521d9c294bebe197c708c3359096e3e3691632 \
    --hash=sha256:864e9b87ac33c3fb9fb4ad48166d4fdb579c351d5c77deb0d34bccb36a775cd9 \
    --hash=sha256:87fb80cbe3557e27e7b28b995c2b2eedf689b8886f941ab93e0e288f0976518a \
    --hash=sha256:8873c4a11c50b9989168881aeb3f08859f469d809941866aa1feefd8be5431f6 \
    --hash=sha256:888d60953908dcf761aa320c3e390ab8556efbdb551ace63921de90f6ae0848d \
    --hash=sha256:8b5fcc4771732191b2b7d1dd68d8f0353f47f8d90b6150f6dce58bf1112442cb \
    --hash=sha256:8f39588af4731c8923c26810eb3b33f76f17633985e40f59c3cd45a33805a895 \
    --hash=sha256:9173db3be74a35cb6731701094b98120f7ee4876a287882a59cdea1fa7da342f \
    --hash=sha256:92f05c9c42bde5785dc48770bc2194d9f7442544156f951e19cd31b096cec562 \
    --hash=sha256:951733b1bbdb71e377cec567b409f1a7881b47cfcad84121aa74cb575fa425ea \
    --hash=sha256:957bb708e8057ab1649ba566456429d691ec9b90d1c9ad1af1ba7ffbbeaf05f2 \
    --hash=sha256:9916fda742cd4eede63b286f58c06718324265d727ce0856eb1aac86d0d150d6 \
    --hash=sha256:9e1d3a4cb7993b708f0ada8d0c84590efd853f169e7147d2202c9da503180242 \
    --hash=sha256:9e4482589065c8ecd761cff522dcd85f2d39e62f551e37e025d1c7d54772def3 \
    --hash=sha256:a5300757f8a68f5b6cc33f57338d72a0e3589c5cc9ad5f8504ea06f028be582a \
    --hash=sha256:a540abfab208e1b7ef2df231c40ef3b6cbb30a0aad6204e9b6a81c10a6794628 \
    --hash=sha256:a5758353650079898dc1b2b0e95aa51fa23a30d020e06f62c430dd08ee56cdd8 \
    --hash=sha256:a64b85a4760337cfefdb27d42da6ed8b58e8cde3f2d57b6ef43e76ef6ea9ef47 \
    --hash=sha256:a655d34b2a6943af32401f3d94f72e9d731f6ad16285815550bf2b4ee69d420a \
    --hash=sha256:a714befaacbd10092ffe4cea0d3c5f008fb9efe9bc322c715bcdfdee414b9a3d \
    --hash=sha256:a760da040b47767b4b873adfb7c3b691e9ba2fc60f113f9d0b88f1a62f323e85 \
    --hash=sha256:addd736a0547d553283adaf4e05d7104e7f2c7b0b092e9b4d28756825f14531f \
    --hash=sha256:ae4613d7d9dda60fcba95f846cc6f808017f1843f392cf9daad14a6534493d71 \
    --hash=sha256:b11b589e00095ec69cf79841a76360f9b079e95b0368a25b5ebb951ab0c157ff \
    --hash=sha256:b3e445b66c80b4eb4234e855ce94d9adc183eedbd632816228d89930b91b2c5b \
    --hash=sha256:b7b893976e7fe42053da64f2aa27239c24252fd2ec6df471e1be197c0addc3b1 \
    --hash=sha256:b84f186a7f0536fe4ff9a9fa12d06d007b9b71d4b5352ddcc41f59ad6522a312 \
    --hash=sha256:b89efc38431793d28b7cd91227e2f952ad7c48df19132b17f43a5fec3c14143b \
    --hash=sha256:b97a38fb4c732b6832db6bf108963adbcd82ef1268ba2025dce390f45af75efa \
    --hash=sha256:b9d74e4eee9ddb64c2e92d5d61472c59c21684c059eb7b68767be9628e977859 \
    --hash=sha256:bb90e7177944b6684738c1fc36aabd2dd00d1de3be7dbe09f91e196f1bc0dc81 \
    --hash=sha256:bec37990e3d6121f29ecfb594bd8f1bf009e9f7926daba2e50e3b27d3892a783 \
    --hash=sha256:bf3c49863c23a1ad6da9c30351aed6cff8d5ddbeb63c5c8420ae54e98c7d0138 \
    --hash=sha256:bf48516e35cf848390ea68850aba53e7c333720d2945b4d2c25b69fc5171723f \
    --hash=sha256:bfc71e6d970419c1309b3640305298643e2a734cad3f7cfb6d2ddee4175ab53d \
    --hash=sha256:c0094897d7d01f184b2d7fe8c56c66d64efe01b31f4b7d34205b391387df1111 \
    --hash=sha256:c03c6eb6ece86dfdcbb34799efaa339b093132e1aceed491ba5e08fe06cdf699 \
    --hash=sha256:c1a9a6651197fbed6f0212591418b9def774fc3f8324f78d1bf0e6a63e5f8aa1 \
    --hash=sha256:c3589f40749acce747510bf5d589d54e376cb0930ea58b35effac97e5312b0c1 \
    --hash=sha256:c4e38dd8f39c43a91d2410ad2b85610701b0979342c3df1d69eaf8e838c757d8 \
    --hash=sha256:c6c8fabf1dafc1f1ddcbb67896d3f93efb092e8c4b6322d7389b944e76a484e5 \
    --hash=sha256:c90fcf7804ea0a54b896ce0f2b9565350220b8d4890fd0db461a476a4c687963 \
    --hash=sha256:c9b602fae1e00b7c035d661ce85575365719192a7b46784bd71cf64c68053aa0 \
    --hash=sha256:ccb64d887a9db1cd76dbc0f92051a1a478a2a67e7f56c62d915cb881d7734704 \
    --hash=sha256:d06fcdecc10fc7954d7c8f27a03c96055fe525274dc84a7b0dbdc3d6b9e03dab \
    --hash=sha256:d0c3082bf79bcd6a614d55916590ad4b8f93200e10b97f463ea5d9d07c9b5f23 \
    --hash=sha256:d49c18f1ea294cf4adde2e5ac256e98c82ea9d708462ce4bf799dffa7cfe8a2c \
    --hash=sha256:d60030baaa7bfbb02d650c126cdcddcb6e33dbff14d819434c8fa2fdcaeeeba5 \
    --hash=sha256:d7cab119d0df0b9413f106b4d7fc34f2872d3574ed3806fb48959c830b1537da \
    --hash=sha256:d9b77b25b4f395f92de6099ab08e8ae2bc7e51dfe157f22900902243a5cc90c7 \
    --hash=sha256:dabee8f4935e731fb46b2a3091bdda0d3d94b3bbfb907d2b4f12eefce4009619 \
    --hash=sha256:db5e82ba15c142425b8406690032df89e39cca4a2e8afbbb9a3d84edc2373ac3 \
    --hash=sha256:dc79d36d0618752265f0d575915bdc5c5130ecb9c9f6b3bcefeae32e4bdfafcf \
    --hash=sha256:ddfa987262763c3c22a8367d2a49c244b018a74c3a8e3ab1a864119ad45c5633 \
    --hash=sha256:e1172147d28d8fbcf8cb8d26c41506169f5ad8fe9ec969cb116835a19d4d8eca \
    --hash=sha256:e11edba5bc344a32b029a7af9d4b3173982dd79eeafa0b9dbd787364414b0509 \
    --hash=sha256:e2c89e9b762c57f59d5e99ee8b20202adb892e35f8d3485741340999ca55058e \
    --hash=sha256:e31f72490b7c12f7790e1e25c3afffd20503ee1bfb43461d7838b871ff244b19 \
    --hash=sha256:e8c65ef3862a8ad6e86492b6ed9327805dd66904c012bd3649dc67d822ed6c34 \
    --hash=sha256:ebb8912f565b8cdbbf27debfe00df04202c20e2f651b9e32767930c5eace3621 \
    --hash=sha256:ed511a0708e2297e1d6431e7fb217e3402791e491e02da800658ace4973df1bb \
    --hash=sha256:edf06545875f3efa31560d94121e95c7fd70d98b1dfedc0157097d79b13b52ea \
    --hash=sha256:f0fe9834e5aeccaf19a0d8feb296d66a24be1a7c9922002f842a682cd5abb787 \
    --hash=sha256:f1a0d5117230dd46b399a30a38afa44f79c99f3168988fdc4f425c3f928b39df \
    --hash=sha256:f37964e4a5e993d2fd45147741e9dff7f34a2d8c00ab94c4ea0514a4677f959e \
    --hash=sha256:f57dc6b8fef170f105d2cf5cdce254f47b137d7755086cf7050f47e16582abba \
    --hash=sha256:f93bc1c3486ef3747e07c9d7c1d0a147b8fbaab975f80e348aed6f71309dfaca \
    --hash=sha256:fb00027a09a8f9f08028b40dce4c933cf73e4833240ed356583fdc9cfa721566 \
    --hash=sha256:fb99cc9d45f48895d9d67f6a0b8a57f08d39c174d9f25ad97a313e0470267b1c \
    --hash=sha256:fdd88ed5e20b1bcdd234421e454962c971aa44b653bdb7f1ea9ef683e90fb649 \
    --hash=sha256:fe3fa1dd453ed5c7f5ea23a26218329790ed7197a99b90e94330e313959a7f52
    # via transformers
rich==15.0.0 \
    --hash=sha256:33bd4ef74232fb73fe9279a257718407f169c09b78a87ad3d296f548e27de0bb \
    --hash=sha256:edd07a4824c6b40189fb7ac9bc4c52536e9780fbbfbddf6f1e2502c31b068c36
    # via typer
safetensors==0.8.0 \
    --hash=sha256:040070828e36dc8e122178bbbd5830ff9e97920affb84cbe0f46442497bed358 \
    --hash=sha256:096ec1a98435df7beb08853bb5aa9081a84f23d0adc67ed1a0a10550f608373f \
    --hash=sha256:2ddf52eac562eda224f99acfa7889d02968c1fd59a5b011ae7d8137c37e9c02d \
    --hash=sha256:3ae091f16662658bdc019a4ff6cb4c085bb7d725eb5978b183ffd265863b6d2d \
    --hash=sha256:4124502b78f03534117c848f87a39b8f31e577b15eff423bf8bfb95f2a8c30d0 \
    --hash=sha256:4a95ae2b05d7726d751da4ebf626a2ca782b706e101bd894c95bc2450b1cffcc \
    --hash=sha256:7a46e5ff292c356d6991e60942ba7f79817682d3a2cef0702136448cb9c4d235 \
    --hash=sha256:7bc0a787ba8a35be368ee3574edfa2b1ad389eebd0a72e482ae275490e3f6c98 \
    --hash=sha256:87eec7ffed2b809f05a398a8becb7d013f19f7837cd15d9748580d6cf30dbaf4 \
    --hash=sha256:8e080062fcde23be189565e1c3305d16751a218ecf9412c8601e64204eb6f846 \
    --hash=sha256:8e9f537aa183a38ace122d27303dcd986b26bd2a7591f9181d7f0c396f4677ca \
    --hash=sha256:c554f85858e05226d3c2828e32395e677434685d6d94594a41643361c5e837f0 \
    --hash=sha256:c80201d22cbf405b80647a60ada77bba06c8fba2da2743ba1e89cdcc39a81f25 \
    --hash=sha256:f7838e5135a406ad3e02efdcb8cf2e5397d368b0154537c4fec682dbc544d452 \
    --hash=sha256:fabaf3e0f18a6618d9b36560682562157f77c2b71fcffc7b432be2baed9d753d \
    --hash=sha256:fcdd41ec4628fee5799f807c73c353629130fbd942aa23d83c623dd6c9d52d78 \
    --hash=sha256:fd6f3f93c9a0a7cc2788ee63fb763353d4bd2e89b0751bc78fcf7dda00bea774
    # via transformers
scipy==1.18.1 \
    --hash=sha256:011413b7426b75012840e35649e00fe0a2c3bae89fed433876e3a99251572efc \
    --hash=sha256:0ac49ea97594532dd44b7136094d35f5440fa06e6d9c6384a74c01764df388c5 \
    --hash=sha256:0e82073ecc7acc6436fac4b31674109c7e1d3e596789767eda01258a8c9e8123 \
    --hash=sha256:0fcb3c93519f27bb4f0c4b0f7802cdcaca7fcf93267b75edda2e9f4e8a55cbd7 \
    --hash=sha256:10ac20c69d880f77f375db44c22e3e6a644f9fefa291d4cd2fb9790a89fc99fd \
    --hash=sha256:11c423f1049c5755ad4409af52a9ada1cff96fe9b50795d4af3619f292901239 \
    --hash=sha256:179ce34a8d0fe273d8883ba59e17e052247d08973dfcb743ca52bb1cce2d60b0 \
    --hash=sha256:1bca3b943fc2567ea49cd02c99abde49da4d5178ec46f624bd8255cda8755beb \
    --hash=sha256:1d73131e358976663dd969e1fb4ed1404b815cd977eaaedc3b3a133ba2d81c35 \
    --hash=sha256:2a0b02f9fc46f8520330c23d45e6560db7e3a0d927232139427637f98943e11d \
    --hash=sha256:2d3ab0e8c69a17dd3559eab8cbb88f258e285c94d572c2719033f90f83290c89 \
    --hash=sha256:30f464bee641fa8e282577c7dce027308403213c6ca8270bba73285c91024bc5 \
    --hash=sha256:33a834464fdabc0f26a45508df31b3cc5d028e04dbf6c5ed398541418e0a12fe \
    --hash=sha256:3ab3523da44749156e1f68b464dc56af11ae4cbc5c739a49d05f32b982eca9f3 \
    --hash=sha256:3c085faa2cfa879c5141df483f836f4d691045a078224a670fa570fa01612d89 \
    --hash=sha256:457fd7a2a8edeb044ab6ffbc0aa03ff6cd18491356e5e0c834d76ce621b916d1 \
    --hash=sha256:49023963c193dacee096301452f223ee24d86ec5807f8df93c0f7221d119e305 \
    --hash=sha256:52c4b7422442aba924d03ad4019852b08a92e64ea187b933135687bfe2747307 \
    --hash=sha256:559ed65f60c1af5a03f3912605a1b5114f522c7c32fb23c3376ae8f03219fe28 \
    --hash=sha256:5632e3ae3d09197c446310cd5187de63e28448ce22f0f67b2b93d97503c0c230 \
    --hash=sha256:5e4d44984abc0020154ea81b247adeddcc3ac5527b975ff798bd1ba0adc513c2 \
    --hash=sha256:75b00eb8fb802090aa903f4ea1c7f5a584779f967361e68b7e98e531cc2d7174 \
    --hash=sha256:78a0d7c918e74a232394117160e7e3db503377572a45bcef8826e4ab8a35feba \
    --hash=sha256:78c0665edead396b1abb4897c41a5c1d9bf090c8a637a4c20a61678e0a264e66 \
    --hash=sha256:7bbf207c4453ce1ad2e00b17313852b33310b83090c2311bdaf97f93c0380d12 \
    --hash=sha256:7f4b8bc363b6d65ee2152bec57568e3c52639bb34c46057b09857a307ed5e21d \
    --hash=sha256:82f201b4c878551d48558337aab270d3c6cca5507b8737c8d8a608d234cccde0 \
    --hash=sha256:83de5453a7799afc9048b4616bd085cef126e36412f0ea2f6370c36a2a3a51e7 \
    --hash=sha256:88f0e784020649f88ea48c9f5ddfa403bf9205820667c0914740b392035afb82 \
    --hash=sha256:8bcf3c1ba5d6456e2effd30fcbd3459b044d683fcdac79a2e6830f0bdf7de487 \
    --hash=sha256:911de823097db8b63f034299d12662db93344e6ffa0b881cbb57748974b70168 \
    --hash=sha256:92c14f5bdbfb6216315ce33e78080474082de8b3830122ba97809bfbe65f75c0 \
    --hash=sha256:95298364e251be3e60249facbeeca03631d3bb7584f85879516ec55ac717b81f \
    --hash=sha256:9554bcc6d715ee87a633a3cc8e7703c6628b100dd29cb8a2efc4c0533c7ff729 \
    --hash=sha256:9f2897bf7737392ad0d5213ea7b6add72a4edf5679b3153106aeb88b6507b3b9 \
    --hash=sha256:a1d33a7836f7ddc1993427966a0823468ec41bcbdb1a9f9942d1d7e57f803ba3 \
    --hash=sha256:ac0333bdf38309aa3dcbe7e3fa7ea29e7a2c37c6ea306a757b700ded8e4596ad \
    --hash=sha256:bff0b729edd992766136b34e39cc76bc2fad905aa58897ee72a9cd000a6d8443 \
    --hash=sha256:c24acac1e18912761c4700239bbc1fd32f615af690f1584d49b35859be51324d \
    --hash=sha256:c35d74ce0e193ff740c2f2be2ac913ddc232fe6c1ff40b26cfecb9c670c63314 \
    --hash=sha256:c825cef2f49e46753726a7181a8e199804a912b29519ada542c6ebc654951899 \
    --hash=sha256:c9d18a33309122074ea483dd92dd444189166b8b2ec429fe9ed5ac73c7a0aa23 \
    --hash=sha256:cbf38d043c1aa4ab306e1ada6ab6eddacc3322a20b7af1b30bc93254b366fe09 \
    --hash=sha256:cd479fc04dd9401e3b4f49e76518768ef99c4f517a98c284eb091fd725719adf \
    --hash=sha256:ceb30a00ce7c92d459819443d29ca486d882b83fb6738bdcbb2a1cce94ac5daa \
    --hash=sha256:cfbf154f2ba187f2ed6cce2639efff7d105f1140573642c0161615b6d91d6a87 \
    --hash=sha256:d2924a03db38dc2e848bca2fe9f077dafb891480b91a00a0963a8cf86dfc31c1 \
    --hash=sha256:d416b16cccfd70fbf62400e84d0bb2f4e6af519a45557f1692c749b37f14b315 \
    --hash=sha256:d65d448389b8436493abcf629cc94ad0cf32aecaf06e1acca1de53cc795f2f12 \
    --hash=sha256:d84a09d0dad90ba6525d8ac1c2334b33e64bf3ccfe9e841f02feb867a22681e4 \
    --hash=sha256:ddef79fb382df40104a19bb7151b3b23e57c1778fcf857c71ceecd9bd264513f \
    --hash=sha256:e3b417bf8c2c7c16e8f58ad91db17783ec911ac16e7b50eb6eab6e809b4f5b07 \
    --hash=sha256:e402cf31eb68f453dbb2d36fc6d722b33f24a55d68b2ae1d92fa6305ca71c298 \
    --hash=sha256:e6fb6a55cc0ba97b59a1f288fb86dc6fce8bdfc0fffcbfd015e3a954bf2a2d93 \
    --hash=sha256:e708533e8b2ae2497d65346538a7dcc92814410b25b81432eac66de0f2af8265 \
    --hash=sha256:ea324d9dd34c38bfb9bec8ca4d1b407db97dbb74029f566b8e322b1b6fe56fe6 \
    --hash=sha256:eb0dfcf4e28a99c12c999744a2ff67c9b06200e20401c7c88186e33552a46331 \
    --hash=sha256:eda632a7981f69730d6281f451db9c1c370993a2c0d7ddb43e2a809a2862b83a \
    --hash=sha256:f29633129f9fa7e88a3f0fca835de2d030bfc9643f7799e1a0c46cee24d38fc7 \
    --hash=sha256:f55fa87b6c612ecd6b058f167c53231b1d14e412efe361d3d6e38b3631c73218 \
    --hash=sha256:fdaf5ea890a6183d0565f51a61799d67081bd5b1cf03c5f4b3fd3732108625c9
    # via -r notebooks/kaggle/requirements-kaggle.in
setuptools==84.2.0 \
    --hash=sha256:6f445ca2ce864605f81e223f262b4c94ffcb9a4a63473bae0ed7be5aab0291da \
    --hash=sha256:8cc107ac7a403105dc8d6477664171f36135fab26adf8b5e0edaebc313f39b21
    # via torch
shellingham==1.5.4 \
    --hash=sha256:7ecfff8f2fd72616f7481040475a65b2bf8af90a56c89140852d1120324e8686 \
    --hash=sha256:8dbca0739d487e5bd35ab3ca4b36e11c4078f3a234bfce294b0a0291363404de
    # via typer
sympy==1.14.0 \
    --hash=sha256:d3d3fe8df1e5a0b42f0e7bdf50541697dbe7d23746e894990c030e2b05e72517 \
    --hash=sha256:e091cc3e99d2141a0ba2847328f5479b05d94a6635cb96148ccb3f34671bd8f5
    # via torch
tokenizers==0.23.3 \
    --hash=sha256:114e2b55ed177179d59f4ab98200a4471e11e78f9e4b5a922d146740f96fcf52 \
    --hash=sha256:1554a6eed34d9d6a78d23360f4e06df8dffab1ae08c7e8488e0b3e3b36cc266f \
    --hash=sha256:1ebf28794e7e4954e20a7f70fbea410b2d1f0418f7dbbca97ca384fcfef38c25 \
    --hash=sha256:1f0823bb00c5fdc98e487354d54dd55a03848d61a1a0bf29a68c77f24f3b26c3 \
    --hash=sha256:2a89614730d7b80940a5d2ed9320e1ec8add5a745c6151d8d05071b7215505b6 \
    --hash=sha256:376851d22bcf9d650a5c3090bb83e6cf9e895fbf0595369fa4cd43c1f69b5f87 \
    --hash=sha256:5cc24bb457dd4a8af89c8fcb40074d570129ec473df2a866c276ee55db4749d7 \
    --hash=sha256:68649e97d5b43c44c031d8d848874a6eecae8f8fe40ea989aa777a5a83aca716 \
    --hash=sha256:7e48734d2de9260d86f03ab056d2cfeeff3869f61dbd49aaa15a2793b5f3458b \
    --hash=sha256:82eb480f6f1c21cea3349dec32cf1a6384c6c1e775f00f83b0d51197bc013687 \
    --hash=sha256:84513ef0aeb8bf8f4ea11a2e8a7ac163ec5288aa115e649a59b470ac5c3107df \
    --hash=sha256:9d2b5c97daf61688c2ad1803ca851800feaba50fb68d5821779e9ea5880d968c \
    --hash=sha256:a4fbb3662f9f59d199d61338e54b4bcc11d07ebbb1aeb3540dacb2be9c521cb7 \
    --hash=sha256:acd5c57b4bd3e56e246e2731a3a3a6825a7a7d89b7e3b761ba80bc521710f04b \
    --hash=sha256:bf501c40b72d2d5c8623620210430e9cac1ce47a46e45b34107b70a1557d46b0 \
    --hash=sha256:c64a0713180ff16829d4e7f39a658b77ea11443af4e1aa46523692943c9b1414 \
    --hash=sha256:cded33237c77caeef62944d32aa9a7ef42bdce2b3497e18d137e072a8c4be438 \
    --hash=sha256:d3407fb7b9c4d75dd68850ffd7180bc0a5d2dbaf0762d888e612f31fec3f9c6b \
    --hash=sha256:ddedfd4b3b4be6be24ff6ca645c4a37fddfd305f6f3e354c54cf10b715c48215 \
    --hash=sha256:de536665495cb4b409d25bade41963f801aff4225c19a6b804b048f7d14e34c7 \
    --hash=sha256:e05ab7baf7f47b406a95fea6f3b0a484b2ddcd9e1d14b68844c457eb755085a3 \
    --hash=sha256:e88646b8580c5ad7f4361477f1298e9cc01771a1ee9aecfe32c47b8ff614cc38 \
    --hash=sha256:ec82e80e65a862275b97c3d90b7a523df8d9519ee48aeb4e9625b2cc909274e0 \
    --hash=sha256:efa3d7318406b4d115dce61ad5061953f1f44b128e79c020ce4615d763e23b6e
    # via transformers
torch==2.14.1 \
    --hash=sha256:06c3ec25f3b497f9a73dc0c36f293f295cfe8d5584fd446238d9a501b30a66d5 \
    --hash=sha256:0b13acde294b401509830d85a8906cbf2fc97425a12a66cce1d9fa13c5106a2a \
    --hash=sha256:0bd2fb0c4098856bc5af59fe5988bf38bdbcdb6550b44273c12ae425cb8ec17e \
    --hash=sha256:1d4df07be7338bbcc4d54085adee21363c91314702b9bd4d78ef72ffac9465ba \
    --hash=sha256:1fee6b20869a90ccec05e35559a706338ac8aab40a9c4fb58f16667e93af36ae \
    --hash=sha256:23011fe29a99b591eabb3a31a25080c62e2c2f0690a3c798744e5489dae50451 \
    --hash=sha256:305a61f61f35f128579f299c5bd33d475f6a01c6307336139632e30856c4854d \
    --hash=sha256:38bee9f2a2ccfc6898172a5075e4ba52522fbe143eeb099674fc67ab390e858d \
    --hash=sha256:420dbf314c180ee4b86e9bc00aee5746a7d6e5bacdd7df925af671ed393f0b2e \
    --hash=sha256:6d530bd11479fb574567af3a1f384af9a77bf5bb9550cde8080b356ca7220d5a \
    --hash=sha256:711713391d26a1ce5e9fbc6c996d954a8c12e8825374cc77b809a6af29539b8c \
    --hash=sha256:9113f94f70429f9f302bf55b090b5411269083e8a72a5d015ffcc5a7f83f2c69 \
    --hash=sha256:957df236525bcd940ccd9ca59074e9821182171253417c0d98e7b82647254799 \
    --hash=sha256:9ee7d50ec80ae5b4258cafc0ea108711988d961f21d0a68cf685650da2812e5f \
    --hash=sha256:b6074b130fd26bc50f5d171f07dfed70db22dd0c354ab34767729249f9e0f042 \
    --hash=sha256:c8f71aabc67bcbfc9373dc131537a5968d04edce73e88add21354a7cd0a76985 \
    --hash=sha256:cee091caf2a6229e248daf41d18ceebf590ba02d9179205610062c64ee5fef03 \
    --hash=sha256:d02a4c48a2ca5fb7654e36e71f710f74494d83f1f10aaff8e059dd554adca956 \
    --hash=sha256:dbe359d705f4d67236743794c296c6dee93a922fd8117eff8c3e880d7d0fb2b9 \
    --hash=sha256:e07306caa1de2a4ac1467e11ecfc92fc44f523dd6a521145039aef46d913963c \
    --hash=sha256:e24b9815001d18e73e76eae1ea25f8ef908c6103e857378ad20f8f6cf5c954f5 \
    --hash=sha256:e65d5fe136e533b23c2d134377f7f126721c7c78dd75b3dbb735af082a8aeb85 \
    --hash=sha256:ef1276fd07adb8d463ac402a24ea7482e45c9a42df8ebdac211aa757779200ba \
    --hash=sha256:f68f5476e2bc0e8f60b74f7ca21acda885c4477af6392f8977e4f0d1ea1aa162
    # via -r notebooks/kaggle/requirements-kaggle.in
tqdm==4.70.1 \
    --hash=sha256:c293e525e6fef9c20e8728fd4612df02a0aa31bb5fe91ecd93e123b1b7bffa73 \
    --hash=sha256:cefd0eca11b2a37a3aee776544d4f4ae913f02688135b5556b8788dfa474afc4
    # via
    #   huggingface-hub
    #   transformers
transformers==5.20.1 \
    --hash=sha256:25b30879e4c243f00630c1079171ede6d4f6bf76e68f978acd47628a640bc8a0 \
    --hash=sha256:5f51ce444193e5d0a751e3fa489ddbc2224ff12f6675a5cfefd4044b09757c19
    # via -r notebooks/kaggle/requirements-kaggle.in
triton==3.8.0 \
    --hash=sha256:1b84e7d512490ba529111260fa6f7cad8b254a6bb5fbdf41d5ef9a5e57f52d0a \
    --hash=sha256:1f0497218e26b7d79773ad9c2a3fa3b539ee69f587a13fac2e552b1d322a8015 \
    --hash=sha256:1f6b48d0591929a3867973acac3dccd4e058585f91bfb41022de496c9ffab304 \
    --hash=sha256:372285307d4c44ee74cee32de0b4f04bd157e071427e38c6f4ee3e3beb2194f4 \
    --hash=sha256:387dae4cb0089a7b6ba1a428ae0782b65c4c58f57d94617cb22ca8593d8ccbca \
    --hash=sha256:398f4b009c7ab08ed9aeb1d1282dee822945b53f399a5e12a4fecb643cf2007d \
    --hash=sha256:68988ac85d5e7086baeda0ddc175af9667db7529b3c5e11a5c0601b8bef2200a \
    --hash=sha256:6d914c52f89dc942b1819db959e7c07a5999f678ec760f677625c706b7e0d743 \
    --hash=sha256:74217bb56ed8692759227758e4c4b3bd2d608a209c1a7a081bf361fb4c2c1bf9 \
    --hash=sha256:a9c404c69ed4a39e8ec632eaf6b9fe058a060bf98979c177f6ef666f06bb8d50 \
    --hash=sha256:b7004666652f500ed854a86988e4b3d69d247188b5d2092b5df1e44f4a954099 \
    --hash=sha256:e91ffa46d095b252248297292dd22bcbacd53a125a0c2eefbbbf74925a320bc3
    # via torch
truststore==0.10.5 \
    --hash=sha256:30d36967ccaded5cbb38d602c433f53600036c79d502f4533a49b60a03bbefcd \
    --hash=sha256:9aaaedaefaf06d8b206278cf8b5012bc897f485a874503501e12d776df78951c
    # via
    #   httpcore2
    #   httpx2
typer==0.27.3 \
    --hash=sha256:d0396f770a560ab1b0a8504e13b5f254b728cedb05c61cf0359e944e50ce8901 \
    --hash=sha256:e50022f28b82a86313e54501317a1db64bf8f8d036ff8cfe5ca7e47675454aff
    # via transformers
typing-extensions==4.16.0 \
    --hash=sha256:481caa481374e813c1b176ada14e97f1f67a4539ce9cfeb3f350d78d6370c2e8 \
    --hash=sha256:dc983d19a509c94dba722ee6abd33940f7c05a89e243c47e907eb4db6f1a43e5
    # via
    #   anyio
    #   httpx2
    #   huggingface-hub
    #   onnx
    #   torch


In [ ]:
# The split and label files, downloaded from the public repository at a fixed commit and checked before use.
import hashlib, pathlib, time, urllib.request
DATA_COMMIT = 'ea10632e6eea1e24665f40c36cfb5f38e45e10a3'
DATA_FILES = {
    'annotations_test.json': (460790, 'd602a8d86ff63ef4ece1dfa20d9a7b5307d3b4066b37f8245b84045f54bcce6e'),
    'annotations_train.json': (8988090, '3b032f97eeff5dc5d67a055cca4cc4d31594b38f2d50f9b2edd73be00b87431b'),
    'annotations_val.json': (395537, '088a2717399a70a8bf767db40c0ef09a51c75536bb90abce903d9b8e813b23c6'),
    'exclusions.json': (118, '43bd5c510543a149a5da078e8b3a2f64d2519507b94d8b58bedc4a8231adcb3d'),
    'split_manifest.json': (9576089, '87e8536e88d637516185fc20a500e0da35443676489f6589d5dc0457e2d1ea8f'),
}
for name, (size, digest) in DATA_FILES.items():
    url = 'https://raw.githubusercontent.com/kurtvalcorza/rice-pest-detector/{commit}/data/{name}'.format(commit=DATA_COMMIT, name=name)
    for attempt in range(4):
        try:
            with urllib.request.urlopen(url, timeout=120) as r:
                body = r.read()
            break
        except OSError as err:
            if attempt == 3:
                raise SystemExit(f'Could not download {name} ({err}). Check that internet is on in the notebook settings.')
            time.sleep(5 * (attempt + 1))
    if len(body) != size or hashlib.sha256(body).hexdigest() != digest:
        raise SystemExit(f'{name}: size or SHA-256 differs from the pinned file; refusing to train on it.')
    pathlib.Path('/tmp/rpd/data', name).write_bytes(body)
    print(f'{name}: {size:,} bytes, SHA-256 ok')

In [ ]:
import hashlib
from pathlib import Path
CARRIED = {
    'rice_pest_detector/__init__.py': 'ad2a8e84b83c9043fe992aed2daeedf5b7a55c3552d3573fb328c70e15a73f2b',
    'rice_pest_detector/dedup.py': '618ae02dbca94cc39bb33f67ef9ba56465d00249b469a9a4a1bae2ec0824f2bb',
    'rice_pest_detector/families.py': '97a00ff6adfd99fb3ebbad2339ea33a49097783e38bd065d5dec89ed6908ed4a',
    'rice_pest_detector/fetch.py': 'f67fd9ac042c198b7e54e3ffa556785757875470fb5e989c68fc209d4aca2820',
    'rice_pest_detector/infer.py': '2629db3c848450b96bb931b4082c4a2a32dc5df6dcb6202dc5ddf67b64d0d261',
    'rice_pest_detector/labels.py': '2cd57832457cb02b7714d5b4a1e478fa4e8a9fc72cc648a5ff06a3b1f2d20864',
    'rice_pest_detector/manifest.py': '8986ddb2e3caa9929d57092fa3023516af02ac246f1604bda47b2fb1738e65ad',
    'rice_pest_detector/metrics.py': '79c74c5b1fcc2da29fc54177bee533c0d22ff65cb31b0170eb932e8381e6062e',
    'rice_pest_detector/split.py': '22050bf1536ab2de29cebdd6cb17cae6fe9e3f1c9bc1c80883e2118a4129db48',
    'rice_pest_detector/train.py': 'f741827839e815bf1847843e1798444ae83740c246f57260f87cf03fbb3e5cd2',
    'rpd_tools/export_onnx.py': 'e384068f3cc806524b583b7cae305ca9ac5464adbf4d71e9162494b70f99c085',
    'kaggle_driver.py': 'fd3718db66a1faf192ddc8d4b11cebc5bc3810b8bc29e33297227feec0262983',
    'requirements-kaggle.lock.txt': 'c1321d4b0f8cb14b61e03a29fb534ea5b2774813646b271f395763c321f4080d',
    'data/annotations_test.json': 'd602a8d86ff63ef4ece1dfa20d9a7b5307d3b4066b37f8245b84045f54bcce6e',
    'data/annotations_train.json': '3b032f97eeff5dc5d67a055cca4cc4d31594b38f2d50f9b2edd73be00b87431b',
    'data/annotations_val.json': '088a2717399a70a8bf767db40c0ef09a51c75536bb90abce903d9b8e813b23c6',
    'data/exclusions.json': '43bd5c510543a149a5da078e8b3a2f64d2519507b94d8b58bedc4a8231adcb3d',
    'data/split_manifest.json': '87e8536e88d637516185fc20a500e0da35443676489f6589d5dc0457e2d1ea8f',
}
for _name, _expected in CARRIED.items():
    _path = Path(WORK, _name)
    if not _path.is_file() or hashlib.sha256(_path.read_bytes()).hexdigest() != _expected:
        raise RuntimeError(f'{_name}: the file written above differs from the build (SHA-256). Run all again.')
print(len(CARRIED), 'carried files match the build (SHA-256), the package lock included')

In [ ]:
import hashlib, io, json, os, shutil, subprocess, time, urllib.error, urllib.request, zipfile
from pathlib import Path
UV_URL = 'https://files.pythonhosted.org/packages/1e/fd/432451d732917c49152a291de3ef171aa6b0f1a22d39780fb2c1f085ca4c/uv-0.12.15-py3-none-manylinux_2_17_x86_64.manylinux2014_x86_64.whl'
UV_SHA256 = 'aee9802f46bae436bd91751bb33ddeb379ef1596b5c19df193219d545d244b60'
UV_BYTES = 20081404
LOCK = Path(WORK, 'requirements-kaggle.lock.txt')
_started = time.time()
for _attempt in range(3):
    try:
        with urllib.request.urlopen(UV_URL, timeout=90) as _response:
            _wheel = _response.read(UV_BYTES + 1)
        break
    except (urllib.error.URLError, TimeoutError, ConnectionError) as _exc:
        if _attempt == 2:
            raise RuntimeError('Could not download uv 0.12.15; check that Internet is on, then Run all.') from _exc
        time.sleep(2 ** _attempt)
if len(_wheel) != UV_BYTES or hashlib.sha256(_wheel).hexdigest() != UV_SHA256:
    raise RuntimeError('The uv 0.12.15 wheel does not match its pinned size and SHA-256: refusing to use it.')
with zipfile.ZipFile(io.BytesIO(_wheel)) as _archive:
    _member = next(n for n in _archive.namelist() if n.endswith('.data/scripts/uv'))
    UV = Path(WORK, 'uv')
    UV.write_bytes(_archive.read(_member))
UV.chmod(0o700)
ENV = dict(os.environ, HF_HUB_DISABLE_TELEMETRY='1', HF_HUB_DISABLE_IMPLICIT_TOKEN='1', DO_NOT_TRACK='1')
for _key in ('HF_TOKEN', 'HUGGING_FACE_HUB_TOKEN', 'PYTHONPATH', 'PYTHONHOME', 'PYTHONSTARTUP'):
    ENV.pop(_key, None)


def _step(name, cmd):
    done = subprocess.run(cmd, env=ENV)
    if done.returncode:
        raise RuntimeError(f'Environment step failed: {name} (exit {done.returncode}). Nothing was trained; '
                           'read the messages above, then Run all again.')


shutil.rmtree(Path(WORK, 'env'), ignore_errors=True)
_step('create the Python 3.12.12 environment',
      [str(UV), 'venv', '--managed-python', '--python', '3.12.12', str(Path(WORK, 'env'))])
PYTHON = str(Path(WORK, 'env', 'bin', 'python'))
_step('install the hash-locked packages',
      [str(UV), 'pip', 'install', '--python', PYTHON, '--require-hashes', '--only-binary', ':all:',
       '--index-url', 'https://pypi.org/simple', '-r', str(LOCK)])
VERSION_CHECK = ('import sys, numpy, PIL, onnxruntime, torch, transformers; '
                 'print("Python", sys.version.split()[0]); print("torch", torch.__version__, "CUDA", torch.version.cuda); '
                 'print("transformers", transformers.__version__); print("numpy", numpy.__version__); '
                 'print("Pillow", PIL.__version__); print("onnxruntime", onnxruntime.__version__); '
                 'assert torch.cuda.is_available(), "CUDA is not visible inside the notebook environment"; '
                 'print("GPU", torch.cuda.get_device_name(0))')
_step('check versions and CUDA inside the environment', [PYTHON, '-c', VERSION_CHECK])

def _record_time(stage, seconds):
    path = Path(RUN_DIR, 'stage_times.json')
    times = json.loads(path.read_text()) if path.exists() else {}
    times[stage] = round(seconds, 1)
    path.write_text(json.dumps(times, indent=1))


def run_stage(stage, *options):
    log = Path(RUN_DIR, f'{stage}.log')
    cmd = [PYTHON, '-u', str(Path(WORK, 'kaggle_driver.py')), '--stage', stage, '--work', WORK,
           '--run-dir', RUN_DIR, *options]
    print('Running stage', stage, 'in the notebook environment; log:', log, flush=True)
    started = time.time()
    with log.open('w', encoding='utf-8') as output:
        child = subprocess.Popen(cmd, env=ENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in child.stdout:
            output.write(line)
            output.flush()
            print(line.rstrip(), flush=True)
        child.wait()
    _record_time(stage, time.time() - started)
    if child.returncode:
        print(log.read_text(encoding='utf-8')[-6000:])
        raise RuntimeError(f'Stage {stage!r} failed (exit {child.returncode}); the end of its log is repeated above.')


_record_time('environment', time.time() - _started)
print(f'environment ready in {time.time() - _started:.0f} s; Kaggle\'s own packages were not changed')

In [ ]:
run_stage('env-log')

In [ ]:
run_stage('fetch')

In [ ]:
run_stage('validate')

In [ ]:
run_stage('resume', RESUME_FROM)

## Training method and settings

- **Method:** full fine-tune of `Roboflow/rf-detr-medium` (revision `1b5b672408f86dd38e05dd3cf3f2e0834e545a59`); every parameter is trainable.
- **Optimiser:** AdamW, learning rate 1e-4 (backbone 1e-5), weight decay 1e-4, gradient clipping 0.1; cosine schedule over 40 epochs.
- **Batch and precision:** batch 4; fp16 autocast, with the Hungarian matcher and the losses in float32.
- **Input:** 576 × 576 pixels; 300 queries.
- **Seeds:** training seed 20261013 and a per-epoch data-loader seed.
- **Checkpoint selection:** after each epoch the model is evaluated on the validation split; `best.pt` keeps the epoch with the highest validation species mAP50. The validation photos are never used for gradient updates, only to choose the checkpoint.

These are the published recipe's values, not tutorial values: the run takes hours (see the top of the notebook). Training loss is not task quality; quality is judged on validation mAP50 and counts. `last.pt` keeps the optimiser state so that a run can be resumed from an attached earlier output.

In [ ]:
run_stage('fit')

In [ ]:
run_stage('evaluate')
_val = json.loads(Path(RUN_DIR, 'val_metrics.json').read_text())['headline']
print('\n| metric | value |\n| :-- | --: |')
for _k in ('map50_species', 'map50_four_class', 'count_accuracy', 'count_accuracy_no_detection_baseline', 'wsb30_mae'):
    print(f'| {_k} | {_val[_k]} |')

In [ ]:
run_stage('export')

In [ ]:
run_stage('reload')

In [ ]:
run_stage('inference')

In [ ]:
RUN_TEST_LOOK = False  # @param {type:"boolean"}
if RUN_TEST_LOOK:
    run_stage('test-look')
else:
    print('Test split: skipped (default). Both looks the rules allow were spent on 13 and 18 Oct 2026.')

In [ ]:
run_stage('summary')
print('machine-readable summary:', Path(RUN_DIR, 'run_summary.json'))
print(f'whole notebook: {(time.time() - SESSION_START) / 3600:.2f} h')

## How to read the results

- **These are this run's numbers.** The evaluation above is one validation pass at the published threshold (0.48): tutorial evidence, not a benchmark. Read your own printed numbers and `run_summary.json`.
- **They will differ slightly from the recorded run.** GPU training is not bit-repeatable, and Kaggle's GPU driver and CUDA libraries differ from the Colab run that produced the published model; NumPy and Pillow are pinned to the versions the project used. For RF-DETR-M the project recorded validation species mAP50 0.933 (run RF-DETR-M, Colab T4, 17 Oct 2026).
- **Count accuracy here is not the recorded figure.** The recorded count accuracy (0.846) is out-of-fold: the threshold was re-chosen on each fold. This notebook counts once at the published threshold, so the two are different estimates.
- **What the evaluation does not show.** It uses validation photos from the same dataset and traps; it says nothing about other traps, cameras or regions. The base models were pretrained on COCO (and RF-DETR's backbone on a large image collection), which cannot be shown to exclude similar images. Scores are raw sigmoid outputs, not calibrated probabilities; the review flag (`review_below` = 0.63) was set by offline calibration on validation.
- **Counting rule.** A detection is counted when its score is at least the published threshold.

## What did not work

- **An older transformers release trained the wrong loss.** The first two runs (r18-1 and r50-1) used transformers 4.57.6, whose RT-DETR main loss was computed over the denoising queries as well. transformers 5.18 fixed it; the published models were retrained on 5.20.1, and the older runs are kept as history only (evaluation record, wave 2).
- **Early peaks, then a slow drift.** All three models reach their best validation species mAP50 within the first four epochs and then drift down slowly (r50-v5 0.908 and RF-DETR-M 0.891 at the last epoch; evaluation record, wave 2). The checkpoint is chosen on validation, so the drift does not reach the exported model, but the best epoch is unstable from run to run.
- **The threshold grid edge.** The pre-registered threshold grid stops at 0.50; one r50-v5 fold and four D-FINE-L folds chose 0.50, so their best threshold may lie above the grid. Recorded, not changed: the grid was fixed before the results (evaluation record, wave 2 and D-FINE-L).
- **Best on validation, weakest on test.** RF-DETR-M had the best out-of-fold validation count accuracy (0.846) but the weakest on the test split (0.806, a post-hoc comparison by owner exception, 18 Oct 2026). Differences of a few points between these models are within run-to-run and split-to-split variation.
- **D-FINE-L's ONNX scores.** Its ONNX scores differ from PyTorch by up to 2.95e-3 with identical kept detections, classes and boxes, above the 1e-3 tolerance. The owner waived the score tolerance for D-FINE-L on 18 Oct 2026; its reload check still enforces kept detections, classes and boxes (evaluation record, D-FINE-L parity).

**Which model the project chose, and why.** r50-v5 is the demo model. It was chosen on validation under rules written before any result: it met rule A against r50-1, and neither RF-DETR-M nor D-FINE-L met rule B against it.

## References

1. Deleña, R.; Balleras, G.; Dequiña, B. J.; Jumawan, R.; Flores, C. Annotated light-trap rice pest image dataset with detector-derived region-of-interest crops for species-sex recognition of rice black bug and white stemborer. Version 1. Zenodo. [doi:10.5281/zenodo.20066074](https://doi.org/10.5281/zenodo.20066074)
2. Y. Zhao et al., "DETRs Beat YOLOs on Real-time Object Detection" (RT-DETR). [arXiv:2304.08069](https://arxiv.org/abs/2304.08069)
3. I. Robinson et al., "RF-DETR: Neural Architecture Search for Real-Time Detection Transformers". [arXiv:2511.09554](https://arxiv.org/abs/2511.09554)
4. Y. Peng et al., "D-FINE: Redefine Regression Task in DETRs as Fine-grained Distribution Refinement". [arXiv:2410.13842](https://arxiv.org/abs/2410.13842)
5. COCO evaluation (pycocotools). [github.com/ppwwyyxx/cocoapi](https://github.com/ppwwyyxx/cocoapi)
6. Rice pest detector, the project repository. [github.com/kurtvalcorza/rice-pest-detector](https://github.com/kurtvalcorza/rice-pest-detector)
7. Rice pest detector, the published models on Kaggle. [kaggle.com/models/kurtvalcorza/rice-pest-detector](https://www.kaggle.com/models/kurtvalcorza/rice-pest-detector)